# Cd47 Deficiency in Mouse Brain Under Excitotoxic Stress

Analysis code for the single-nucleus RNA-sequencing study of hippocampus from wild-type and *Cd47* knockout mice treated with vehicle, subconvulsive kainic acid (KA3, 3 mg/kg) or convulsive kainic acid (KA25, 25 mg/kg).

The notebook runs three complementary analyses on the same dataset: nucleus-level and mouse-level pseudobulk differential expression (Approach 1), weighted gene co-expression network analysis with module preservation and mouse-level eigengene models (Approach 2), and whole-transcriptome distribution comparison (Approach 3).

Sequencing data are deposited in GEO under accession GSE344481. Outputs have been cleared; run the cells in order.

### 1. Load, quality-control and integrate the single-nucleus libraries
Reads the 15 Cell Ranger raw matrices, applies per-nucleus QC (>150 and <7,000 detected genes, <5% mitochondrial counts), removes Scrublet doublets, drops mitochondrial genes, and stores a pre-normalisation copy for the pseudobulk analysis. Counts are then normalised to 10,000 UMIs per nucleus and log1p-transformed, highly variable genes selected, and PCA, neighbour graph, UMAP and Leiden clustering computed.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import time
import os
from harmony import harmonize
import pickle
import scanpy.external as sce
#from util_gen_marker import load_marker_genes

fraction = 0.1
name_of_method = 'batch'
name_validation = ""#"KA3_wt1"+"_"+"KA3_ko1"
time_start = time.time()
# Define paths for each dataset in a nested dictionary
file_dir_dataset_2022 = "/home/kiamari/Single_cell/Single_cell_datasets/"

file_suffix_dataset_2022 = "_raw_feature_bc_matrix.h5"
condition_2022 = {
    'Veh_WT':["Veh_wt1", "Veh_wt2", "Veh_wt3"],
    'Veh_KO':["Veh_ko2", "Veh_ko3", "Veh_ko4"],
    'KA3_WT':["KA3_wt1", "KA3_wt2", "KA3_wt3"],# kept "KA3_wt1" for validation
    'KA3_KO':["KA3_ko1", "KA3_ko2", "KA3_ko4"],# kept "KA3_ko1" for validation
    'KA25_WT':["KA25_wt4"],
    'KA25_KO':["KA25_ko2", "KA25_ko4"]
}
datasets = {
    'batch_2022': {
        "WT": {
            "Veh": [file_dir_dataset_2022+x+file_suffix_dataset_2022 for x in condition_2022['Veh_WT']],  # Replace with actual AnnData objects
            "KA3": [file_dir_dataset_2022+x+file_suffix_dataset_2022 for x in condition_2022['KA3_WT']],
            "KA25": [file_dir_dataset_2022+x+file_suffix_dataset_2022 for x in condition_2022['KA25_WT']],
        },
        "KO": {
            "Veh": [file_dir_dataset_2022+x+file_suffix_dataset_2022 for x in condition_2022['Veh_KO']],
            "KA3": [file_dir_dataset_2022+x+file_suffix_dataset_2022 for x in condition_2022['KA3_KO']],
            "KA25": [file_dir_dataset_2022+x+file_suffix_dataset_2022 for x in condition_2022['KA25_KO']]
        }
    }
}


sex_batch_2022 = {
    "KA25_ko2": "male",
    "KA25_ko4": "female",
    "KA25_wt4": "female",
    "KA3_ko1": "male",
    "KA3_ko2": "male",
    "KA3_ko4": "female",
    "KA3_wt1": "male",
    "KA3_wt2": "male",
    "KA3_wt3": "female",
    "Veh_ko2": "male",
    "Veh_ko3": "female",
    "Veh_ko4": "female",
    "Veh_wt1": "male",
    "Veh_wt2": "male",
    "Veh_wt3": "female"
}


# file_path_combination = f"adata_combined_harmonized_DiffThrshld_{fraction}_{name_of_method}_NO_mt.h5ad"
#file_path_combination = f"adata_combined_harmonized_DiffThrshld_{fraction}_{name_of_method}_NO_mt_NO_SexInBatchEffect_Validation_{name_validation}.h5ad"
file_path_combination = f"adata_combined_YekBasteh_Neuron_And_Glia_harmonized_DiffThrshld_{fraction}_{name_of_method}_NO_mt_NO_SexInBatchEffect_REVISION_Pseudobulk.h5ad"
file_path_combination_ALLGenes = f"adata_combined_YekBasteh_ALLGenes_Neuron_And_Glia_harmonized_DiffThrshld_{fraction}_{name_of_method}_NO_mt_NO_SexInBatchEffect_REVISION_Pseudobulk.h5ad"
file_path_combination_pseudobulk = f"adata_pseudobulk_{fraction}_{name_of_method}_NO_mt_NO_SexInBatchEffect_REVISION_Pseudobulk.h5ad"
    
if os.path.exists(file_path_combination) and os.path.exists(file_path_combination_pseudobulk):
    # Load the existing file
    print("File exists ... Loading adata_combined from file...")
    adata_combined = sc.read_h5ad(file_path_combination)
    adata_combined_all_genes = sc.read_h5ad(file_path_combination_ALLGenes)
    pseudobulk_adata = sc.read_h5ad(file_path_combination_pseudobulk)
else:
    print("File ddoes NOT exist!\n Running the program ...")
    # Initialize an empty dictionary to store adata objects for each sample
    adatas = list()

    # Load and preprocess datasets
    for batch, batch_datasets in datasets.items():
        for group, group_datasets in batch_datasets.items():
            for condition, condition_datasets in group_datasets.items():
                for i, file_path in enumerate(condition_datasets):
                    print("===>",file_path)
                    # Read the data
                    print(f"batch {batch}\n group {group}, condition {condition}\n file_path {file_path}")
                    sample_id = f"{batch}_{condition}_{i+1}"
                    if batch == 'batch_2022':
                        adata = sc.read_10x_h5(file_path)
                        print(f"==>{batch}")
                        id_mice = file_path.replace(file_dir_dataset_2022, "").replace("_raw_feature_bc_matrix.h5", "")
                        print(f"id_mice:{id_mice}")
                        sex = sex_batch_2022[id_mice]

                        # Diff Thresholds because of 
                        Lower_Th, Upper_Th, Th_mit = 150, 7000, 5
                    else:
                        raise ValueError("Not correct batch name!!!!")
                    adata.var_names_make_unique()
                    
                    # Add metadata for batch and condition
                    print(f"batch+sex:{batch+sex}")
                    adata.obs['batch_sex'] = batch+sex
                    adata.obs['sex'] = sex
                    adata.obs['batch'] = batch
                    adata.obs['group'] = group
                    adata.obs['condition'] = condition
                    adata.obs['group_condition'] = f"{group}_{condition}"
                    
                    adata.obs['sample_id'] = f"{group}_{condition}_{i+1}"
                    
                    #adata = ad.concat(adatas, label="sample")
                    
                    print("Available observations (obs):", adata.obs.keys())
                    print("Available variables (var):", adata.var.keys())       
                    # var_names_df = pd.DataFrame(adata.var_names, columns=["gene"])
                    # var_names_df.to_csv(file_path_dataset+"filtered_var_names_CD47.csv", index=False)
                    #print(f"dataset:{sample_id} BEFORE filteration has {adata.obs["sample"].value_counts()}")
                    adata
                    
                    ## Quality Control
                    adata.var["mt"] = adata.var_names.str.startswith("mt-")# adata.var_names.str.startswith("MT-")
                    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)#, log1p=True)
                    
                    ## Violin Shapes Before Filtering 
                    sc.pl.violin(adata,["n_genes_by_counts", "total_counts", "pct_counts_mt"],jitter=0.4,multi_panel=True,save=sample_id+"DiffThrshld_TEST_no_filteration.png")
                    
                    ## Filtering Out 
                    
                    adata = adata[
                    (adata.obs["n_genes_by_counts"] > Lower_Th) & 
                    (adata.obs["n_genes_by_counts"] < Upper_Th) & 
                    (adata.obs["pct_counts_mt"] < Th_mit)]

                    ## ---- NEW: Keep raw counts for later
                    adata.layers["counts"] = adata.X.copy()

                    #print(f"dataset:{sample_id} AFTER filteration {adata.obs["sample"].value_counts()}")
                    adata
                    ## Violin Shapes After Filtering
                    sc.pl.violin(adata,["n_genes_by_counts", "total_counts", "pct_counts_mt"],jitter=0.4,multi_panel=True,save=sample_id+"DiffThrshld_TEST_filteration.png")

                    ## Doublet Detection
                    # # sc.pp.scrublet(adata)
                    # sce.pp.scrublet(adata, copy=True)
                    # ## --- NEW: Doublets (per-sample) and filter
                    # adata = adata[~adata.obs["predicted_doublet"]].copy()
                    print("Columns before scrublet:", adata.obs.columns)

                    adata_with_doublets = sce.pp.scrublet(adata, copy=True)
                    
                    # IMPORTANT: Perform the filtering on the new object, not the original one
                    adata = adata_with_doublets[~adata_with_doublets.obs["predicted_doublet"]].copy()
                    print("Columns after scrublet:", adata.obs.columns)


                    
                    
                    # Store each processed adata in the adatas dictionary
                    adatas.append(adata)        
                    











    unique_keys = [
        f"{adata.obs['batch'].iloc[0]}_{adata.obs['sample_id'].iloc[0]}" 
        for adata in adatas
    ]
    
    # --- START OF CORRECTED ANALYSIS PIPELINE ---
    
    # 2. CONCATENATION: Use the new unique_keys.
    # We'll create a new column, e.g., 'sample_id_unique', to store these.
    adata_combined = sc.concat(
        adatas, 
        label="sample_id_unique",  # Name of the new column with unique IDs
        keys=unique_keys,          # Use the combined, unique keys
        join="outer", 
        merge="unique", 
        index_unique="-"
    )
    
    # At this point, adata_combined.obs will have several useful columns:
    # - 'sample_id_unique': The unique key we just created (e.g., 'batch_2022_WT_Veh_1')
    # - 'batch': The original batch ID.
    # - 'sample_id': The original, non-unique sample ID.
    # - 'group', 'condition', etc.
    
    # Drop mitochondrial genes
    mt_mask = adata_combined.var_names.str.startswith(("mt-"))
    adata_combined = adata_combined[:, ~mt_mask].copy()

    
    ###### ======= Psudobulk analysis (begin: before normalization and logp1 stuff)========
    pseudobulk_adata = adata_combined.copy()
    print("Shape (cells × genes):", pseudobulk_adata.shape)
    print("\nMetadata columns:", list(pseudobulk_adata.obs.columns))
    print("\nFirst few samples:")
    print(pseudobulk_adata.obs.head())
    print("\nLayers available:", list(pseudobulk_adata.layers.keys()))

    ###### ======= Psudobulk analysis (end) ========

    
    # Normalize/log1p
    sc.pp.normalize_total(adata_combined, target_sum=1e4)
    sc.pp.log1p(adata_combined)
    adata_combined_all_genes = adata_combined.copy()
    
    # 3. HVG SELECTION: Find highly variable genes based on the technical BATCH.
    sc.pp.highly_variable_genes(adata_combined, min_mean=0.0125, max_mean=3, min_disp=0.5)
    sc.pp.highly_variable_genes(
        adata_combined, 
        flavor="seurat_v3", 
        n_top_genes=3000, 
        batch_key="batch"  # Use the original 'batch' column
    )
    
    adata_combined = adata_combined[:, adata_combined.var["highly_variable"]].copy()
    
    # 4. PCA + HARMONY: Run Harmony using the correct batch key.
    sc.pp.pca(adata_combined, n_comps=50)
    
    # adata_combined.obsm["X_harmony"] = harmonize(
    #     adata_combined.obsm["X_pca"], 
    #     adata_combined.obs, 
    #     batch_key="batch"  # CRITICAL: Still use the 'batch' column here
    # )
    
    # Graph/UMAP/cluster using the corrected embeddings
    sc.pp.neighbors(adata_combined, use_rep="X_pca") # use X_harmony if 
    sc.tl.umap(adata_combined)
    sc.tl.leiden(adata_combined, resolution=0.5)
    
    # Save
    adata_combined.write(file_path_combination)
    
    adata_combined_all_genes.write(file_path_combination_ALLGenes)

    pseudobulk_adata.write(file_path_combination_pseudobulk)


In [ ]:
# Ensure PCA is computed
# Plot the elbow plot (explained variance)
sc.pl.pca_variance_ratio(adata_combined, log=True, n_pcs=50)  # adjust n_pcs as neede


In [ ]:
# import matplotlib.pyplot as plt
# import scanpy as sc

# # subsets (keeps existing UMAP coords in .obsm['X_umap'])
# adata_2022 = adata_combined[adata_combined.obs["batch"] == "batch_2022"].copy()

# fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# # tip: color by something informative *within* each batch, e.g. 'condition' or 'cell_type'
# sc.pl.umap(adata_2022, color="condition", legend_loc="on data",
#            title="batch_2022", show=False, ax=axs[1])

# plt.tight_layout(); plt.show()
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from scanpy.plotting import _utils

# Use a copy to ensure the original object is not modified
adata = adata_combined.copy()

# --- FIX: Ensure UMAP coordinates exist ---
# This is the critical step. It calculates the UMAP and stores it in adata.obsm['X_umap'].
# This must be run before any code that tries to access those coordinates.
print("Calculating UMAP...")
sc.pp.neighbors(adata)
sc.tl.umap(adata)
print("UMAP calculation complete.")
# -----------------------------------------

# --- Color Map Preparation (this part is correct) ---
adata.obs["condition"] = adata.obs["condition"].astype("category")
conds = list(adata.obs["condition"].cat.categories)
_utils.add_colors_for_categorical_sample_annotation(adata, "condition")
condition_colors = dict(zip(conds, adata.uns['condition_colors']))
# ---


fig, axes = plt.subplots(len(batches), len(conds),
                         figsize=(4*len(conds), 4*len(batches)))
if len(batches) == 1: axes = np.array([axes])
if len(conds) == 1:   axes = axes[:, np.newaxis]

for i, b in enumerate(batches):
    for j, c in enumerate(conds):
        ax = axes[i, j]
        
        # Create a boolean mask for the desired cells
        cell_mask = (adata.obs["batch"] == b) & (adata.obs["condition"] == c)
        
        # Get the subset of cells
        a_bc = adata[cell_mask]

        if a_bc.n_obs == 0:
            ax.set_axis_off()
            ax.set_title(f"{b} • {c}\n(no cells)", fontsize=9)
            continue

        # Get UMAP coordinates for this subset directly from the full adata object
        umap_coords = adata[cell_mask].obsm['X_umap']
        
        plot_color = condition_colors[c]
        
        # Use matplotlib's scatter for direct and error-free plotting
        ax.scatter(
            umap_coords[:, 0], 
            umap_coords[:, 1],
            c=plot_color,
            s=10,
            rasterized=True
        )
        
        # Clean up the plot to look like scanpy's default
        ax.set_title(f"{b} • {c}")
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='both', which='both', length=0)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        for spine in ax.spines.values():
            spine.set_visible(False)

plt.tight_layout()
plt.show()

### 2. Leiden clusters
Inspects the unbiased Leiden clustering on the UMAP.

In [ ]:
print("leiden")
sc.tl.leiden(adata_combined, resolution=0.5)
# Ensure log-normalized data is stored
adata_combined.layers["log_norm"] = adata_combined.X.copy()


### 3. Cluster-enriched marker genes
Wilcoxon rank-sum test of each cluster against all others, used to assign cell identities.

In [ ]:
print("ranking")
# Rank genes
adata_combined.X = adata_combined.layers["log_norm"]  # Use log-normalized data
sc.tl.rank_genes_groups(adata_combined, groupby="leiden", method="wilcoxon")


# Visualize batch distribution
sc.pl.umap(adata_combined, color=["batch", "leiden"],save=f"corrected_version_NO_mt_YekBasteh.png", legend_loc='on data')


# Violin plot for top 5 marker genes across clusters
########sc.pl.rank_genes_groups_violin(adata_combined, n_genes=5, jitter=0.4, save=f"Violin-plot-for-top-5-marker-genes_DiffThrshld_corrected_{fraction}_{name_of_method}.png")  # Replace with genes of interest

# Plot expression of specific genes across UMAP
#sc.pl.umap(adata_combined, color=['MS4A1', 'CD3E', 'NKG7', 'LYZ', 'CD14'], save=f"Expression-of-specific-genes_{fraction}_{name_of_method}.png")  # Replace with genes of interest

# Heatmap of the top 10 marker genes for each cluster
sc.pl.rank_genes_groups_heatmap(adata_combined, n_genes=10, show_gene_labels=True, save=f"Top-10-marker-genes-for-clusters_DiffThrshld_corrected_{fraction}_{name_of_method}_NO_mt_YekBasteh.png")



######. Get CSV files
# Access rank_genes_groups results
results_ranking = sc.tl.rank_genes_groups(adata_combined, groupby="leiden", method="wilcoxon", pts=True)

ranked_genes_filtered_LowQuality_WITH_Cortical = adata_combined.uns['rank_genes_groups']

# Create a DataFrame to store all metrics for genes with p-value < 0.01 for each cluster
filtered_genes_data_filtered_LowQuality_WITH_Cortical = []

# Extract genes with p-value < 0.01 for each cluster
groups = ranked_genes_filtered_LowQuality_WITH_Cortical['names'].dtype.names
for group in groups:
    # Create a DataFrame for each cluster
    group_data = pd.DataFrame({
        'gene': ranked_genes_filtered_LowQuality_WITH_Cortical['names'][group],
        'logfoldchange': ranked_genes_filtered_LowQuality_WITH_Cortical['logfoldchanges'][group],
        'pval': ranked_genes_filtered_LowQuality_WITH_Cortical['pvals'][group],
        'pval_adj': ranked_genes_filtered_LowQuality_WITH_Cortical['pvals_adj'][group],
        'pts': ranked_genes_filtered_LowQuality_WITH_Cortical['pts'][group],
        'pts_rest': ranked_genes_filtered_LowQuality_WITH_Cortical['pts_rest'][group],
        'score': ranked_genes_filtered_LowQuality_WITH_Cortical['scores'][group]
    })
    group_data['cluster'] = group  # Add cluster label
    
    # Filter genes with p-value < 0.01
    filtered_group_data = group_data[group_data['pval_adj'] < 0.01]
    filtered_genes_data_filtered_LowQuality_WITH_Cortical.append(filtered_group_data)

# Combine results into a single DataFrame
filtered_genes_df_filtered_LowQuality_WITH_Cortical = pd.concat(filtered_genes_data_filtered_LowQuality_WITH_Cortical, ignore_index=True)

# Save the filtered genes with all metrics to a single CSV
filtered_genes_df_filtered_LowQuality_WITH_Cortical.to_csv('filtered_genes_with_metrics_pval_below_0.01_NO_mt_YekBasteh.csv', index=False)

# Inspect the filtered DataFrame
print(f"Filtered Genes with p-value < 0.01 and all metrics:\n{filtered_genes_df_filtered_LowQuality_WITH_Cortical.head()}")

# Save the genes grouped by cluster into a dictionary for future use
top_genes_by_cluster_WITH_Cortical = {
    cluster: group for cluster, group in filtered_genes_df_filtered_LowQuality_WITH_Cortical.groupby('cluster')
}

# Save genes by cluster along with all metrics into separate CSVs
for cluster, cluster_data in top_genes_by_cluster_WITH_Cortical.items():
    cluster_data.to_csv(f"./CSV_files/HOPEFULY_filtered_LowQuality_cluster_{cluster}_genes_with_metrics_pval_below_0.01_NO_mt_YekBasteh.csv", index=False)

print("Filtered genes with all metrics and cluster-specific CSV files saved successfully.")



### 4. Neuronal subclustering
Sub-clusters the neuronal compartment to resolve CA1, CA3, dentate gyrus and inhibitory populations.

In [ ]:
import scanpy as sc
import pandas as pd
import os

# Assume 'adata_combined' is your object with the original 26 clusters.
# Assume 'adata_whole' is your object with the original raw counts.

# --- 1. Initial Filtering of Unwanted Clusters ---

# Define clusters to remove based on your initial annotation
clusters_to_remove = ['0', '1', '3', '9', '10', '15']
print(f"Performing initial filtering to remove clusters: {clusters_to_remove}")

# Create a clean base AnnData object
mask_to_keep = ~adata_combined.obs['leiden'].isin(clusters_to_remove)
adata_filtered = adata_combined[mask_to_keep].copy()

print(f"Original cell count: {adata_combined.n_obs}")
print(f"Cell count after initial filtering: {adata_filtered.n_obs}\n")


# --- 2. Define a Reusable Function for Sub-clustering ---

def subcluster_and_analyze(adata_base, clusters_to_subset, group_name, new_resolution, adata_raw_counts):
    """
    Subsets an AnnData object, re-runs the analysis pipeline at a higher resolution,
    and saves the results.
    """
    print(f"\n{'='*20} Starting Sub-clustering for: {group_name.upper()} {'='*20}")
    
    # --- A. Create the Subset ---
    mask_subset = adata_base.obs['leiden'].isin(clusters_to_subset)
    adata_sub = adata_base[mask_subset].copy()
    print(f"Created '{group_name}' subset with {adata_sub.n_obs} cells.")

    # --- B. Re-run Core Analysis Pipeline ---
    print("Re-running analysis: HVGs, Scale, PCA, Neighbors...")
    sc.pp.highly_variable_genes(adata_sub, flavor="seurat_v3", n_top_genes=2000, batch_key='batch')
    
    adata_pca = adata_sub[:, adata_sub.var.highly_variable].copy()
    sc.pp.scale(adata_pca, max_value=10)
    sc.tl.pca(adata_pca, svd_solver='arpack')
    
    adata_sub.obsm['X_pca'] = adata_pca.obsm['X_pca']
    sc.pp.neighbors(adata_sub, n_neighbors=15, n_pcs=30)

    # --- C. Re-cluster and Visualize ---
    leiden_key = f'leiden_{group_name.lower()}_res{new_resolution}'
    sc.tl.leiden(adata_sub, resolution=new_resolution, key_added=leiden_key)
    sc.tl.umap(adata_sub)
    
    print(f"Visualizing new sub-clusters for {group_name}...")
    sc.pl.umap(adata_sub, color=[leiden_key, 'batch'],
                title=f'{group_name} Sub-clusters (res={new_resolution})',
                save=f'_{group_name}_subclusters.png',
                legend_loc='on data')

    # --- D. Find and Save New Marker Genes ---
    print(f"Ranking genes for {group_name} sub-clusters...")
    
    # Populate the .raw attribute for correct statistical testing
    adata_sub.raw = adata_raw_counts[adata_sub.obs_names, :].copy()
    
    sc.tl.rank_genes_groups(adata_sub, groupby=leiden_key, method='wilcoxon', use_raw=True)
    
    # Create a dedicated directory for the new CSV files
    output_dir = f"./CSV_files_{group_name}_subclustered"
    os.makedirs(output_dir, exist_ok=True)
    
    result = adata_sub.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    
    for group in groups:
        df = pd.DataFrame({
            'gene': result['names'][group],
            'logfoldchange': result['logfoldchanges'][group],
            'pval_adj': result['pvals_adj'][group],
            'score': result['scores'][group]
        })
        
        # Save all ranked genes for this sub-cluster
        filename = os.path.join(output_dir, f"subcluster_{group}_markers.csv")
        df.to_csv(filename, index=False)
        
    print(f"Marker gene CSVs for {group_name} saved in '{output_dir}'.")
    print(f"{'='*20} Finished Analysis for: {group_name.upper()} {'='*20}\n")
    
    return adata_sub


# --- 3. Define Cluster Groups and Execute the Analyses ---

# Define the clusters for each major cell type based on your annotations
glia_clusters = ['14', '11', '13', '12', '5', '8']  # Microglia, Oligo, Astro
neuron_clusters = ['7', '6', '4', '2']             # CA3, Inhibitory, CA1, DG

# Set the desired high resolution for sub-clustering
high_resolution = 1.5

# --- Run the analysis for GLIA ---
adata_glia_subclustered = subcluster_and_analyze(
    adata_base=adata_filtered,
    clusters_to_subset=glia_clusters,
    group_name="Glia",
    new_resolution=high_resolution,
    adata_raw_counts=adata_combined_all_genes
)

# --- Run the analysis for NEURONS ---
adata_neurons_subclustered = subcluster_and_analyze(
    adata_base=adata_filtered,
    clusters_to_subset=neuron_clusters,
    group_name="Neurons",
    new_resolution=high_resolution,
    adata_raw_counts=adata_combined_all_genes
)

print("\nAll sub-clustering analyses are complete.")

In [ ]:



print(adata_neurons_subclustered.obs.keys())
print(adata_neurons_subclustered.shape)

print("-"*50)
print(adata_glia_subclustered.obs.keys())
print(adata_glia_subclustered.shape)

print("-"*50)
print(f"adata_filtered: {adata_filtered.shape}")

print(f"adata_combined_all_genes: {adata_combined_all_genes.shape}")




In [ ]:
import pandas as pd

# --- Detect the Leiden keys added inside subcluster_and_analyze (robust to resolution formatting) ---
def find_leiden_key(adata, group_prefix):
    # e.g., "leiden_glia_res1.5" or "leiden_neurons_res1.5"
    prefix = f"leiden_{group_prefix.lower()}_res"
    matches = [c for c in adata.obs.columns if c.startswith(prefix)]
    if not matches:
        raise KeyError(f"No Leiden column starting with '{prefix}' found in adata.obs.")
    if len(matches) > 1:
        # If multiple (unlikely), pick the last one created
        matches.sort()
    return matches[-1]

glia_leiden_key   = find_leiden_key(adata_glia_subclustered,   "Glia")
neuron_leiden_key = find_leiden_key(adata_neurons_subclustered, "Neurons")

# --- Your subcluster → cell-type maps ---
glia_map = {
    # Astro
    '0':'Astro','3':'Astro','19':'Astro','26':'Astro',
    # Oligo
    '1':'Oligo','2':'Oligo','5':'Oligo','6':'Oligo','8':'Oligo',
    '9':'Oligo','10':'Oligo','18':'Oligo','23':'Oligo',
    # Microglia
    '4':'Microglia','11':'Microglia','15':'Microglia','21':'Microglia'
}

neuron_map = {
    # DG
    '0':'DG','1':'DG','3':'DG','17':'DG',
    # CA3
    '2':'CA3','12':'CA3',
    # CA1
    '4':'CA1','5':'CA1','6':'CA1','10':'CA1','13':'CA1',
    # Inhibitory
    '7':'Inhibitory','8':'Inhibitory','9':'Inhibitory','11':'Inhibitory'
}

# --- Build labeled Series from subclustered AnnData objects (index = cell barcodes) ---
glia_labels = adata_glia_subclustered.obs[glia_leiden_key].map(glia_map).dropna()
neuron_labels = adata_neurons_subclustered.obs[neuron_leiden_key].map(neuron_map).dropna()

# Merge; if any overlap, keep Glia’s label first (adjust keep='first' if you prefer Neurons)
cell_type_labels = pd.concat([glia_labels, neuron_labels])
cell_type_labels = cell_type_labels[~cell_type_labels.index.duplicated(keep='first')]

print(f"Total labeled cells: {len(cell_type_labels)}")
print("Label counts:\n", cell_type_labels.value_counts())

# --- Apply labels to BOTH objects (HVG-only and all-genes) ---
# 1) adata_filtered (HVGs)
idx_filtered = adata_filtered.obs_names.intersection(cell_type_labels.index)
adata_filtered.obs.loc[idx_filtered, "cell_type"] = cell_type_labels.loc[idx_filtered].values
adata_filtered_masked = adata_filtered[idx_filtered].copy()

print(f"\nadata_filtered: kept {adata_filtered_masked.n_obs} / {adata_filtered.n_obs} cells")
print("adata_filtered cell_type counts:\n", adata_filtered_masked.obs["cell_type"].value_counts())

# 2) adata_combined_all_genes (all genes)
idx_all = adata_combined_all_genes.obs_names.intersection(cell_type_labels.index)
adata_combined_all_genes.obs.loc[idx_all, "cell_type"] = cell_type_labels.loc[idx_all].values
adata_combined_all_genes_masked = adata_combined_all_genes[idx_all].copy()

print(f"\nadata_combined_all_genes: kept {adata_combined_all_genes_masked.n_obs} / {adata_combined_all_genes.n_obs} cells")
print("adata_combined_all_genes cell_type counts:\n", adata_combined_all_genes_masked.obs["cell_type"].value_counts())


# --- Build boolean masks for each object ---
mask_filtered = adata_filtered.obs_names.isin(cell_type_labels.index)
mask_all      = adata_combined_all_genes.obs_names.isin(cell_type_labels.index)

# --- Subset in place (overwrite the original objects) ---
adata_filtered = adata_filtered[mask_filtered].copy()
adata_combined = adata_filtered.copy()
adata_combined_all_genes = adata_combined_all_genes[mask_all].copy()

# --- Add the cell_type column (now guaranteed to exist for all cells) ---
adata_filtered.obs["cell_type"] = cell_type_labels.loc[adata_filtered.obs_names].values
adata_combined_all_genes.obs["cell_type"] = cell_type_labels.loc[adata_combined_all_genes.obs_names].values

# --- Quick check ---
print(f"adata_filtered new shape: {adata_filtered.shape}")
print(adata_filtered.obs["cell_type"].value_counts())

print(f"\nadata_combined_all_genes new shape: {adata_combined_all_genes.shape}")
print(adata_combined_all_genes.obs["cell_type"].value_counts())


In [ ]:
# print(f"{adata_filtered_ALL_genes.obs['celltype'].unique()}")
print(f"{adata_combined_all_genes.obs['cell_type'].unique()}")


### 5. Assign cell types
Maps Leiden cluster indices to the seven annotated populations and excludes clusters that could not be confidently assigned.

In [ ]:
# # Clusters marked with REMOVE are excluded
# clusters_to_keep = list(cluster_to_celltype.keys())

# # Subset the AnnData object to include only relevant clusters
# adata_filtered = adata_filtered_whole[adata_filtered_whole.obs['leiden'].isin(clusters_to_keep)].copy()

# Add new celltype annotations based on cluster mapping

adata_filtered.obs['celltype'] = adata_filtered.obs["cell_type"].copy()#adata_filtered.obs['leiden'].map(cluster_to_celltype)

sc.pl.umap(adata_filtered, color=["celltype"],legend_loc='on data', save='Annotated_Final_Clustering_YekBasteh.png')


In [ ]:
sc.pl.umap(adata_filtered, color=["leiden"],legend_loc='on data')#,


sc.pl.umap(adata_filtered, color=["leiden"])#,legend_loc='on data'

sc.pl.umap(adata_filtered, color=["group", "condition","sex","batch"],legend_loc='on data')


In [ ]:
# adata_filtered_ALL_genes.obs['celltype'] = (
#     adata_filtered.obs['celltype'].reindex(adata_filtered_ALL_genes.obs_names)
# )

# all_cats = ['Astro','CA1','CA3','DG','Inhibitory','Microglia','Oligo']
# adata_filtered_ALL_genes.obs['celltype'] = (
#     adata_filtered_ALL_genes.obs['celltype']
#       .astype('category')
#       .cat.set_categories(all_cats)
# )

adata_combined_all_genes.obs['celltype'] = adata_combined_all_genes.obs["cell_type"].copy()
print(f"adata_combined_all_genes {adata_combined_all_genes.obs['celltype'].unique()}")

adata_filtered_ALL_genes = adata_combined_all_genes.copy()

# adata_filtered_ALL_genes = adata_combined[~adata_combined.obs['leiden'].isin(clusters_to_remove)].copy()
#adata_filtered[:, adata_filtered.var_names.isin(significant_genes)].copy()
adata_filtered_ALL_genes.obs['celltype'] = adata_filtered.obs['celltype']

# set_of_celltype_names = ['Microglia','Astro', 'CA1', 'CA3', 'Oligo', 'DG', 'Inhibitory']
# adata_filtered_significant_genes = adata_filtered_ALL_genes[adata_filtered_ALL_genes.obs['celltype'].isin(set_of_celltype_names)].copy()

adata_filtered_significant_genes = adata_filtered_ALL_genes.copy()

# *** since th scope of this project is lacking cd47, we keep it out
gene_to_remove = 'Cd47'
# Keep all genes except the one you want to remove
adata_filtered_significant_genes = adata_filtered_significant_genes[:, adata_filtered_significant_genes.var_names != gene_to_remove].copy()
# Save the new AnnData object
adata_filtered_significant_genes.write("adata_filtered_significant_genes_NO_mt_YekBasteh.h5ad")
print(f"{adata_filtered_ALL_genes.obs['celltype'].unique()}")
print(f"{adata_filtered_significant_genes.obs['celltype'].unique()}")

print(f"adata_filtered_significant_genes saved successfully with {adata_filtered_significant_genes.shape[1]} significant genes.")

### 6. Load saved objects
Reads the processed AnnData objects for downstream analysis.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import time
import os
from harmony import harmonize
import pickle
###  Perform Trajectory Inference

# if os.path.exists(f"adata_filtered_0.1_batch_sex.h5ad"):
#     # Load the existing file
#     print("File exists ... Loading adata_combined from file...")
#     adata_filtered = sc.read_h5ad(f"adata_filtered_0.1_batch_sex.h5ad")
# else:
#     print("File DOES NOT exist!\n Running the program ...")
#     # Initialize an empty dictionary to store adata objects for each sample
    

###  Perform Trajectory Inference
print()

# Assign pseudotime
sc.tl.pca(adata_filtered_significant_genes, svd_solver='arpack', n_comps=50)

# 2) Neighbors (this creates the graph diffmap needs)
sc.pp.neighbors(adata_filtered_significant_genes, n_neighbors=15, n_pcs=30)  # adjust as needed

# 3) Diffusion map (now it will work)
sc.tl.diffmap(adata_filtered_significant_genes, n_comps=50)
sc.tl.umap(adata_filtered_significant_genes)

sc.pl.umap(adata_filtered_significant_genes, color='celltype', legend_loc='on data')

print(f"here")
# sc.tl.paga(adata_filtered_significant_genes, groups='celltype')
# sc.pl.paga(adata_filtered_significant_genes, color=["celltype"])



In [ ]:
# Extract top N genes for a specific cluster
def get_top_genes_from_rank(adata, n_genes=50, pval_threshold=0.01):
    """
    Extract top genes from adata.uns['rank_genes_groups'] as a DataFrame.

    Parameters:
        adata : AnnData object after rank_genes_groups analysis.
        n_genes : int, number of top genes to return.
        pval_threshold : float, adjusted p-value threshold for filtering genes.

    Returns:
        List of top genes based on adjusted p-value and significance.
    """
    # Extract results as a DataFrame using scanpy helper
    result_df = sc.get.rank_genes_groups_df(adata, group=None)  # Extract for all groups
    
    # Filter for significant genes based on p-value
    filtered_df = result_df[result_df['pvals_adj'] < pval_threshold]
    
    # Return top N genes
    top_genes = filtered_df['names'].head(n_genes).tolist()

    return top_genes

### 7. Nuclei counts
Tabulates retained nuclei per mouse and per cell type (Supplementary Tables 1.2, 1.3, 1.5).

In [ ]:
# =============================================================================
# SEQUENCING / QC TABLES REQUESTED BY REVIEWER 1, COMMENT 3
#
# PLACEMENT: insert as a NEW CELL immediately AFTER cell 16
#            (the cell ending "adata_filtered_significant_genes saved
#            successfully with ... significant genes.")
#            i.e. BEFORE the markdown cell "## Trajectory & READING FILES".
#
#            At that point adata_filtered_significant_genes carries all genes,
#            full .obs, and the celltype labels, and adata_combined_all_genes
#            has not yet been overwritten in cell 21.
#
# PRODUCES
#   Table A  nuclei per mouse x cell type, with per-brain totals
#   Table B  per-mouse QC and sequencing depth
#   Table C  depth per cell type x mouse (median UMI / median genes)
#   Table D  nuclei and depth per cell type x genotype x treatment
#
# NOTE ON DEPTH
#   total_counts / n_genes_by_counts were computed by sc.pp.calculate_qc_metrics
#   in cell 0 on the RAW per-sample matrix, one value per barcode, so they
#   survive filtering, HVG selection, PCA, Harmony and annotation untouched.
#   Because every barcode also carries a celltype label, depth can be grouped
#   by cell type. Where pseudobulk_adata.layers["counts"] is available the
#   depth is recomputed from it so the values exclude mitochondrial genes.
#
#   Read-level metrics (total reads, mean reads per nucleus, sequencing
#   saturation, fraction reads in cells) are library-level CellRanger outputs
#   and cannot be attributed to individual barcodes or cell types. Set CR_DIR
#   below to merge them at the per-mouse level.
# =============================================================================

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp

# ---- config -----------------------------------------------------------------
ADATA_ANN     = adata_filtered_significant_genes   # annotated, all genes
SAMPLE_KEY    = "sample_id_unique"
CELLTYPE_KEY  = "celltype"
GROUP_KEY     = "group"
CONDITION_KEY = "condition"

CELLTYPE_ORDER = ["Astro", "CA1", "CA3", "DG", "Inhibitory", "Microglia", "Oligo"]
COND_ORDER     = ["Veh", "KA3", "KA25"]

# Minimum nuclei per mouse x cell type you required for network construction.
MIN_NUCLEI_FOR_NETWORK = 50

# CellRanger run directories, for the read-level metrics. "" to skip.
CR_DIR = "/home/kiamari/Single_cell/Single_cell_datasets/"

OUTDIR = "QC_tables_revision"
os.makedirs(OUTDIR, exist_ok=True)


# ---- 1. raw per-nucleus depth ------------------------------------------------
obs = ADATA_ANN.obs

umi = None
genes = None
if "pseudobulk_adata" in dir() and "counts" in pseudobulk_adata.layers:
    shared = obs.index.intersection(pseudobulk_adata.obs_names)
    if len(shared) == ADATA_ANN.n_obs:
        M = pseudobulk_adata[obs.index].layers["counts"]
        M = M if sp.issparse(M) else sp.csr_matrix(M)
        umi   = pd.Series(np.asarray(M.sum(axis=1)).ravel(), index=obs.index)
        genes = pd.Series(np.asarray((M > 0).sum(axis=1)).ravel(), index=obs.index)
        print("Depth recomputed from pseudobulk_adata.layers['counts'] "
              "(mitochondrial genes already removed).")

if umi is None:
    umi   = obs["total_counts"].astype(float)
    genes = obs["n_genes_by_counts"].astype(float)
    print("Depth taken from .obs['total_counts'] / .obs['n_genes_by_counts'] "
          "(raw counts as computed in cell 0).")


# ---- 2. tidy nucleus-level frame --------------------------------------------
nuc = pd.DataFrame({
    "sample":    obs[SAMPLE_KEY].astype(str).str.replace("batch_2022_", "",
                                                         regex=False),
    "group":     obs[GROUP_KEY].astype(str),
    "condition": obs[CONDITION_KEY].astype(str),
    "sex":       obs["sex"].astype(str),
    "celltype":  obs[CELLTYPE_KEY].astype("object"),
    "umi":       umi.values,
    "genes":     genes.values,
    "pct_mt":    obs["pct_counts_mt"].astype(float).values,
}, index=obs.index)

nuc["condition"] = pd.Categorical(nuc["condition"], COND_ORDER, ordered=True)

ann = nuc[nuc["celltype"].notna()].copy()          # annotated nuclei only

print(f"\nNuclei passing QC        : {len(nuc):,}")
print(f"Nuclei assigned a cell type: {len(ann):,} "
      f"({100*len(ann)/len(nuc):.1f}%)")
print(f"Mice                     : {nuc['sample'].nunique()}")
print(nuc.groupby(["condition", "group"], observed=True)["sample"]
         .nunique().to_string())


# ---- 3. TABLE A  nuclei per mouse x cell type -------------------------------
tabA = (
    ann.pivot_table(index=["condition", "group", "sample"],
                    columns="celltype", values="umi",
                    aggfunc="size", fill_value=0, observed=True)
       .reindex(columns=CELLTYPE_ORDER, fill_value=0)
       .astype(int)
)
tabA.insert(len(tabA.columns), "Total_annotated", tabA.sum(axis=1))
tabA["Total_post_QC"] = (
    nuc.groupby(["condition", "group", "sample"], observed=True).size()
       .reindex(tabA.index).astype(int)
)
tabA.loc[("All", "All", "All"), :] = tabA.sum(axis=0)
tabA = tabA.astype(int)

print("\n" + "=" * 78)
print("TABLE A   nuclei recovered per mouse and per cell type")
print("=" * 78)
print(tabA.to_string())


# ---- 4. TABLE B  per-mouse QC and depth -------------------------------------
tabB = (
    ann.groupby(["condition", "group", "sample"], observed=True)
       .agg(sex              =("sex", "first"),
            nuclei_annotated =("umi", "size"),
            median_UMI       =("umi", "median"),
            mean_UMI         =("umi", "mean"),
            median_genes     =("genes", "median"),
            mean_genes       =("genes", "mean"),
            median_pct_mt    =("pct_mt", "median"))
       .reset_index()
)
post_qc = (nuc.groupby(["condition", "group", "sample"], observed=True)
              .size().rename("nuclei_post_QC").reset_index())
tabB = tabB.merge(post_qc, on=["condition", "group", "sample"], how="left")
tabB["pct_annotated"] = (100 * tabB["nuclei_annotated"]
                         / tabB["nuclei_post_QC"]).round(1)

for c in ["median_UMI", "mean_UMI", "median_genes", "mean_genes"]:
    tabB[c] = tabB[c].round(0).astype(int)
tabB["median_pct_mt"] = tabB["median_pct_mt"].round(2)

tabB = tabB[["condition", "group", "sample", "sex",
             "nuclei_post_QC", "nuclei_annotated", "pct_annotated",
             "median_UMI", "mean_UMI", "median_genes", "mean_genes",
             "median_pct_mt"]]

print("\n" + "=" * 78)
print("TABLE B   per-mouse quality control and sequencing depth")
print("=" * 78)
print(tabB.to_string(index=False))


# ---- 5. TABLE C  depth per cell type x mouse --------------------------------
tabC = (
    ann.groupby(["condition", "group", "sample", "celltype"], observed=True)
       .agg(n_nuclei     =("umi", "size"),
            median_UMI   =("umi", "median"),
            median_genes =("genes", "median"))
       .reset_index()
)
tabC["median_UMI"]   = tabC["median_UMI"].round(0).astype(int)
tabC["median_genes"] = tabC["median_genes"].round(0).astype(int)
tabC["below_threshold"] = tabC["n_nuclei"] < MIN_NUCLEI_FOR_NETWORK

print("\n" + "=" * 78)
print("TABLE C   median UMI per nucleus, cell type x mouse")
print("=" * 78)
print(tabC.pivot_table(index=["condition", "group", "sample"],
                       columns="celltype", values="median_UMI",
                       observed=True)
          .reindex(columns=CELLTYPE_ORDER).astype("Int64").to_string())

print("\nmedian genes detected per nucleus, cell type x mouse")
print(tabC.pivot_table(index=["condition", "group", "sample"],
                       columns="celltype", values="median_genes",
                       observed=True)
          .reindex(columns=CELLTYPE_ORDER).astype("Int64").to_string())

n_below = int(tabC["below_threshold"].sum())
print(f"\nmouse x cell-type strata with < {MIN_NUCLEI_FOR_NETWORK} nuclei: "
      f"{n_below} of {len(tabC)}")
if n_below:
    print(tabC.loc[tabC["below_threshold"],
                   ["condition", "group", "sample", "celltype",
                    "n_nuclei", "median_UMI"]].to_string(index=False))


# ---- 6. TABLE D  cell type x genotype x treatment ---------------------------
tabD = (
    ann.groupby(["celltype", "condition", "group"], observed=True)
       .agg(n_mice        =("sample", "nunique"),
            n_nuclei      =("umi", "size"),
            min_per_mouse =("umi", "size"),
            median_UMI    =("umi", "median"),
            median_genes  =("genes", "median"))
       .reset_index()
)
per_mouse_min = (
    ann.groupby(["celltype", "condition", "group", "sample"], observed=True)
       .size().groupby(level=[0, 1, 2], observed=True).min()
       .rename("min_per_mouse").reset_index()
)
tabD = (tabD.drop(columns="min_per_mouse")
            .merge(per_mouse_min, on=["celltype", "condition", "group"],
                   how="left"))
tabD["median_UMI"]   = tabD["median_UMI"].round(0).astype(int)
tabD["median_genes"] = tabD["median_genes"].round(0).astype(int)
tabD = tabD.sort_values(["celltype", "condition", "group"])

print("\n" + "=" * 78)
print("TABLE D   nuclei and depth per cell type x genotype x treatment")
print("=" * 78)
print(tabD.to_string(index=False))


# ---- 7. optional CellRanger read-level metrics ------------------------------
cr = None
if CR_DIR:
    rows = []
    for samp in tabB["sample"]:
        for cand in (os.path.join(CR_DIR, samp, "outs", "metrics_summary.csv"),
                     os.path.join(CR_DIR, samp + "_metrics_summary.csv"),
                     os.path.join(CR_DIR, samp, "metrics_summary.csv")):
            if os.path.exists(cand):
                m = pd.read_csv(cand)
                m.insert(0, "sample", samp)
                rows.append(m)
                break
    if rows:
        cr = pd.concat(rows, ignore_index=True)
        wanted = ["number of reads", "mean reads", "saturation",
                  "reads in cells", "confidently to the transcriptome",
                  "valid barcodes", "estimated number of cells", "q30"]
        cr = cr[["sample"] + [c for c in cr.columns
                              if c != "sample"
                              and any(k in c.lower() for k in wanted)]]
        tabB = tabB.merge(cr, on="sample", how="left")
        print("\n" + "=" * 78)
        print("CellRanger read-level metrics (library level)")
        print("=" * 78)
        print(cr.to_string(index=False))

if cr is None:
    print("\nNo metrics_summary.csv found. Total reads, mean reads per "
          "nucleus, sequencing saturation and fraction reads in cells must be "
          "taken from the CellRanger web_summary / metrics_summary files; "
          "they are library-level and cannot be split by cell type.")


# # ---- 8. write out -----------------------------------------------------------
# xlsx = os.path.join(OUTDIR, "SupplementaryTables_QC_R1C3.xlsx")
# with pd.ExcelWriter(xlsx) as w:
#     tabA.to_excel(w, sheet_name="A_nuclei_mouse_x_celltype")
#     tabB.to_excel(w, sheet_name="B_per_mouse_QC", index=False)
#     tabC.to_excel(w, sheet_name="C_depth_celltype_x_mouse", index=False)
#     tabD.to_excel(w, sheet_name="D_celltype_x_genotype_x_trt", index=False)

# for name, df, idx in [("A_nuclei_mouse_x_celltype", tabA, True),
#                       ("B_per_mouse_QC", tabB, False),
#                       ("C_depth_celltype_x_mouse", tabC, False),
#                       ("D_celltype_x_genotype_x_trt", tabD, False)]:
#     df.to_csv(os.path.join(OUTDIR, name + ".csv"), index=idx)

# print(f"\nSaved: {xlsx}")
# print(f"Saved: CSV copies in {OUTDIR}/")


# # ---- 9. LaTeX for the supplement -------------------------------------------
# # pandas .to_latex needs jinja2 >= 3.1.2; skipped silently if unavailable.
# try:
#     print("\n" + "=" * 78)
#     print("LaTeX  Table A")
#     print("=" * 78)
#     print(tabA.to_latex(
#         caption=("Nuclei recovered per mouse after quality control and per "
#                  "annotated cell type."), label="tab:S1"))

#     print("=" * 78)
#     print("LaTeX  Table B")
#     print("=" * 78)
#     print(tabB.to_latex(index=False, float_format="%.1f",
#         caption="Per-mouse quality-control and sequencing-depth metrics.",
#         label="tab:S2"))
# except ImportError as e:
#     print(f"\nLaTeX export skipped ({e}). The CSV/XLSX files above contain "
#           f"the same tables.")


### 8. Approach 1 - nucleus-level differential expression, vehicle condition
Compares Cd47 KO with WT within the vehicle arm for each cell type and draws the volcano plots (Fig. 2C, Figs. S1).

In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

sex_genes_to_remove = ["Xist", "Tsix", "Uty", "Eif2s3y", "Kdm5d", "Ddx3y"]


for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['condition'] == 'Veh']
    # Perform differential expression analysis
    sc.tl.rank_genes_groups(cluster_data, groupby='group',reference='WT', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'Veh_Vali_{name_validation}_{cluster}_YekBasteh.png')

    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------
    

    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])


        # Calculate -log10(p-values)
        neg_log_pvals = -np.log10(p_values_array)

        ######## CHANGED !!
        # # Apply significance threshold
        # significant = neg_log_pvals > 1.5
        # sig_log2fc_values = log2fc_values_array[significant]
        # sig_neg_log_pvals = neg_log_pvals[significant]
        # sig_genes = genes_array[significant]

        ####################
        # Apply significance threshold
        significant = neg_log_pvals > 1.5
        
        # Filter fold change range
        # Filter fold change range:
        log2fc_range_mask = ((log2fc_values_array > 0) & (log2fc_values_array < 5.0)) | \
                            ((log2fc_values_array < -0) & (log2fc_values_array > -5.0))

        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)

        # Combine both masks
        valid_mask = significant & log2fc_range_mask & sex_gene_mask
        
        # Apply to arrays
        sig_log2fc_values = log2fc_values_array[valid_mask]
        sig_neg_log_pvals = neg_log_pvals[valid_mask]
        sig_genes = genes_array[valid_mask]

        ####################

        
        # Count upregulated and downregulated genes
        up_count = np.sum(sig_log2fc_values > 0)
        down_count = np.sum(sig_log2fc_values < 0)
    
        up_counts.append(up_count)
        down_counts.append(down_count)
    

        ### ------- NEW Adjust Text in Plots -----------
        # --- Volcano Plot with Gene Annotations (Corrected Version) ---
        from adjustText import adjust_text  # <-- 1. IMPORT THE LIBRARY
        
        plt.figure(figsize=(10, 8)) # Increased figure size for better spacing
        
        # Scatter plot: all significant genes
        plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.5, s=10) # s=10 for smaller dots
        
        # Highlight significant upregulated genes (red)
        upregulated = sig_log2fc_values > 0
        plt.scatter(
            sig_log2fc_values[upregulated], 
            sig_neg_log_pvals[upregulated], 
            color="red", label="Upregulated", s=15
        )
        
        # Highlight significant downregulated genes (blue)
        downregulated = sig_log2fc_values < 0
        plt.scatter(
            sig_log2fc_values[downregulated], 
            sig_neg_log_pvals[downregulated], 
            color="blue", label="Downregulated", s=15
        )
        
        # --- ANNOTATION CHANGES START HERE ---
        
        # 2. CREATE AN EMPTY LIST TO HOLD TEXT OBJECTS
        texts_to_adjust = []
        top_n_genes = 20
        
        # Sort by p-values to select the top genes to annotate
        # Upregulated genes
        if np.any(upregulated):
            sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
            for idx in sorted_idx_up:
                gene_name = sig_genes[upregulated][idx]
                x = sig_log2fc_values[upregulated][idx]
                y = sig_neg_log_pvals[upregulated][idx]
                # 3. INSTEAD OF PLOTTING, APPEND THE TEXT OBJECT TO THE LIST
                texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        
        # Downregulated genes
        if np.any(downregulated):
            sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
            for idx in sorted_idx_down:
                gene_name = sig_genes[downregulated][idx]
                x = sig_log2fc_values[downregulated][idx]
                y = sig_neg_log_pvals[downregulated][idx]
                # 3. APPEND THE TEXT OBJECT TO THE LIST
                texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        print(f"Veh - Downregulated {'\n'.join(sig_genes[downregulated][-top_n_genes:])}")
        print(f"Veh - Upregulated {'\n'.join(sig_genes[upregulated][-top_n_genes:])}")
        # 4. CALL adjust_text ONCE TO AUTOMATICALLY POSITION LABELS AND DRAW LINES
        if texts_to_adjust:
            adjust_text(texts_to_adjust, 
                        arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
                        ax=plt.gca()) # Specify the axes
        
        # --- ANNOTATION CHANGES END HERE ---
        
        # Add significance threshold lines
        plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change")
        plt.ylabel("-log10(p-value)")
        plt.title(f"Volcano Plot for Cell Type {cluster} under Vehicle (Veh) Condition")
        plt.legend()
        plt.tight_layout() # Helps fit everything neatly
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_Veh_YekBasteh.png")

        plt.show()
        ### ------ END of Adjust text to plots ---------------

        # # ----------------- Volcano Plot with Gene Annotations ----------------------
        # plt.figure(figsize=(8, 6))
    
        # # Scatter plot: significant genes only
        # plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.6)
    
        # # Highlight significant upregulated genes (red)
        # upregulated = sig_log2fc_values > 0
        # plt.scatter(
        #     sig_log2fc_values[upregulated], 
        #     sig_neg_log_pvals[upregulated], 
        #     color="red", label="Upregulated"
        # )
    
        # # Highlight significant downregulated genes (blue)
        # downregulated = sig_log2fc_values < 0
        # plt.scatter(
        #     sig_log2fc_values[downregulated], 
        #     sig_neg_log_pvals[downregulated], 
        #     color="blue", label="Downregulated"
        # )
    
        # # Annotate top upregulated and downregulated genes
        # top_n_genes = 20
        
        # # Sort by p-values and annotate
        # sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
        # sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
    
        # for idx in sorted_idx_up:
        #     gene_name = sig_genes[upregulated][idx]
        #     plt.text(sig_log2fc_values[upregulated][idx], sig_neg_log_pvals[upregulated][idx], 
        #              gene_name, fontsize=9, ha='left', va='bottom', color='black')
    
        # for idx in sorted_idx_down:
        #     gene_name = sig_genes[downregulated][idx]
        #     plt.text(sig_log2fc_values[downregulated][idx], sig_neg_log_pvals[downregulated][idx], 
        #              gene_name, fontsize=9, ha='right', va='bottom', color='black')
    
        # # Add significance threshold lines
        # plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        # plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
    
        # plt.xlabel("log2 Fold Change")
        # plt.ylabel("-log10(p-value)")
        # plt.title(f"Volcano Plot for Cell Type {cluster} under Vehicle (Veh) Condition")
        # plt.legend()
        # plt.grid(True, linestyle='--', alpha=0.6)
        # plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_Veh_YekBasteh.png")
        # plt.show()
























        
    # for group_name in group_names:
    #     # Convert to dictionary for easy retrieval
    #     log2fc_dict[cluster][group_name] = {gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])}
    #     #print(f"group_name{group_name} cluster {cluster}: {log2fc_dict[cluster][group_name]}")
    #     p_values_dict[cluster][group_name]={gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])}
    #     # Convert dict values to NumPy array
    #     log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
    #     p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
    #     genes_array = np.array(cluster_genes[group_name]) 
    #     # Count upregulated and downregulated genes
    #     up_count = np.sum(log2fc_values_array > 0)  # Count upregulated genes
    #     down_count = np.sum(log2fc_values_array < 0)  # Count downregulated genes

    #     up_counts.append(up_count)
    #     down_counts.append(down_count)

    #     #print(f"Cluster {cluster}, Condition {group_name}: {up_count} upregulated, {down_count} downregulated genes.")
    #     # -----------------  Volcano Plot with Gene Annotations  ----------------------
    #     plt.figure(figsize=(8, 6))

    #     # Scatter plot: All genes (gray)
    #     plt.scatter(log2fc_values_array, -np.log10(p_values_array), color="grey", alpha=0.6)

    #     # Highlight upregulated genes (red)
    #     upregulated = log2fc_values_array > 0
    #     plt.scatter(
    #         log2fc_values_array[upregulated], 
    #         -np.log10(p_values_array[upregulated]), 
    #         color="red", label="Upregulated"
    #     )

    #     # Highlight downregulated genes (blue)
    #     downregulated = log2fc_values_array < 0
    #     plt.scatter(
    #         log2fc_values_array[downregulated], 
    #         -np.log10(p_values_array[downregulated]), 
    #         color="blue", label="Downregulated"
    #     )

    #     # Annotate top upregulated and downregulated genes
    #     top_n_genes = 10  # Adjust number of labels for each side
        
    #     # Separate significant upregulated and downregulated genes
    #     sorted_idx_up = np.argsort(p_values_array[upregulated])[:top_n_genes]  # Select top upregulated genes
    #     sorted_idx_down = np.argsort(p_values_array[downregulated])[:top_n_genes]  # Select top downregulated genes
        
    #     # Label upregulated genes on the right side of the dashed line (log2FC > 0)
    #     for idx in sorted_idx_up:
    #         gene_name = genes_array[upregulated][idx]  # Extract gene name
    #         plt.text(log2fc_values_array[upregulated][idx], -np.log10(p_values_array[upregulated][idx]), 
    #                  gene_name, fontsize=9, ha='left', va='bottom', color='black')  # Align text to the left
        
    #     # Label downregulated genes on the left side of the dashed line (log2FC < 0)
    #     for idx in sorted_idx_down:
    #         gene_name = genes_array[downregulated][idx]  # Extract gene name
    #         plt.text(log2fc_values_array[downregulated][idx], -np.log10(p_values_array[downregulated][idx]), 
    #                  gene_name, fontsize=9, ha='right', va='bottom', color='black')  # Align text to the right
        
    #     # Add significance threshold lines
    #     plt.axhline(y=-np.log10(0.05), linestyle="dashed", color="black", alpha=0.7)  # p-value threshold
    #     plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)  # No change threshold
        
    #     plt.xlabel("log2 Fold Change")
    #     plt.ylabel("-log10(p-value)")
    #     plt.title(f"Volcano Plot - Cluster {cluster} ({group_name} vs. Veh)")
    #     plt.legend()
    #     plt.show()
    



## Low dose Injection Comparsion

In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

sex_genes_to_remove = ["Xist", "Tsix", "Uty", "Eif2s3y", "Kdm5d", "Ddx3y"]

up_genes_Veh = {}
down_genes_Veh = {}

for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['condition'] == 'Veh']
    # Perform differential expression analysis
    sc.tl.rank_genes_groups(cluster_data, groupby='group',reference='WT', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'Veh_Vali_{name_validation}_{cluster}_YekBasteh.png')

    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------
    

    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])
        
        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
        # Calculate -log10(p-value) and handle p-values of 0
        with np.errstate(divide='ignore'):
            neg_log_pvals = -np.log10(p_values_array)
        neg_log_pvals[np.isinf(neg_log_pvals)] = np.nanmax(neg_log_pvals[np.isfinite(neg_log_pvals)]) # Replace inf with max finite value
        neg_log_pvals = np.nan_to_num(neg_log_pvals) # Replace any remaining NaNs with 0

        # --- 1. DEFINE THRESHOLDS ---
        LOG2FC_THRESHOLD = 0.5
        P_ADJ_THRESHOLD = 0.05
        NEG_LOG_P_THRESHOLD = -np.log10(P_ADJ_THRESHOLD)

        # --- 2. CREATE BOOLEAN MASKS FOR EACH CATEGORY ---
        
        # Category 1: Significant AND Upregulated
        up_mask = (log2fc_values_array > LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 2: Significant AND Downregulated
        down_mask = (log2fc_values_array < -LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 3: Significant but low fold-change
        sig_low_fc_mask = (p_values_array < P_ADJ_THRESHOLD) & ~up_mask & ~down_mask & sex_gene_mask

        # Category 4: Not significant (all others)
        not_sig_mask = ~(up_mask | down_mask | sig_low_fc_mask)

        # --- 3. PLOT EACH CATEGORY ---
        plt.figure(figsize=(12, 10))
        
        # # Plot 1: Not significant -> Grey
        # plt.scatter(log2fc_values_array[not_sig_mask], neg_log_pvals[not_sig_mask], 
        #             color="grey", alpha=0.4, s=10, label=f'p ≥ {P_ADJ_THRESHOLD}')
        
        # Plot 2: Significant, but low fold-change -> Orange
        plt.scatter(log2fc_values_array[sig_low_fc_mask], neg_log_pvals[sig_low_fc_mask], 
                    color="orange", s=15, alpha=0.7, label=f'p < {P_ADJ_THRESHOLD} & |log2FC| < {LOG2FC_THRESHOLD}')
        
        # Plot 3: Significant AND Downregulated -> Blue
        plt.scatter(log2fc_values_array[down_mask], neg_log_pvals[down_mask], 
                    color="blue", s=20, label=f'Downregulated')
        
        # Plot 4: Significant AND Upregulated -> Red
        plt.scatter(log2fc_values_array[up_mask], neg_log_pvals[up_mask], 
                    color="red", s=20, label=f'Upregulated')

        # --- 4. ANNOTATE TOP GENES ---
        texts_to_adjust = []
        top_n_genes = 15
        
        # Annotate top Upregulated genes (from the red points)
        if np.any(up_mask):
            up_genes, up_pvals, up_fc = genes_array[up_mask], neg_log_pvals[up_mask], log2fc_values_array[up_mask]
            sorted_idx_up = np.argsort(up_pvals)[-top_n_genes:]
            for idx in sorted_idx_up:
                texts_to_adjust.append(plt.text(up_fc[idx], up_pvals[idx], up_genes[idx], fontsize=20))

        # Annotate top Downregulated genes (from the blue points)
        if np.any(down_mask):
            down_genes, down_pvals, down_fc = genes_array[down_mask], neg_log_pvals[down_mask], log2fc_values_array[down_mask]
            sorted_idx_down = np.argsort(down_pvals)[-top_n_genes:]
            for idx in sorted_idx_down:
                texts_to_adjust.append(plt.text(down_fc[idx], down_pvals[idx], down_genes[idx], fontsize=20))

        if texts_to_adjust:
            # adjust_text(texts_to_adjust, 
            #             arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
            #             ax=plt.gca())
            adjust_text(
                texts_to_adjust,
                ax=plt.gca(),
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
                force_points=2.9,#0.5,        # how strongly labels avoid points
                force_text=2.9,#0.7,          # how strongly labels avoid each other
                expand_points=(1.2, 1.4), # extra padding around points
                expand_text=(1.2, 1.4),   # extra padding around texts
                only_move={'points': 'y', 'text': 'xy'},  # allow free movement in both x & y
                lim=500,                 # number of iterations for optimization
            )



        
        print(f"Veh - Downregulated {down_genes}")
        print(f"Veh - Upregulated {up_genes}")

        
        up_genes_Veh[cluster] = up_genes
        down_genes_Veh[cluster] = down_genes


        
        # --- 5. ADD THRESHOLD LINES AND LABELS ---
        plt.axhline(y=NEG_LOG_P_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=-LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change", fontsize=22,fontweight='bold')
        plt.ylabel("-log10(Adjusted p-value)", fontsize=22,fontweight='bold')
        # plt.title(f"Volcano Plot: {cluster} ({group_name} vs. WT in Vehicle)", fontsize=16, fontweight='bold')
        plt.title(f"Volcano Plot for {cluster} Population - saline 0.9% (Veh) Condition", fontsize=22, fontweight='bold')

        #plt.legend(loc='upper left')
        plt.legend(
            loc='upper left',
            fontsize=18,        # Increase legend text size
            markerscale=2.0,    # Scale up legend markers (optional)
            frameon=True,       # Keep a frame (set to False to remove)
            framealpha=0.8,     # Slight transparency
            handlelength=1.5,   # Length of the color/marker handles
            title_fontsize=18    # Font size for legend title
        )

        plt.tick_params(axis='both', which='major', labelsize=20)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        
        # Save and show the plot
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_Veh_YekBasteh.png", dpi=300)
        plt.show()


In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['condition'] == 'KA3']
    # Perform differential expression analysis
    sc.tl.rank_genes_groups(cluster_data, groupby='group',reference='WT', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'KA3_Vali_{name_validation}_{cluster}_YekBasteh.png')

    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------
    

    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])
        
        # Calculate -log10(p-values)
        neg_log_pvals = -np.log10(p_values_array)
    
        # Apply significance threshold
        significant = neg_log_pvals > 1.5

        log2fc_range_mask = ((log2fc_values_array > 0) & (log2fc_values_array < 5.0)) | \
                            ((log2fc_values_array < -0) & (log2fc_values_array > -5.0))


        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
        
        # Combine both masks
        valid_mask = significant & log2fc_range_mask & sex_gene_mask
        
        # Apply to arrays
        sig_log2fc_values = log2fc_values_array[valid_mask]
        sig_neg_log_pvals = neg_log_pvals[valid_mask]
        sig_genes = genes_array[valid_mask]

        
        # sig_log2fc_values = log2fc_values_array[significant]
        # sig_neg_log_pvals = neg_log_pvals[significant]
        # sig_genes = genes_array[significant]
    
        # Count upregulated and downregulated genes
        up_count = np.sum(sig_log2fc_values > 0)
        down_count = np.sum(sig_log2fc_values < 0)
    
        up_counts.append(up_count)
        down_counts.append(down_count)
    

        ### ------- NEW Adjust Text in Plots -----------
        # --- Volcano Plot with Gene Annotations (Corrected Version) ---
        from adjustText import adjust_text  # <-- 1. IMPORT THE LIBRARY
        
        plt.figure(figsize=(10, 8)) # Increased figure size for better spacing
        
        # Scatter plot: all significant genes
        plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.5, s=10) # s=10 for smaller dots
        
        # Highlight significant upregulated genes (red)
        upregulated = sig_log2fc_values > 0
        plt.scatter(
            sig_log2fc_values[upregulated], 
            sig_neg_log_pvals[upregulated], 
            color="red", label="Upregulated", s=15
        )
        
        # Highlight significant downregulated genes (blue)
        downregulated = sig_log2fc_values < 0
        plt.scatter(
            sig_log2fc_values[downregulated], 
            sig_neg_log_pvals[downregulated], 
            color="blue", label="Downregulated", s=15
        )
        
        # --- ANNOTATION CHANGES START HERE ---
        
        # 2. CREATE AN EMPTY LIST TO HOLD TEXT OBJECTS
        texts_to_adjust = []
        top_n_genes = 20
        
        # Sort by p-values to select the top genes to annotate
        # Upregulated genes
        if np.any(upregulated):
            sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
            for idx in sorted_idx_up:
                gene_name = sig_genes[upregulated][idx]
                x = sig_log2fc_values[upregulated][idx]
                y = sig_neg_log_pvals[upregulated][idx]
                # 3. INSTEAD OF PLOTTING, APPEND THE TEXT OBJECT TO THE LIST
                texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        
        # Downregulated genes
        if np.any(downregulated):
            sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
            for idx in sorted_idx_down:
                gene_name = sig_genes[downregulated][idx]
                x = sig_log2fc_values[downregulated][idx]
                y = sig_neg_log_pvals[downregulated][idx]
                # 3. APPEND THE TEXT OBJECT TO THE LIST
                texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        print(f"KA3 - Downregulated {'\n'.join(sig_genes[downregulated][-top_n_genes:])}")
        print(f"KA3 - Upregulated {'\n'.join(sig_genes[upregulated][-top_n_genes:])}")
        # 4. CALL adjust_text ONCE TO AUTOMATICALLY POSITION LABELS AND DRAW LINES
        if texts_to_adjust:
            adjust_text(texts_to_adjust, 
                        arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
                        ax=plt.gca()) # Specify the axes
        
        # --- ANNOTATION CHANGES END HERE ---
        
        # Add significance threshold lines
        plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change")
        plt.ylabel("-log10(p-value)")
        plt.title(f"Volcano Plot for Cell Type {cluster} under Low-dose (KA3) Condition")
        plt.legend()
        plt.tight_layout() # Helps fit everything neatly
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KA3_YekBasteh.png")

        plt.show()
        ### ------ END of Adjust text to plots ---------------

        # # ----------------- Volcano Plot with Gene Annotations ----------------------
        # plt.figure(figsize=(8, 6))
    
        # # Scatter plot: significant genes only
        # plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.6)
    
        # # Highlight significant upregulated genes (red)
        # upregulated = sig_log2fc_values > 0
        # plt.scatter(
        #     sig_log2fc_values[upregulated], 
        #     sig_neg_log_pvals[upregulated], 
        #     color="red", label="Upregulated"
        # )
    
        # # Highlight significant downregulated genes (blue)
        # downregulated = sig_log2fc_values < 0
        # plt.scatter(
        #     sig_log2fc_values[downregulated], 
        #     sig_neg_log_pvals[downregulated], 
        #     color="blue", label="Downregulated"
        # )
    
        # # Annotate top upregulated and downregulated genes
        # top_n_genes = 20
        
        # # Sort by p-values and annotate
        # sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
        # sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
    
        # for idx in sorted_idx_up:
        #     gene_name = sig_genes[upregulated][idx]
        #     plt.text(sig_log2fc_values[upregulated][idx], sig_neg_log_pvals[upregulated][idx], 
        #              gene_name, fontsize=9, ha='left', va='bottom', color='black')
    
        # for idx in sorted_idx_down:
        #     gene_name = sig_genes[downregulated][idx]
        #     plt.text(sig_log2fc_values[downregulated][idx], sig_neg_log_pvals[downregulated][idx], 
        #              gene_name, fontsize=9, ha='right', va='bottom', color='black')
    
        # # Add significance threshold lines
        # plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        # plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
    
        # plt.xlabel("log2 Fold Change")
        # plt.ylabel("-log10(p-value)")
        # plt.title(f"Volcano Plot for Cell Type {cluster} under Low-dose (KA3) Condition")
        # plt.legend()
        # plt.grid(True, linestyle='--', alpha=0.6)
        # plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KA3_YekBasteh.png")
        # plt.show()

### 9. Nucleus-level differential expression, KA3
Same contrast within the subconvulsive dose (Fig. 2D, Figs. S2).

In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

up_genes_KA3 = {}
down_genes_KA3 = {}

for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['condition'] == 'KA3']
    # Perform differential expression analysis
    sc.tl.rank_genes_groups(cluster_data, groupby='group',reference='WT', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'KA3_Vali_{name_validation}_{cluster}_YekBasteh.png')

    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------
    

    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])
        
        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
        # Calculate -log10(p-value) and handle p-values of 0
        with np.errstate(divide='ignore'):
            neg_log_pvals = -np.log10(p_values_array)
        neg_log_pvals[np.isinf(neg_log_pvals)] = np.nanmax(neg_log_pvals[np.isfinite(neg_log_pvals)]) # Replace inf with max finite value
        neg_log_pvals = np.nan_to_num(neg_log_pvals) # Replace any remaining NaNs with 0

        # --- 1. DEFINE THRESHOLDS ---
        LOG2FC_THRESHOLD = 0.5
        P_ADJ_THRESHOLD = 0.05
        NEG_LOG_P_THRESHOLD = -np.log10(P_ADJ_THRESHOLD)

        # --- 2. CREATE BOOLEAN MASKS FOR EACH CATEGORY ---
        
        # Category 1: Significant AND Upregulated
        up_mask = (log2fc_values_array > LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 2: Significant AND Downregulated
        down_mask = (log2fc_values_array < -LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 3: Significant but low fold-change
        sig_low_fc_mask = (p_values_array < P_ADJ_THRESHOLD) & ~up_mask & ~down_mask & sex_gene_mask

        # Category 4: Not significant (all others)
        not_sig_mask = ~(up_mask | down_mask | sig_low_fc_mask)


        # --- NEW: PRINT GENE NAMES ---
        # --- NEW: PRINT CLEAN GENE NAMES (NO QUOTES) ---
        up_genes_list = genes_array[up_mask].tolist()
        down_genes_list = genes_array[down_mask].tolist()

        print(f"\n{'='*60}")
        print(f"CLUSTER: {cluster} | GROUP: {group_name}")
        print(f"{'='*60}")
        
        # Join the list into a single string separated by commas
        print(f"UPREGULATED ({len(up_genes_list)} genes):")
        print(", ".join(up_genes_list) if up_genes_list else "None")
        
        print(f"\nDOWNREGULATED ({len(down_genes_list)} genes):")
        print(", ".join(down_genes_list) if down_genes_list else "None")
        print(f"{'='*60}\n")
        
        # --- 3. PLOT EACH CATEGORY ---
        plt.figure(figsize=(12, 10))
        
        # # Plot 1: Not significant -> Grey
        # plt.scatter(log2fc_values_array[not_sig_mask], neg_log_pvals[not_sig_mask], 
        #             color="grey", alpha=0.4, s=10, label=f'p ≥ {P_ADJ_THRESHOLD}')
        
        # Plot 2: Significant, but low fold-change -> Orange
        plt.scatter(log2fc_values_array[sig_low_fc_mask], neg_log_pvals[sig_low_fc_mask], 
                    color="orange", s=15, alpha=0.7, label=f'p < {P_ADJ_THRESHOLD} & |log2FC| < {LOG2FC_THRESHOLD}')
        
        # Plot 3: Significant AND Downregulated -> Blue
        plt.scatter(log2fc_values_array[down_mask], neg_log_pvals[down_mask], 
                    color="blue", s=20, label=f'Downregulated')
        
        # Plot 4: Significant AND Upregulated -> Red
        plt.scatter(log2fc_values_array[up_mask], neg_log_pvals[up_mask], 
                    color="red", s=20, label=f'Upregulated')

        # --- 4. ANNOTATE TOP GENES ---
        texts_to_adjust = []
        top_n_genes = 15
        
        # Annotate top Upregulated genes (from the red points)
        if np.any(up_mask):
            up_genes, up_pvals, up_fc = genes_array[up_mask], neg_log_pvals[up_mask], log2fc_values_array[up_mask]
            sorted_idx_up = np.argsort(up_pvals)[-top_n_genes:]
            for idx in sorted_idx_up:
                texts_to_adjust.append(plt.text(up_fc[idx], up_pvals[idx], up_genes[idx], fontsize=20))

        # Annotate top Downregulated genes (from the blue points)
        if np.any(down_mask):
            down_genes, down_pvals, down_fc = genes_array[down_mask], neg_log_pvals[down_mask], log2fc_values_array[down_mask]
            sorted_idx_down = np.argsort(down_pvals)[-top_n_genes:]
            for idx in sorted_idx_down:
                texts_to_adjust.append(plt.text(down_fc[idx], down_pvals[idx], down_genes[idx], fontsize=20))


        up_genes_KA3[cluster] = up_genes
        down_genes_KA3[cluster] = down_genes

        
        if texts_to_adjust:
            # adjust_text(texts_to_adjust, 
            #             arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
            #             ax=plt.gca())
            adjust_text(
                texts_to_adjust,
                ax=plt.gca(),
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
                force_points=2.9,#0.5,        # how strongly labels avoid points
                force_text=2.9,#0.7,          # how strongly labels avoid each other
                expand_points=(1.2, 1.4), # extra padding around points
                expand_text=(1.2, 1.4),   # extra padding around texts
                only_move={'points': 'y', 'text': 'xy'},  # allow free movement in both x & y
                lim=500,                 # number of iterations for optimization
            )
        # --- 5. ADD THRESHOLD LINES AND LABELS ---
        plt.axhline(y=NEG_LOG_P_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=-LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change", fontsize=22,fontweight='bold')
        plt.ylabel("-log10(Adjusted p-value)", fontsize=22,fontweight='bold')
        plt.title(f"Volcano Plot for {cluster} Population - Low-dose (KA3) Condition", fontsize=22, fontweight='bold')


        plt.legend(
            loc='upper center',
            fontsize=18,        # Increase legend text size
            markerscale=2.0,    # Scale up legend markers (optional)
            frameon=True,       # Keep a frame (set to False to remove)
            framealpha=0.8,     # Slight transparency
            handlelength=1.5,   # Length of the color/marker handles
            title_fontsize=18    # Font size for legend title
        )

        plt.tick_params(axis='both', which='major', labelsize=20)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)

        plt.tight_layout() # Helps fit everything neatly
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KA3_YekBasteh.png")

        plt.show()


### 10. Nucleus-level differential expression, KA25
Same contrast within the convulsive dose (Fig. 2E, Figs. S3).

In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['condition'] == 'KA25']
    # Perform differential expression analysis
    sc.tl.rank_genes_groups(cluster_data, groupby='group',reference='WT', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'KA25_Vali_{name_validation}_{cluster}_YekBasteh.png')

    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------
    

    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])
        
        # Calculate -log10(p-values)
        neg_log_pvals = -np.log10(p_values_array)
    
        # Apply significance threshold
        significant = neg_log_pvals > 1.5


        log2fc_range_mask = ((log2fc_values_array > 0) & (log2fc_values_array < 5.0)) | \
                            ((log2fc_values_array < -0) & (log2fc_values_array > -5.0))


        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
        
        # Combine both masks
        valid_mask = significant & log2fc_range_mask & sex_gene_mask
        
        # Apply to arrays
        sig_log2fc_values = log2fc_values_array[valid_mask]
        sig_neg_log_pvals = neg_log_pvals[valid_mask]
        sig_genes = genes_array[valid_mask]
        
        # sig_log2fc_values = log2fc_values_array[significant]
        # sig_neg_log_pvals = neg_log_pvals[significant]
        # sig_genes = genes_array[significant]
    
        # Count upregulated and downregulated genes
        up_count = np.sum(sig_log2fc_values > 0)
        down_count = np.sum(sig_log2fc_values < 0)
    
        up_counts.append(up_count)
        down_counts.append(down_count)



        ### ------- NEW Adjust Text in Plots -----------
        # --- Volcano Plot with Gene Annotations (Corrected Version) ---
        from adjustText import adjust_text  # <-- 1. IMPORT THE LIBRARY
        
        plt.figure(figsize=(10, 8)) # Increased figure size for better spacing
        
        # Scatter plot: all significant genes
        plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.5, s=10) # s=10 for smaller dots
        
        # Highlight significant upregulated genes (red)
        upregulated = sig_log2fc_values > 0
        plt.scatter(
            sig_log2fc_values[upregulated], 
            sig_neg_log_pvals[upregulated], 
            color="red", label="Upregulated", s=15
        )
        
        # Highlight significant downregulated genes (blue)
        downregulated = sig_log2fc_values < 0
        plt.scatter(
            sig_log2fc_values[downregulated], 
            sig_neg_log_pvals[downregulated], 
            color="blue", label="Downregulated", s=15
        )
        
        # --- ANNOTATION CHANGES START HERE ---
        
        # 2. CREATE AN EMPTY LIST TO HOLD TEXT OBJECTS
        texts_to_adjust = []
        top_n_genes = 20
        
        # Sort by p-values to select the top genes to annotate
        # Upregulated genes
        if np.any(upregulated):
            sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
            for idx in sorted_idx_up:
                gene_name = sig_genes[upregulated][idx]
                x = sig_log2fc_values[upregulated][idx]
                y = sig_neg_log_pvals[upregulated][idx]
                # 3. INSTEAD OF PLOTTING, APPEND THE TEXT OBJECT TO THE LIST
                texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        
        # Downregulated genes
        if np.any(downregulated):
            sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
            for idx in sorted_idx_down:
                gene_name = sig_genes[downregulated][idx]
                x = sig_log2fc_values[downregulated][idx]
                y = sig_neg_log_pvals[downregulated][idx]
                # 3. APPEND THE TEXT OBJECT TO THE LIST
                texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        print(f"KA25 - Downregulated {'\n'.join(sig_genes[downregulated][-top_n_genes:])}")
        print(f"KA25 - Upregulated {'\n'.join(sig_genes[upregulated][-top_n_genes:])}")
        # 4. CALL adjust_text ONCE TO AUTOMATICALLY POSITION LABELS AND DRAW LINES
        if texts_to_adjust:
            adjust_text(texts_to_adjust, 
                        arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
                        ax=plt.gca()) # Specify the axes
        
        # --- ANNOTATION CHANGES END HERE ---
        
        # Add significance threshold lines
        plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change")
        plt.ylabel("-log10(p-value)")
        plt.title(f"Volcano Plot for Cell Type {cluster} under High-dose (KA25) Condition")
        plt.legend()
        plt.tight_layout() # Helps fit everything neatly
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KA25_YekBasteh.png")

        plt.show()
        ### ------ END of Adjust text to plots ---------------

        
        # # ----------------- Volcano Plot with Gene Annotations ----------------------
        # plt.figure(figsize=(8, 6))
    
        # # Scatter plot: significant genes only
        # plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.6)
    
        # # Highlight significant upregulated genes (red)
        # upregulated = sig_log2fc_values > 0
        # plt.scatter(
        #     sig_log2fc_values[upregulated], 
        #     sig_neg_log_pvals[upregulated], 
        #     color="red", label="Upregulated"
        # )
    
        # # Highlight significant downregulated genes (blue)
        # downregulated = sig_log2fc_values < 0
        # plt.scatter(
        #     sig_log2fc_values[downregulated], 
        #     sig_neg_log_pvals[downregulated], 
        #     color="blue", label="Downregulated"
        # )
    
        # # Annotate top upregulated and downregulated genes
        # top_n_genes = 20
        
        # # Sort by p-values and annotate
        # sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
        # sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
    
        # for idx in sorted_idx_up:
        #     gene_name = sig_genes[upregulated][idx]
        #     plt.text(sig_log2fc_values[upregulated][idx], sig_neg_log_pvals[upregulated][idx], 
        #              gene_name, fontsize=9, ha='left', va='bottom', color='black')
    
        # for idx in sorted_idx_down:
        #     gene_name = sig_genes[downregulated][idx]
        #     plt.text(sig_log2fc_values[downregulated][idx], sig_neg_log_pvals[downregulated][idx], 
        #              gene_name, fontsize=9, ha='right', va='bottom', color='black')
    
        # # Add significance threshold lines
        # plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        # plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
    
        # plt.xlabel("log2 Fold Change")
        # plt.ylabel("-log10(p-value)")
        # plt.title(f"Volcano Plot for Cell Type {cluster} under High-dose (KA25) Condition")
        # plt.legend()
        # plt.grid(True, linestyle='--', alpha=0.6)
        # plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KA25_YekBasteh.png")
        # plt.show()

In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

up_genes_KA25 = {}
down_genes_KA25 = {}

for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['condition'] == 'KA25']
    # Perform differential expression analysis
    sc.tl.rank_genes_groups(cluster_data, groupby='group',reference='WT', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'KA25_Vali_{name_validation}_{cluster}_YekBasteh.png')

    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------
    

    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])
        
        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
        # Calculate -log10(p-value) and handle p-values of 0
        with np.errstate(divide='ignore'):
            neg_log_pvals = -np.log10(p_values_array)
        neg_log_pvals[np.isinf(neg_log_pvals)] = np.nanmax(neg_log_pvals[np.isfinite(neg_log_pvals)]) # Replace inf with max finite value
        neg_log_pvals = np.nan_to_num(neg_log_pvals) # Replace any remaining NaNs with 0

        # --- 1. DEFINE THRESHOLDS ---
        LOG2FC_THRESHOLD = 0.5
        P_ADJ_THRESHOLD = 0.05
        NEG_LOG_P_THRESHOLD = -np.log10(P_ADJ_THRESHOLD)

        # --- 2. CREATE BOOLEAN MASKS FOR EACH CATEGORY ---
        
        # Category 1: Significant AND Upregulated
        up_mask = (log2fc_values_array > LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 2: Significant AND Downregulated
        down_mask = (log2fc_values_array < -LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 3: Significant but low fold-change
        sig_low_fc_mask = (p_values_array < P_ADJ_THRESHOLD) & ~up_mask & ~down_mask & sex_gene_mask

        # Category 4: Not significant (all others)
        not_sig_mask = ~(up_mask | down_mask | sig_low_fc_mask)





        # --- NEW: PRINT GENE NAMES ---
        # --- NEW: PRINT CLEAN GENE NAMES (NO QUOTES) ---
        up_genes_list = genes_array[up_mask].tolist()
        down_genes_list = genes_array[down_mask].tolist()

        print(f"\n{'='*60}")
        print(f"CLUSTER: {cluster} | GROUP: {group_name}")
        print(f"{'='*60}")
        
        # Join the list into a single string separated by commas
        print(f"UPREGULATED ({len(up_genes_list)} genes):")
        print(", ".join(up_genes_list) if up_genes_list else "None")
        
        print(f"\nDOWNREGULATED ({len(down_genes_list)} genes):")
        print(", ".join(down_genes_list) if down_genes_list else "None")
        print(f"{'='*60}\n")
        
        # --- 3. PLOT EACH CATEGORY ---
        plt.figure(figsize=(12, 10))
        
        # # Plot 1: Not significant -> Grey
        # plt.scatter(log2fc_values_array[not_sig_mask], neg_log_pvals[not_sig_mask], 
        #             color="grey", alpha=0.4, s=10, label=f'p ≥ {P_ADJ_THRESHOLD}')
        
        # Plot 2: Significant, but low fold-change -> Orange
        plt.scatter(log2fc_values_array[sig_low_fc_mask], neg_log_pvals[sig_low_fc_mask], 
                    color="orange", s=15, alpha=0.7, label=f'p < {P_ADJ_THRESHOLD} & |log2FC| < {LOG2FC_THRESHOLD}')
        
        # Plot 3: Significant AND Downregulated -> Blue
        plt.scatter(log2fc_values_array[down_mask], neg_log_pvals[down_mask], 
                    color="blue", s=20, label=f'Downregulated')
        
        # Plot 4: Significant AND Upregulated -> Red
        plt.scatter(log2fc_values_array[up_mask], neg_log_pvals[up_mask], 
                    color="red", s=20, label=f'Upregulated')

        # --- 4. ANNOTATE TOP GENES ---
        texts_to_adjust = []
        top_n_genes = 15
        
        # Annotate top Upregulated genes (from the red points)
        if np.any(up_mask):
            up_genes, up_pvals, up_fc = genes_array[up_mask], neg_log_pvals[up_mask], log2fc_values_array[up_mask]
            sorted_idx_up = np.argsort(up_pvals)[-top_n_genes:]
            for idx in sorted_idx_up:
                texts_to_adjust.append(plt.text(up_fc[idx], up_pvals[idx], up_genes[idx], fontsize=20))

        # Annotate top Downregulated genes (from the blue points)
        if np.any(down_mask):
            down_genes, down_pvals, down_fc = genes_array[down_mask], neg_log_pvals[down_mask], log2fc_values_array[down_mask]
            sorted_idx_down = np.argsort(down_pvals)[-top_n_genes:]
            for idx in sorted_idx_down:
                texts_to_adjust.append(plt.text(down_fc[idx], down_pvals[idx], down_genes[idx], fontsize=20))

        up_genes_KA25[cluster] = up_genes
        down_genes_KA25[cluster] = down_genes

        
        if texts_to_adjust:
            # adjust_text(texts_to_adjust, 
            #             arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
            #             ax=plt.gca())
            adjust_text(
                texts_to_adjust,
                ax=plt.gca(),
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
                force_points=2.9,#0.5,        # how strongly labels avoid points
                force_text=2.9,#0.7,          # how strongly labels avoid each other
                expand_points=(1.2, 1.4), # extra padding around points
                expand_text=(1.2, 1.4),   # extra padding around texts
                only_move={'points': 'y', 'text': 'xy'},  # allow free movement in both x & y
                lim=500,                 # number of iterations for optimization
            )
        
        # --- 5. ADD THRESHOLD LINES AND LABELS ---
        plt.axhline(y=NEG_LOG_P_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=-LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change", fontsize=22,fontweight='bold')
        plt.ylabel("-log10(Adjusted p-value)", fontsize=22,fontweight='bold')
        plt.title(f"Volcano Plot for {cluster} Population - High-dose (KA25) Condition", fontsize=22,fontweight='bold')


        plt.legend(
            loc='upper left',
            fontsize=18,        # Increase legend text size
            markerscale=2.0,    # Scale up legend markers (optional)
            frameon=True,       # Keep a frame (set to False to remove)
            framealpha=0.8,     # Slight transparency
            handlelength=1.5,   # Length of the color/marker handles
            title_fontsize=18    # Font size for legend title
        )

        plt.tick_params(axis='both', which='major', labelsize=20)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.tight_layout() # Helps fit everything neatly
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KA25_YekBasteh.png")

        plt.show()

### 11. Treatment contrasts within each genotype
KA versus vehicle within WT and within KO separately (Figs. S4-S5).

In [ ]:

adata_combined_all_genes = adata_filtered_significant_genes.copy()

import matplotlib.pyplot as plt
import csv
import seaborn as sns
import numpy as np

# Extract the number of upregulated and downregulated genes per cluster
clusters = adata_combined_all_genes.obs["celltype"].unique()
# top_genes_after_remove = {}
# up_counts = []
# down_counts = []
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}

for cluster in clusters:
    print(f"For cluster {cluster} ...")
    # Subset the data for the current cluster
    cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
    cluster_data = cluster_data[cluster_data.obs['group'] == 'WT']
    # print(cluster_data.obs['condition'].unique())

    # #print(cluster_data.obs['condition'].value_counts())
    # print(eeee)
    # Perform differential expression analysis

    #reference='Veh',
    sc.tl.rank_genes_groups(cluster_data, groupby='condition', reference='Veh', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')

    sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False, save=f'Vali_{name_validation}_{cluster}_YekBasteh_WT_Response.png')


    # -----------------  top genes   ----------------------
    top_genes = get_top_genes_from_rank(cluster_data)
    if top_genes:
        enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
    
        # Transform p-values to -log10(p-value)
        enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
    
        top_n = 50
        top_enriched = enrichment_results.head(top_n)
        print(f"top enriched {top_enriched}")
    
        # Create a bar plot using transformed p-values
        plt.figure(figsize=(10, 6))
        sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        plt.xlabel('-log10(p-value)')
        plt.ylabel('GO Term')
        plt.title(f'Top {top_n} Enriched GO Terms')
        plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        plt.show()
    else:
        print(f"No significant DEGs found for cluster {cluster}.")
    # -----------------------------------------------------


    
    ## ********************* NEW ****************************
    # Extract top genes and scores for the current cluster
    cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    #print(f"cluster_scores {cluster_scores}")
    log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    #print(f"log2fc_values {log2fc_values}")
    # Check available group names in rank_genes_groups
    group_names = list(cluster_genes.dtype.names)
    #print(f"Available groups in rank_genes_groups: {group_names}")

    log2fc_dict[cluster] = dict()
    p_values_dict[cluster]=dict()
    for group_name in group_names:
        # Convert to dictionaries
        log2fc_dict[cluster][group_name] = {
            gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
        }
        p_values_dict[cluster][group_name] = {
            gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
        }
        
        # Convert dict values to NumPy arrays
        log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        genes_array = np.array(cluster_genes[group_name])
        
        sex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
        # Calculate -log10(p-value) and handle p-values of 0
        with np.errstate(divide='ignore'):
            neg_log_pvals = -np.log10(p_values_array)
        neg_log_pvals[np.isinf(neg_log_pvals)] = np.nanmax(neg_log_pvals[np.isfinite(neg_log_pvals)]) # Replace inf with max finite value
        neg_log_pvals = np.nan_to_num(neg_log_pvals) # Replace any remaining NaNs with 0

        # --- 1. DEFINE THRESHOLDS ---
        LOG2FC_THRESHOLD = 0.5
        P_ADJ_THRESHOLD = 0.05
        NEG_LOG_P_THRESHOLD = -np.log10(P_ADJ_THRESHOLD)

        # --- 2. CREATE BOOLEAN MASKS FOR EACH CATEGORY ---
        
        # Category 1: Significant AND Upregulated
        up_mask = (log2fc_values_array > LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 2: Significant AND Downregulated
        down_mask = (log2fc_values_array < -LOG2FC_THRESHOLD) & (p_values_array < P_ADJ_THRESHOLD) & sex_gene_mask
        
        # Category 3: Significant but low fold-change
        sig_low_fc_mask = (p_values_array < P_ADJ_THRESHOLD) & ~up_mask & ~down_mask & sex_gene_mask

        # Category 4: Not significant (all others)
        not_sig_mask = ~(up_mask | down_mask | sig_low_fc_mask)





        # --- NEW: PRINT GENE NAMES ---
        # --- NEW: PRINT CLEAN GENE NAMES (NO QUOTES) ---
        up_genes_list = genes_array[up_mask].tolist()
        down_genes_list = genes_array[down_mask].tolist()

        print(f"\n{'='*60}")
        print(f"CLUSTER: {cluster} | GROUP: {group_name}")
        print(f"{'='*60}")
        
        # Join the list into a single string separated by commas
        print(f"UPREGULATED ({len(up_genes_list)} genes):")
        print(", ".join(up_genes_list) if up_genes_list else "None")
        
        print(f"\nDOWNREGULATED ({len(down_genes_list)} genes):")
        print(", ".join(down_genes_list) if down_genes_list else "None")
        print(f"{'='*60}\n")
        # --- 3. PLOT EACH CATEGORY ---
        plt.figure(figsize=(12, 10))
        
        # # Plot 1: Not significant -> Grey
        # plt.scatter(log2fc_values_array[not_sig_mask], neg_log_pvals[not_sig_mask], 
        #             color="grey", alpha=0.4, s=10, label=f'p ≥ {P_ADJ_THRESHOLD}')
        
        # Plot 2: Significant, but low fold-change -> Orange
        plt.scatter(log2fc_values_array[sig_low_fc_mask], neg_log_pvals[sig_low_fc_mask], 
                    color="orange", s=15, alpha=0.7, label=f'p < {P_ADJ_THRESHOLD} & |log2FC| < {LOG2FC_THRESHOLD}')
        
        # Plot 3: Significant AND Downregulated -> Blue
        plt.scatter(log2fc_values_array[down_mask], neg_log_pvals[down_mask], 
                    color="blue", s=20, label=f'Downregulated')
        
        # Plot 4: Significant AND Upregulated -> Red
        plt.scatter(log2fc_values_array[up_mask], neg_log_pvals[up_mask], 
                    color="red", s=20, label=f'Upregulated')

        # --- 4. ANNOTATE TOP GENES ---
        texts_to_adjust = []
        top_n_genes = 15
        
        # Annotate top Upregulated genes (from the red points)
        if np.any(up_mask):
            up_genes, up_pvals, up_fc = genes_array[up_mask], neg_log_pvals[up_mask], log2fc_values_array[up_mask]
            sorted_idx_up = np.argsort(up_pvals)[-top_n_genes:]
            for idx in sorted_idx_up:
                texts_to_adjust.append(plt.text(up_fc[idx], up_pvals[idx], up_genes[idx], fontsize=20))

        # Annotate top Downregulated genes (from the blue points)
        if np.any(down_mask):
            down_genes, down_pvals, down_fc = genes_array[down_mask], neg_log_pvals[down_mask], log2fc_values_array[down_mask]
            sorted_idx_down = np.argsort(down_pvals)[-top_n_genes:]
            for idx in sorted_idx_down:
                texts_to_adjust.append(plt.text(down_fc[idx], down_pvals[idx], down_genes[idx], fontsize=20))

        
        if texts_to_adjust:
            # adjust_text(texts_to_adjust, 
            #             arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
            #             ax=plt.gca())
            adjust_text(
                texts_to_adjust,
                ax=plt.gca(),
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
                force_points=2.9,#0.5,        # how strongly labels avoid points
                force_text=2.9,#0.7,          # how strongly labels avoid each other
                expand_points=(1.2, 1.4), # extra padding around points
                expand_text=(1.2, 1.4),   # extra padding around texts
                only_move={'points': 'y', 'text': 'xy'},  # allow free movement in both x & y
                lim=500,                 # number of iterations for optimization
            )

        # # ---- PRINT gene names (all significant up/down) ----
        # # Upregulated
        # if np.any(up_mask):
        #     up_genes = genes_array[up_mask]
        #     up_pvals = neg_log_pvals[up_mask]
        #     top_up_idx = np.argsort(up_pvals)[-top_n_genes:][::-1]
        
        #     print(f"\n[{cluster} | {group_name}] Upregulated (TOP {top_n_genes}):")
        #     print("\n".join(up_genes[top_up_idx]))
        
        # # Downregulated
        # if np.any(down_mask):
        #     down_genes = genes_array[down_mask]
        #     down_pvals = neg_log_pvals[down_mask]
        #     top_down_idx = np.argsort(down_pvals)[-top_n_genes:][::-1]
        
        #     print(f"\n[{cluster} | {group_name}] Downregulated (TOP {top_n_genes}):")
        #     print("\n".join(down_genes[top_down_idx]))

            
        # --- 5. ADD THRESHOLD LINES AND LABELS ---
        plt.axhline(y=NEG_LOG_P_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        plt.axvline(x=-LOG2FC_THRESHOLD, linestyle="--", color="black", alpha=0.7)
        
        plt.xlabel("log2 Fold Change", fontsize=22,fontweight='bold')
        plt.ylabel("-log10(Adjusted p-value)", fontsize=22,fontweight='bold')
        plt.title(f"Volcano Plot - {cluster} - Changes in WT-{group_name} relative to WT-Vehicle.", fontsize=22,fontweight='bold')


        plt.legend(
            loc='upper left',
            fontsize=18,        # Increase legend text size
            markerscale=2.0,    # Scale up legend markers (optional)
            frameon=True,       # Keep a frame (set to False to remove)
            framealpha=0.8,     # Slight transparency
            handlelength=1.5,   # Length of the color/marker handles
            title_fontsize=18    # Font size for legend title
        )

        plt.tick_params(axis='both', which='major', labelsize=20)
        plt.grid(True, which='both', linestyle='--', linewidth=0.5)
        plt.tight_layout() # Helps fit everything neatly
        plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_YekBasteh_WT_Response.png")

        plt.show()

        
    
    # ## ********************* NEW ****************************
    # # Extract top genes and scores for the current cluster
    # cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
    # cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
    # #print(f"cluster_scores {cluster_scores}")
    # log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
    # p_values = cluster_data.uns['rank_genes_groups']['pvals_adj'] 
    # #print(f"log2fc_values {log2fc_values}")
    # # Check available group names in rank_genes_groups
    # group_names = list(cluster_genes.dtype.names)
    # #print(f"Available groups in rank_genes_groups: {group_names}")

    # log2fc_dict[cluster] = dict()
    # p_values_dict[cluster]=dict()
    # for group_name in group_names:
    #     # Convert to dictionaries
    #     log2fc_dict[cluster][group_name] = {
    #         gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])
    #     }
    #     p_values_dict[cluster][group_name] = {
    #         gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])
    #     }
        
    #     # Convert dict values to NumPy arrays
    #     log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
    #     p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
    #     genes_array = np.array(cluster_genes[group_name])
        
    #     # Calculate -log10(p-values)
    #     neg_log_pvals = -np.log10(p_values_array)
    
    #     # Apply significance threshold
    #     significant = neg_log_pvals > 1.5


    #     log2fc_range_mask = ((log2fc_values_array > 0) & (log2fc_values_array < 5.0)) | \
    #                         ((log2fc_values_array < -0) & (log2fc_values_array > -5.0))

    #     ssex_gene_mask = ~np.isin(genes_array, sex_genes_to_remove)
    #     # Combine both masks
    #     valid_mask = significant & log2fc_range_mask & sex_gene_mask
        
    #     # Apply to arrays
    #     sig_log2fc_values = log2fc_values_array[valid_mask]
    #     sig_neg_log_pvals = neg_log_pvals[valid_mask]
    #     sig_genes = genes_array[valid_mask]
    #     # sig_log2fc_values = log2fc_values_array[significant]
    #     # sig_neg_log_pvals = neg_log_pvals[significant]
    #     # sig_genes = genes_array[significant]
    
    #     # Count upregulated and downregulated genes
    #     up_count = np.sum(sig_log2fc_values > 0)
    #     down_count = np.sum(sig_log2fc_values < 0)
    
    #     up_counts.append(up_count)
    #     down_counts.append(down_count)



    #     ### ------- NEW Adjust Text in Plots -----------
    #     # --- Volcano Plot with Gene Annotations (Corrected Version) ---
    #     from adjustText import adjust_text  # <-- 1. IMPORT THE LIBRARY
        
    #     plt.figure(figsize=(10, 8)) # Increased figure size for better spacing
        
    #     # Scatter plot: all significant genes
    #     plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.5, s=10) # s=10 for smaller dots
        
    #     # Highlight significant upregulated genes (red)
    #     upregulated = sig_log2fc_values > 0
    #     plt.scatter(
    #         sig_log2fc_values[upregulated], 
    #         sig_neg_log_pvals[upregulated], 
    #         color="red", label="Upregulated", s=15
    #     )
        
    #     # Highlight significant downregulated genes (blue)
    #     downregulated = sig_log2fc_values < 0
    #     plt.scatter(
    #         sig_log2fc_values[downregulated], 
    #         sig_neg_log_pvals[downregulated], 
    #         color="blue", label="Downregulated", s=15
    #     )
        
    #     # --- ANNOTATION CHANGES START HERE ---
        
    #     # 2. CREATE AN EMPTY LIST TO HOLD TEXT OBJECTS
    #     texts_to_adjust = []
    #     top_n_genes = 20
        
    #     # Sort by p-values to select the top genes to annotate
    #     # Upregulated genes
    #     if np.any(upregulated):
    #         sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
    #         for idx in sorted_idx_up:
    #             gene_name = sig_genes[upregulated][idx]
    #             x = sig_log2fc_values[upregulated][idx]
    #             y = sig_neg_log_pvals[upregulated][idx]
    #             # 3. INSTEAD OF PLOTTING, APPEND THE TEXT OBJECT TO THE LIST
    #             texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))
        
    #     # Downregulated genes
    #     if np.any(downregulated):
    #         sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
    #         for idx in sorted_idx_down:
    #             gene_name = sig_genes[downregulated][idx]
    #             x = sig_log2fc_values[downregulated][idx]
    #             y = sig_neg_log_pvals[downregulated][idx]
    #             # 3. APPEND THE TEXT OBJECT TO THE LIST
    #             texts_to_adjust.append(plt.text(x, y, gene_name, fontsize=9, color='black'))

    #     print(f"KO - Downregulated {'\n'.join(sig_genes[downregulated][-top_n_genes:])}")
    #     print(f"KO - Upregulated {'\n'.join(sig_genes[upregulated][-top_n_genes:])}")
    #     # 4. CALL adjust_text ONCE TO AUTOMATICALLY POSITION LABELS AND DRAW LINES
    #     if texts_to_adjust:
    #         adjust_text(texts_to_adjust, 
    #                     arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
    #                     ax=plt.gca()) # Specify the axes
        
    #     # --- ANNOTATION CHANGES END HERE ---
        
    #     # Add significance threshold lines
    #     plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
    #     plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
        
    #     plt.xlabel("log2 Fold Change")
    #     plt.ylabel("-log10(p-value)")
    #     plt.title(f"Volcano Plot for Cell Type {cluster} under knockout (KO) Condition Comparing {group_name} to Vehicle.")
    #     plt.legend()
    #     plt.tight_layout() # Helps fit everything neatly
    #     plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KO_YekBasteh.png", dpi=300) # Higher DPI for better quality
    #     plt.show()
    #     ### ------ END of Adjust text to plots ---------------



        # # ----------------- Volcano Plot with Gene Annotations ----------------------
        # plt.figure(figsize=(8, 6))
    
        # # Scatter plot: significant genes only
        # plt.scatter(sig_log2fc_values, sig_neg_log_pvals, color="grey", alpha=0.6)
    
        # # Highlight significant upregulated genes (red)
        # upregulated = sig_log2fc_values > 0
        # plt.scatter(
        #     sig_log2fc_values[upregulated], 
        #     sig_neg_log_pvals[upregulated], 
        #     color="red", label="Upregulated"
        # )
    
        # # Highlight significant downregulated genes (blue)
        # downregulated = sig_log2fc_values < 0
        # plt.scatter(
        #     sig_log2fc_values[downregulated], 
        #     sig_neg_log_pvals[downregulated], 
        #     color="blue", label="Downregulated"
        # )
    
        # # Annotate top upregulated and downregulated genes
        # top_n_genes = 20
        
        # # Sort by p-values and annotate
        # sorted_idx_up = np.argsort(sig_neg_log_pvals[upregulated])[-top_n_genes:]
        # sorted_idx_down = np.argsort(sig_neg_log_pvals[downregulated])[-top_n_genes:]
    
        # for idx in sorted_idx_up:
        #     gene_name = sig_genes[upregulated][idx]
        #     plt.text(sig_log2fc_values[upregulated][idx], sig_neg_log_pvals[upregulated][idx], 
        #              gene_name, fontsize=9, ha='left', va='bottom', color='black')
    
        # for idx in sorted_idx_down:
        #     gene_name = sig_genes[downregulated][idx]
        #     plt.text(sig_log2fc_values[downregulated][idx], sig_neg_log_pvals[downregulated][idx], 
        #              gene_name, fontsize=9, ha='right', va='bottom', color='black')
    
        # # Add significance threshold lines
        # plt.axhline(y=1.5, linestyle="dashed", color="black", alpha=0.7)
        # plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)
    
        # plt.xlabel("log2 Fold Change")
        # plt.ylabel("-log10(p-value)")
        # plt.title(f"Volcano Plot for Cell Type {cluster} under knockout (KO) Condition Comparing {group_name} to Vehicle.")
        # #f"Volcano Plot for Cell Type {cluster} under High-dose (KA25) Condition"
        # plt.savefig(f"VolcanoPlot-Valid_{name_validation}-Cluster_{cluster}_{group_name}_KO_YekBasteh.png")
        # plt.legend()
        # plt.show()

In [ ]:
import pandas as pd
SAMPLE_KEY,CELLTYPE_KEY="sample_id_unique","celltype"
BATCH_KEY,GROUP_KEY,CONDITION_KEY="batch","group","condition"
KEEP_BATCH="batch_2022"
KEEP_CONDITIONS=["Veh","KA3","KA25"]
CASE_GROUP,CONTROL_GROUP="KO","WT"
pb_cells=pseudobulk_adata.copy()
ct=adata_filtered_significant_genes.obs[CELLTYPE_KEY].dropna().astype(str)
common=pb_cells.obs_names.intersection(ct.index)
pb_cells=pb_cells[common].copy()
pb_cells.obs[CELLTYPE_KEY]=ct.loc[pb_cells.obs_names].values
pb_cells=pb_cells[(pb_cells.obs[BATCH_KEY].astype(str)==KEEP_BATCH)&(pb_cells.obs[CONDITION_KEY].astype(str).isin(KEEP_CONDITIONS))&(pb_cells.obs[GROUP_KEY].astype(str).isin([CASE_GROUP,CONTROL_GROUP]))].copy()
print(pb_cells.obs.groupby([CONDITION_KEY,GROUP_KEY,SAMPLE_KEY,CELLTYPE_KEY],observed=True).size().unstack(fill_value=0).to_string())

### 12. Mouse-level pseudobulk differential expression
Sums raw counts across nuclei within each mouse x cell type to give one profile per animal, filters low-count genes, normalises library size, and tests KO versus WT within each treatment condition with Benjamini-Hochberg correction (Fig. S6). Sex-linked genes are excluded.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse, stats

try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except Exception:
    HAS_ADJUST_TEXT = False


# =============================================================================
# CONFIG
# =============================================================================

SAMPLE_KEY = "sample_id_unique"
CELLTYPE_KEY = "celltype"

BATCH_KEY = "batch"
GROUP_KEY = "group"
CONDITION_KEY = "condition"

KEEP_BATCH = "batch_2022"
KEEP_CONDITION = "Veh"

CASE_GROUP = "KO"
CONTROL_GROUP = "WT"

# One-sided Welch t-test:
#
# "greater" tests KO > WT
# "less" tests KO < WT
TEST_ALTERNATIVE = "greater"

MIN_CELLS = 10
MIN_GENE_COUNT = 10

LOG2FC_TH = 0.5
PVAL_TH = 0.05

TOP_N_LABELS = 15

RUN_BALANCED = False
RANDOM_SEED = 24

TARGET_GENES = [
    "Apoe",
    "Rplp1",
    "Fth1"
]

SEX_GENES = [
    "Xist",
    "Tsix",
    "Uty",
    "Eif2s3y",
    "Kdm5d",
    "Ddx3y"
]


if TEST_ALTERNATIVE not in {
    "greater",
    "less"
}:
    raise ValueError(
        "TEST_ALTERNATIVE must be "
        "'greater' or 'less'."
    )


# =============================================================================
# CONDITION-SPECIFIC GENE DICTIONARIES
# =============================================================================
#
# Expected variables:
#
# up_genes_Veh
# down_genes_Veh
#
# up_genes_KA3
# down_genes_KA3
#
# up_genes_KA25
# down_genes_KA25
#
# Supported format 1:
#
# up_genes_Veh = {
#     "Astro": np.array([...]),
#     "CA1": np.array([...]),
#     ...
# }
#
# Supported format 2:
#
# up_genes_Veh = {
#     "Astro": {
#         "KO": np.array([...])
#     },
#     "CA1": {
#         "KO": np.array([...])
#     }
# }
# =============================================================================

GENES_BY_CONDITION = {
    "Veh": {
        "up": up_genes_Veh,
        "down": down_genes_Veh
    },
    "KA3": {
        "up": up_genes_KA3,
        "down": down_genes_KA3
    },
    "KA25": {
        "up": up_genes_KA25,
        "down": down_genes_KA25
    }
}


if KEEP_CONDITION not in GENES_BY_CONDITION:
    raise ValueError(
        f"No predefined gene dictionaries were found for "
        f"KEEP_CONDITION='{KEEP_CONDITION}'."
    )


def normalize_key(value):
    """
    Normalize dictionary keys and cell-type names
    for robust matching.
    """

    return str(value).strip().lower()


def find_matching_key(
    dictionary,
    requested_key
):
    """
    Find a dictionary key using exact matching first,
    then stripped case-insensitive matching.
    """

    if requested_key in dictionary:
        return requested_key

    requested_normalized = normalize_key(
        requested_key
    )

    for key in dictionary.keys():

        if normalize_key(key) == requested_normalized:
            return key

    return None


def convert_gene_values_to_set(
    values
):
    """
    Convert a list, NumPy array, Series, set, or similar
    collection of genes to a clean string set.
    """

    if values is None:
        return set()

    if isinstance(values, str):
        return {
            values
        }

    values_array = np.asarray(
        values,
        dtype=object
    ).ravel()

    output = set()

    for gene in values_array:

        if pd.isna(gene):
            continue

        gene_string = str(
            gene
        ).strip()

        if gene_string:
            output.add(
                gene_string
            )

    return output


def extract_gene_set(
    gene_dictionary,
    celltype,
    group_name=CASE_GROUP
):
    """
    Extract genes for one cell type.

    Supports:

    1. Flat format:
       gene_dictionary["Astro"] = array([...])

    2. Nested format:
       gene_dictionary["Astro"]["KO"] = array([...])
    """

    if not isinstance(
        gene_dictionary,
        dict
    ):
        raise TypeError(
            "Expected a dictionary for predefined genes, "
            f"but received {type(gene_dictionary)}."
        )

    matching_celltype_key = find_matching_key(
        gene_dictionary,
        celltype
    )

    if matching_celltype_key is None:
        return set()

    value = gene_dictionary[
        matching_celltype_key
    ]

    # Handle nested format:
    # dictionary[celltype][group_name]
    if isinstance(
        value,
        dict
    ):

        matching_group_key = find_matching_key(
            value,
            group_name
        )

        if matching_group_key is None:
            return set()

        value = value[
            matching_group_key
        ]

    return convert_gene_values_to_set(
        value
    )


def get_predefined_genes(
    condition,
    celltype
):
    """
    Return predefined upregulated, downregulated,
    and combined genes for one condition and cell type.
    """

    condition_data = GENES_BY_CONDITION[
        condition
    ]

    up_gene_set = extract_gene_set(
        condition_data["up"],
        celltype,
        CASE_GROUP
    )

    down_gene_set = extract_gene_set(
        condition_data["down"],
        celltype,
        CASE_GROUP
    )

    return {
        "up": up_gene_set,
        "down": down_gene_set,
        "all": (
            up_gene_set
            | down_gene_set
        )
    }


print(
    f"Using cell-type-specific predefined genes "
    f"for condition: {KEEP_CONDITION}"
)

print(
    f"One-sided Welch t-test: "
    f"{CASE_GROUP} {TEST_ALTERNATIVE} "
    f"{CONTROL_GROUP}"
)

print(
    "\nAvailable up-gene dictionary keys:"
)

print(
    list(
        GENES_BY_CONDITION[
            KEEP_CONDITION
        ]["up"].keys()
    )
)

print(
    "\nAvailable down-gene dictionary keys:"
)

print(
    list(
        GENES_BY_CONDITION[
            KEEP_CONDITION
        ]["down"].keys()
    )
)


# =============================================================================
# PREPARE RAW DATA
# =============================================================================

pb_cells = pseudobulk_adata.copy()


if "counts" in pb_cells.layers:

    pb_cells.X = (
        pb_cells.layers["counts"]
        .copy()
    )

    print(
        "\nUsing raw counts layer."
    )

else:

    print(
        "\nWARNING: using pseudobulk_adata.X."
    )


# Add cell-type annotation
ct = (
    adata_filtered_significant_genes
    .obs[CELLTYPE_KEY]
    .dropna()
    .astype(str)
)


common = (
    pb_cells.obs_names
    .intersection(
        ct.index
    )
)


pb_cells = pb_cells[
    common
].copy()


pb_cells.obs[CELLTYPE_KEY] = (
    ct.loc[
        pb_cells.obs_names
    ].values
)


# Keep requested batch, condition, and groups
keep = (
    (
        pb_cells.obs[BATCH_KEY]
        .astype(str)
        == KEEP_BATCH
    )
    & (
        pb_cells.obs[CONDITION_KEY]
        .astype(str)
        == KEEP_CONDITION
    )
    & (
        pb_cells.obs[GROUP_KEY]
        .astype(str)
        .isin(
            [
                CASE_GROUP,
                CONTROL_GROUP
            ]
        )
    )
)


pb_cells = pb_cells[
    keep
].copy()


print(
    f"\nAfter batch/condition/group filtering: "
    f"{pb_cells.shape}"
)


# =============================================================================
# REMOVE SEX GENES AND CONVERT TO SPARSE
# =============================================================================

pb_cells = pb_cells[
    :,
    ~pb_cells.var_names
    .astype(str)
    .isin(
        SEX_GENES
    )
].copy()


pb_cells.X = sparse.csr_matrix(
    pb_cells.X
)


print(
    "\npb_cells:",
    pb_cells.shape
)


# =============================================================================
# CELL COUNTS AFTER SUBSAMPLING
# =============================================================================

cell_counts = (
    pb_cells.obs
    .groupby(
        [
            GROUP_KEY,
            SAMPLE_KEY,
            CELLTYPE_KEY
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


print(
    "\nCELL COUNTS PER MOUSE × CELL TYPE"
)


print(
    cell_counts.pivot_table(
        index=[
            GROUP_KEY,
            SAMPLE_KEY
        ],
        columns=CELLTYPE_KEY,
        values="n_cells",
        fill_value=0,
        observed=True
    ).to_string()
)


# =============================================================================
# MAKE PSEUDOBULK
# =============================================================================

def make_pseudobulk(
    adata,
    balanced=False
):
    """
    Create raw pseudobulk sums separately for
    each mouse and each cell type.
    """

    output = {}

    celltypes = sorted(
        adata.obs[CELLTYPE_KEY]
        .astype(str)
        .unique()
    )

    for celltype in celltypes:

        ad_ct = adata[
            adata.obs[CELLTYPE_KEY]
            .astype(str)
            == celltype
        ]

        counts = (
            ad_ct.obs[SAMPLE_KEY]
            .astype(str)
            .value_counts()
        )

        samples = counts[
            counts >= MIN_CELLS
        ].index

        if len(samples) == 0:

            print(
                f"{celltype}: no samples have "
                f"at least {MIN_CELLS} cells."
            )

            continue

        target_n = (
            int(
                counts.loc[
                    samples
                ].min()
            )
            if balanced
            else None
        )

        expression_rows = []
        metadata_rows = []

        for sample in samples:

            ad_mouse = ad_ct[
                ad_ct.obs[SAMPLE_KEY]
                .astype(str)
                == sample
            ].copy()

            original_n = (
                ad_mouse.n_obs
            )

            if balanced:

                local_seed = (
                    RANDOM_SEED
                    + sum(
                        map(
                            ord,
                            str(sample)
                            + str(celltype)
                        )
                    )
                )

                local_rng = np.random.default_rng(
                    local_seed
                )

                selected_positions = (
                    local_rng.choice(
                        original_n,
                        size=target_n,
                        replace=False
                    )
                )

                ad_mouse = ad_mouse[
                    selected_positions
                ].copy()

            metadata_row = (
                ad_mouse.obs.iloc[0]
            )

            expression_rows.append(
                np.asarray(
                    ad_mouse.X.sum(
                        axis=0
                    )
                ).ravel()
            )

            metadata_rows.append({
                SAMPLE_KEY: str(
                    sample
                ),
                GROUP_KEY: str(
                    metadata_row[
                        GROUP_KEY
                    ]
                ),
                "original_n_cells": (
                    original_n
                ),
                "used_n_cells": (
                    ad_mouse.n_obs
                )
            })

        expression = pd.DataFrame(
            np.vstack(
                expression_rows
            ),
            index=[
                row[SAMPLE_KEY]
                for row in metadata_rows
            ],
            columns=(
                adata.var_names
                .astype(str)
            )
        )

        metadata = pd.DataFrame(
            metadata_rows,
            index=expression.index
        )

        output[celltype] = (
            expression,
            metadata
        )

    return output


# =============================================================================
# MULTIPLE-TESTING ADJUSTMENT
# =============================================================================

def bh_adjust(
    p_values
):
    """
    Benjamini-Hochberg FDR correction.
    """

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    if len(p_values) == 0:

        return np.asarray(
            [],
            dtype=float
        )

    order = np.argsort(
        p_values
    )

    ordered_p = p_values[
        order
    ]

    adjusted = (
        ordered_p
        * len(p_values)
        / np.arange(
            1,
            len(p_values) + 1
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    output = np.empty_like(
        adjusted
    )

    output[
        order
    ] = np.clip(
        adjusted,
        0,
        1
    )

    return output


# =============================================================================
# TARGET-GENE SUMMARY
# =============================================================================

def print_target_summary(
    pb,
    analysis_name
):
    """
    Print mouse-level raw pseudobulk sums and
    one-sided Welch t-tests for TARGET_GENES.
    """

    mouse_rows = []
    comparison_rows = []

    for celltype, (
        expression,
        metadata
    ) in pb.items():

        available_target_genes = [
            gene
            for gene in TARGET_GENES
            if gene in expression.columns
        ]

        for sample in expression.index:

            group = str(
                metadata.loc[
                    sample,
                    GROUP_KEY
                ]
            )

            n_cells = int(
                metadata.loc[
                    sample,
                    "used_n_cells"
                ]
            )

            for gene in available_target_genes:

                gene_sum = float(
                    expression.loc[
                        sample,
                        gene
                    ]
                )

                mouse_rows.append({
                    GROUP_KEY: group,
                    SAMPLE_KEY: sample,
                    CELLTYPE_KEY: celltype,
                    "gene": gene,
                    "n_cells": n_cells,
                    "mouse_sum": gene_sum
                })

    mouse_summary = pd.DataFrame(
        mouse_rows
    )

    print(
        "\n"
        + "=" * 120
    )

    print(
        f"TARGET-GENE MOUSE-LEVEL "
        f"PSEUDOBULK SUMS: "
        f"{analysis_name}"
    )

    print(
        "=" * 120
    )

    if mouse_summary.empty:

        print(
            "No target genes were found."
        )

        return (
            mouse_summary,
            pd.DataFrame()
        )

    mouse_summary = (
        mouse_summary
        .sort_values(
            [
                "gene",
                CELLTYPE_KEY,
                GROUP_KEY,
                SAMPLE_KEY
            ]
        )
        .reset_index(
            drop=True
        )
    )

    for gene in TARGET_GENES:

        gene_data = mouse_summary[
            mouse_summary["gene"]
            == gene
        ]

        if gene_data.empty:

            print(
                f"\n{gene}: not found"
            )

            continue

        print(
            "\n"
            + "-" * 120
        )

        print(
            f"GENE: {gene} — "
            "MOUSE-LEVEL SUMS"
        )

        print(
            "-" * 120
        )

        print(
            gene_data[
                [
                    GROUP_KEY,
                    SAMPLE_KEY,
                    CELLTYPE_KEY,
                    "n_cells",
                    "mouse_sum"
                ]
            ].to_string(
                index=False,
                float_format=(
                    lambda value:
                    f"{value:.6f}"
                )
            )
        )

    # -------------------------------------------------------------------------
    # Target-gene KO-versus-WT tests
    # -------------------------------------------------------------------------

    for celltype in sorted(
        mouse_summary[
            CELLTYPE_KEY
        ].unique()
    ):

        for gene in TARGET_GENES:

            subset = mouse_summary[
                (
                    mouse_summary[
                        CELLTYPE_KEY
                    ]
                    == celltype
                )
                & (
                    mouse_summary[
                        "gene"
                    ]
                    == gene
                )
            ]

            if subset.empty:
                continue

            case_sums = subset.loc[
                subset[GROUP_KEY]
                == CASE_GROUP,
                "mouse_sum"
            ].to_numpy(
                dtype=float
            )

            control_sums = subset.loc[
                subset[GROUP_KEY]
                == CONTROL_GROUP,
                "mouse_sum"
            ].to_numpy(
                dtype=float
            )

            n_case = len(
                case_sums
            )

            n_control = len(
                control_sums
            )

            if (
                n_case == 0
                or n_control == 0
            ):
                continue

            case_mean_sum = (
                case_sums.mean()
            )

            control_mean_sum = (
                control_sums.mean()
            )

            fold_change = (
                case_mean_sum
                / control_mean_sum
                if control_mean_sum > 0
                else np.inf
            )

            log2fc_ratio = np.log2(
                (
                    case_mean_sum + 1
                )
                / (
                    control_mean_sum + 1
                )
            )

            case_log_sums = np.log2(
                case_sums + 1
            )

            control_log_sums = np.log2(
                control_sums + 1
            )

            volcano_log2fc = (
                case_log_sums.mean()
                - control_log_sums.mean()
            )

            if (
                n_case >= 2
                and n_control >= 2
            ):

                t_stat, p_value = (
                    stats.ttest_ind(
                        case_log_sums,
                        control_log_sums,
                        equal_var=False,
                        nan_policy="omit",
                        alternative=(
                            TEST_ALTERNATIVE
                        )
                    )
                )

                if not np.isfinite(
                    t_stat
                ):
                    t_stat = 0.0

                if not np.isfinite(
                    p_value
                ):
                    p_value = 1.0

            else:

                t_stat = np.nan
                p_value = np.nan

            comparison_rows.append({
                CELLTYPE_KEY: celltype,
                "gene": gene,
                "test_alternative": (
                    TEST_ALTERNATIVE
                ),
                "n_KO_mice": n_case,
                "n_WT_mice": n_control,
                "KO_mean_mouse_sum": (
                    case_mean_sum
                ),
                "WT_mean_mouse_sum": (
                    control_mean_sum
                ),
                "fold_change_KO_over_WT": (
                    fold_change
                ),
                "log2FC_ratio_of_mean_sums": (
                    log2fc_ratio
                ),
                "volcano_log2FC": (
                    volcano_log2fc
                ),
                "t_stat": t_stat,
                "p_value": p_value
            })

    comparison_summary = pd.DataFrame(
        comparison_rows
    )

    if comparison_summary.empty:

        print(
            "\nNo target-gene comparisons "
            "could be calculated."
        )

        return (
            mouse_summary,
            comparison_summary
        )

    comparison_summary[
        "padj"
    ] = np.nan

    valid_mask = comparison_summary[
        "p_value"
    ].notna()

    if valid_mask.any():

        comparison_summary.loc[
            valid_mask,
            "padj"
        ] = bh_adjust(
            comparison_summary.loc[
                valid_mask,
                "p_value"
            ].to_numpy(
                dtype=float
            )
        )

    comparison_summary = (
        comparison_summary
        .sort_values(
            [
                "gene",
                CELLTYPE_KEY
            ]
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "\n"
        + "=" * 170
    )

    print(
        f"TARGET-GENE "
        f"{CASE_GROUP} VERSUS "
        f"{CONTROL_GROUP} COMPARISON — "
        f"ONE-SIDED TEST: "
        f"{TEST_ALTERNATIVE.upper()}"
    )

    print(
        "=" * 170
    )

    for gene in TARGET_GENES:

        gene_results = comparison_summary[
            comparison_summary["gene"]
            == gene
        ]

        if gene_results.empty:
            continue

        print(
            "\n"
            + "-" * 170
        )

        print(
            f"GENE: {gene} — "
            f"{CASE_GROUP} "
            f"{TEST_ALTERNATIVE} "
            f"{CONTROL_GROUP}"
        )

        print(
            "-" * 170
        )

        print(
            gene_results[
                [
                    CELLTYPE_KEY,
                    "test_alternative",
                    "n_KO_mice",
                    "n_WT_mice",
                    "KO_mean_mouse_sum",
                    "WT_mean_mouse_sum",
                    "fold_change_KO_over_WT",
                    "log2FC_ratio_of_mean_sums",
                    "volcano_log2FC",
                    "t_stat",
                    "p_value",
                    "padj"
                ]
            ].to_string(
                index=False,
                float_format=(
                    lambda value:
                    f"{value:.6g}"
                )
            )
        )

    return (
        mouse_summary,
        comparison_summary
    )


# =============================================================================
# DIFFERENTIAL EXPRESSION
# =============================================================================

def pseudobulk_de(
    pb,
    celltype
):
    """
    Run one-sided Welch t-tests using mouse-level
    log2(raw pseudobulk sum + 1).
    """

    expression, metadata = pb[
        celltype
    ]

    X = expression.to_numpy(
        dtype=float
    )

    genes = (
        expression.columns
        .astype(str)
        .to_numpy()
    )

    # Keep genes with sufficient total counts
    keep_gene = (
        X.sum(
            axis=0
        )
        >= MIN_GENE_COUNT
    )

    X = X[
        :,
        keep_gene
    ]

    genes = genes[
        keep_gene
    ]

    if X.shape[1] == 0:

        print(
            f"{celltype}: no genes passed "
            f"MIN_GENE_COUNT={MIN_GENE_COUNT}."
        )

        return None

    case_mask = (
        metadata[GROUP_KEY]
        .astype(str)
        .to_numpy()
        == CASE_GROUP
    )

    control_mask = (
        metadata[GROUP_KEY]
        .astype(str)
        .to_numpy()
        == CONTROL_GROUP
    )

    print(
        f"\n{celltype}: "
        f"{CASE_GROUP}={case_mask.sum()}, "
        f"{CONTROL_GROUP}={control_mask.sum()}"
    )

    if (
        case_mask.sum() < 2
        or control_mask.sum() < 2
    ):
        return None

    X_log = np.log2(
        X + 1
    )

    log2fc = (
        X_log[
            case_mask
        ].mean(
            axis=0
        )
        - X_log[
            control_mask
        ].mean(
            axis=0
        )
    )

    t_stat, p_value = stats.ttest_ind(
        X_log[
            case_mask
        ],
        X_log[
            control_mask
        ],
        axis=0,
        equal_var=False,
        nan_policy="omit",
        alternative=TEST_ALTERNATIVE
    )

    t_stat = np.asarray(
        t_stat
    ).ravel()

    p_value = np.asarray(
        p_value
    ).ravel()

    t_stat = np.where(
        np.isfinite(
            t_stat
        ),
        t_stat,
        0.0
    )

    p_value = np.where(
        np.isfinite(
            p_value
        ),
        p_value,
        1.0
    )

    result = pd.DataFrame({
        "gene": genes,
        "log2FC": log2fc,
        "t_stat": t_stat,
        "p_value": p_value,
        "padj": bh_adjust(
            p_value
        )
    })

    result[
        "neg_log10_p"
    ] = -np.log10(
        np.maximum(
            result["p_value"],
            1e-300
        )
    )

    return (
        result
        .sort_values(
            "p_value"
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# VOLCANO PLOT
# =============================================================================

def plot_volcano(
    result,
    title
):
    """
    Plot one cell type's filtered genes.
    """

    if TEST_ALTERNATIVE == "greater":

        significant_mask = (
            (
                result["p_value"]
                < PVAL_TH
            )
            & (
                result["log2FC"]
                >= LOG2FC_TH
            )
        )

    elif TEST_ALTERNATIVE == "less":

        significant_mask = (
            (
                result["p_value"]
                < PVAL_TH
            )
            & (
                result["log2FC"]
                <= -LOG2FC_TH
            )
        )

    else:

        raise ValueError(
            "TEST_ALTERNATIVE must be "
            "'greater' or 'less'."
        )

    up = result[
        significant_mask
        & (
            result["log2FC"]
            > 0
        )
    ].copy()

    down = result[
        significant_mask
        & (
            result["log2FC"]
            < 0
        )
    ].copy()

    print(
        f"\nUpregulated: {len(up)}, "
        f"Downregulated: {len(down)}"
    )

    print(
        f"One-sided direction: "
        f"{CASE_GROUP} "
        f"{TEST_ALTERNATIVE} "
        f"{CONTROL_GROUP}"
    )

    # print(
    #     f"ALL SIGNIFICANT UPREGULATED GENES — {celltype}"
    # )
    print(
        "\n"
        + "=" * 100
    )
    
    print(
        title
    )
    
    print(
        "=" * 100
    )
    
    print(
        "\nALL SIGNIFICANT UPREGULATED GENES"
    )

    if up.empty:

        print(
            "None"
        )

    else:

        print(
            up[
                [
                    "gene",
                    "predefined_direction",
                    "log2FC",
                    "t_stat",
                    "p_value",
                    "padj"
                ]
            ]
            .sort_values(
                "p_value"
            )
            .to_string(
                index=False,
                float_format=(
                    lambda value:
                    f"{value:.6g}"
                )
            )
        )

    print(
        "\nALL SIGNIFICANT DOWNREGULATED GENES"
    )

    if down.empty:

        print(
            "None"
        )

    else:

        print(
            down[
                [
                    "gene",
                    "predefined_direction",
                    "log2FC",
                    "t_stat",
                    "p_value",
                    "padj"
                ]
            ]
            .sort_values(
                "p_value"
            )
            .to_string(
                index=False,
                float_format=(
                    lambda value:
                    f"{value:.6g}"
                )
            )
        )

    target_up = up[
        up["gene"].isin(
            TARGET_GENES
        )
    ]

    target_down = down[
        down["gene"].isin(
            TARGET_GENES
        )
    ]

    significant_targets = pd.concat(
        [
            target_up,
            target_down
        ],
        ignore_index=True
    )

    print(
        "\nSIGNIFICANT TARGET GENES"
    )

    if significant_targets.empty:

        print(
            "None"
        )

    else:

        print(
            significant_targets[
                [
                    "gene",
                    "predefined_direction",
                    "log2FC",
                    "t_stat",
                    "p_value",
                    "padj"
                ]
            ].to_string(
                index=False,
                float_format=(
                    lambda value:
                    f"{value:.6g}"
                )
            )
        )

    plt.figure(
        figsize=(
            10,
            8
        )
    )

    # All retained predefined genes
    plt.scatter(
        result["log2FC"],
        result["neg_log10_p"],
        s=15,
        alpha=0.5,
        label="Not significant"
    )

    # Significant up genes
    if not up.empty:

        plt.scatter(
            up["log2FC"],
            up["neg_log10_p"],
            s=30,
            label="Significant up"
        )

    # Significant down genes
    if not down.empty:

        plt.scatter(
            down["log2FC"],
            down["neg_log10_p"],
            s=30,
            label="Significant down"
        )

    plt.axhline(
        -np.log10(
            PVAL_TH
        ),
        linestyle="--"
    )

    plt.axvline(
        LOG2FC_TH,
        linestyle="--"
    )

    plt.axvline(
        -LOG2FC_TH,
        linestyle="--"
    )

    # labels = pd.concat(
    #     [
    #         up.nsmallest(
    #             TOP_N_LABELS,
    #             "p_value"
    #         ),
    #         down.nsmallest(
    #             TOP_N_LABELS,
    #             "p_value"
    #         ),
    #         target_up,
    #         target_down
    #     ],
    #     ignore_index=True
    # ).drop_duplicates(
    #     "gene"
    # )
    
    # Label every gene passing the p-value and log2FC thresholds
    labels = pd.concat(
        [
            up,
            down
        ],
        ignore_index=True
    ).drop_duplicates(
        "gene"
    )

    texts = [
        plt.text(
            row.log2FC,
            row.neg_log10_p,
            row.gene,
            fontsize=8
        )
        for row in labels.itertuples()
    ]

    # if (
    #     HAS_ADJUST_TEXT
    #     and texts
    # ):

    #     adjust_text(
    #         texts,
    #         ax=plt.gca()
    #     )
    
    if HAS_ADJUST_TEXT and texts:
    
        adjust_text(
            texts,
            ax=plt.gca(),
    
            # Draw a line/arrow from moved label to its gene point
            arrowprops=dict(
                arrowstyle="-",
                color="gray",
                linewidth=0.6,
                alpha=0.8
            ),
    
            # Push labels away from points and from each other
            force_points=(1.5, 2.0),
            force_text=(1.5, 2.0),
    
            # Add spacing around points and labels
            expand_points=(1.3, 1.5),
            expand_text=(1.3, 1.5),
    
            # Labels can move in both directions
            only_move={
                "points": "xy",
                "text": "xy"
            },
    
            # More optimization iterations
            iter_lim=1000
        )

    plt.xlabel(
        f"log2FC "
        f"({CASE_GROUP} vs. "
        f"{CONTROL_GROUP})"
    )

    plt.ylabel(
        "-log10(adj p-value)"
    )

    plt.title(
        title
    )

    plt.legend()



    plt.tight_layout()
    celltype = str(
        result["celltype"].iloc[0]
    )
    filename = (
        f"Primary_pseudobulk_"
        f"{celltype}_"
        f"{CASE_GROUP}_vs_{CONTROL_GROUP}_"
        f"{KEEP_CONDITION}_Revision.png"
    )
    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()


# =============================================================================
# RUN ANALYSIS
# =============================================================================

def run_analysis(
    balanced=False
):
    """
    Run pseudobulk analysis for each cell type.

    For each cell type, retain only genes in:

    up_genes_CONDITION[celltype]
    union
    down_genes_CONDITION[celltype]
    """

    analysis_name = (
        "Balanced-cell sensitivity"
        if balanced
        else "Primary pseudobulk"
    )

    pseudobulk = make_pseudobulk(
        pb_cells,
        balanced=balanced
    )

    (
        target_summary,
        target_comparison
    ) = print_target_summary(
        pseudobulk,
        analysis_name
    )

    results = []

    for celltype in sorted(
        pseudobulk
    ):

        result = pseudobulk_de(
            pseudobulk,
            celltype
        )

        if result is None:
            continue

        n_before = len(
            result
        )

        predefined = get_predefined_genes(
            KEEP_CONDITION,
            celltype
        )

        predefined_up = predefined[
            "up"
        ]

        predefined_down = predefined[
            "down"
        ]

        genes_to_keep = predefined[
            "all"
        ]

        print(
            f"\n{celltype}: "
            f"{len(predefined_up)} predefined up genes, "
            f"{len(predefined_down)} predefined down genes, "
            f"{len(genes_to_keep)} total genes "
            f"for {KEEP_CONDITION}"
        )

        if len(
            genes_to_keep
        ) == 0:

            print(
                f"{celltype}: no matching predefined "
                f"genes were found."
            )

            print(
                f"Requested cell-type key: "
                f"{repr(celltype)}"
            )

            print(
                "Available up keys:"
            )

            print(
                [
                    repr(key)
                    for key in GENES_BY_CONDITION[
                        KEEP_CONDITION
                    ]["up"].keys()
                ]
            )

            print(
                "Available down keys:"
            )

            print(
                [
                    repr(key)
                    for key in GENES_BY_CONDITION[
                        KEEP_CONDITION
                    ]["down"].keys()
                ]
            )

            continue

        result = result[
            result["gene"]
            .astype(str)
            .isin(
                genes_to_keep
            )
        ].copy()

        n_after = len(
            result
        )

        print(
            f"{celltype}: "
            f"volcano gene filtering "
            f"{n_before} -> {n_after} genes"
        )

        if result.empty:

            print(
                f"{celltype}: predefined genes existed, "
                f"but none survived "
                f"MIN_GENE_COUNT={MIN_GENE_COUNT}."
            )

            continue

        result[
            "predefined_direction"
        ] = np.select(
            [
                result["gene"]
                .astype(str)
                .isin(
                    predefined_up
                ),
                result["gene"]
                .astype(str)
                .isin(
                    predefined_down
                )
            ],
            [
                "up",
                "down"
            ],
            default="unknown"
        )

        result[
            "celltype"
        ] = celltype

        result[
            "analysis"
        ] = analysis_name

        result[
            "condition"
        ] = KEEP_CONDITION

        result[
            "test_alternative"
        ] = TEST_ALTERNATIVE

        results.append(
            result
        )

    # Plot every retained cell type
    for result in results:

        celltype = str(
            result[
                "celltype"
            ].iloc[0]
        )

        plot_volcano(
            result,
            f"{analysis_name}: {celltype}\n"
            f"{CASE_GROUP} vs "
            f"{CONTROL_GROUP}, "
            f"{KEEP_CONDITION}\n"
            f"Cell-type-specific predefined genes; "
            f"one-sided: {TEST_ALTERNATIVE}"
        )

    results_df = (
        pd.concat(
            results,
            ignore_index=True
        )
        if results
        else pd.DataFrame()
    )

    return (
        results_df,
        target_summary,
        target_comparison,
        pseudobulk
    )


# =============================================================================
# PRIMARY ANALYSIS
# =============================================================================

(
    primary_results_df,
    primary_target_summary_df,
    primary_target_comparison_df,
    primary_pb
) = run_analysis(
    balanced=False
)


# =============================================================================
# OPTIONAL BALANCED-CELL ANALYSIS
# =============================================================================

if RUN_BALANCED:

    (
        balanced_results_df,
        balanced_target_summary_df,
        balanced_target_comparison_df,
        balanced_pb
    ) = run_analysis(
        balanced=True
    )

else:

    balanced_results_df = (
        pd.DataFrame()
    )

    balanced_target_summary_df = (
        pd.DataFrame()
    )

    balanced_target_comparison_df = (
        pd.DataFrame()
    )

    balanced_pb = {}


# =============================================================================
# FINAL OUTPUT SHAPES
# =============================================================================

print(
    "\nPrimary filtered DE results:",
    primary_results_df.shape
)

print(
    "Primary target mouse-level summary:",
    primary_target_summary_df.shape
)

print(
    "Primary target comparison:",
    primary_target_comparison_df.shape
)

print(
    "Balanced filtered DE results:",
    balanced_results_df.shape
)

### 13. Pseudobulk cell counts
Reports the nuclei contributing to each pseudobulk profile.

In [ ]:
# =============================================================================
# CELL COUNTS PER MOUSE x CELL TYPE
# =============================================================================
pb_cells = pseudobulk_adata.copy()
# =============================================================================
# CELL COUNTS PER MOUSE x CELL TYPE
# =============================================================================

if CELLTYPE_KEY not in pb_cells.obs.columns:
    print(
        f"\nWARNING: '{CELLTYPE_KEY}' not found in pb_cells.obs. "
        f"Skipping per-mouse counts.\n"
        f"Available columns: {list(pb_cells.obs.columns)}"
    )
else:
    cell_counts_before = (
        pb_cells.obs
        .groupby(
            [
                GROUP_KEY,
                SAMPLE_KEY,
                CELLTYPE_KEY
            ],
            observed=True
        )
        .size()
        .rename(
            "n_cells"
        )
        .reset_index()
    )

    print(
        "\nCELL COUNTS PER MOUSE x CELL TYPE"
    )

    print(
        cell_counts_before.pivot_table(
            index=[
                GROUP_KEY,
                SAMPLE_KEY
            ],
            columns=CELLTYPE_KEY,
            values="n_cells",
            fill_value=0,
            observed=True
        ).to_string()
    )


### 14. Metacell construction
Partitions nuclei by k-means on the first 30 principal components within each mouse x cell type, targeting ~15 nuclei per metacell, so that nuclei from different animals are never aggregated together.

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad, scanpy as sc
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

SAMPLE_KEY, CELLTYPE_KEY = "sample_id_unique", "celltype"
K_METACELL, MIN_CELLS, N_PCS = 15, 10, 30
src = adata_filtered_significant_genes


def _raw_counts(a):
    if "counts" in a.layers: return a.layers["counts"]
    if a.raw is not None and set(a.var_names).issubset(set(a.raw.var_names)):
        return a.raw[:, a.var_names].X
    print("  ! No raw counts found — aggregating adata.X (normalized) instead.")
    return a.X


def build_metacells(a, k=K_METACELL, n_pcs=N_PCS, min_cells=MIN_CELLS, seed=0):
    X = _raw_counts(a); X = sp.csr_matrix(X) if not sp.issparse(X) else X.tocsr()
    if "X_pca_harmony" in a.obsm: emb = np.asarray(a.obsm["X_pca_harmony"])
    elif "X_pca" in a.obsm:       emb = np.asarray(a.obsm["X_pca"])
    else:
        print("  ! No PCA in .obsm — computing one.")
        t = a.copy(); sc.pp.scale(t, max_value=10); sc.tl.pca(t, n_comps=n_pcs)
        emb = t.obsm["X_pca"]
    emb = emb[:, :min(n_pcs, emb.shape[1])]

    rows, meta, skipped = [], [], []
    for (samp, ct), idx in a.obs.groupby([SAMPLE_KEY, CELLTYPE_KEY], observed=True).indices.items():
        n = len(idx)
        if n < min_cells: skipped.append(f"{samp}/{ct}(n={n})"); continue
        n_mc = max(1, int(round(n / k)))
        lab = (np.zeros(n, int) if n_mc == 1 else
               KMeans(n_clusters=n_mc, n_init=10, random_state=seed).fit_predict(emb[idx]))
        for c in range(n_mc):
            mem = idx[lab == c]
            if len(mem) == 0: continue
            rows.append(np.asarray(X[mem].sum(axis=0)).ravel())
            meta.append({"metacell_id": f"{samp}|{ct}|mc{c}", SAMPLE_KEY: samp,
                         CELLTYPE_KEY: ct, "n_cells": len(mem),
                         "group": a.obs["group"].iloc[mem[0]],
                         "condition": a.obs["condition"].iloc[mem[0]]})
    if skipped:
        print(f"  skipped {len(skipped)} group(s) below {min_cells} nuclei: {', '.join(skipped)}")

    mc = ad.AnnData(sp.csr_matrix(np.vstack(rows)),
                    obs=pd.DataFrame(meta).set_index("metacell_id"), var=a.var.copy())
    sc.pp.normalize_total(mc, target_sum=1e4); sc.pp.log1p(mc)
    return mc


metacells = build_metacells(src)
print(f"\n{metacells.n_obs:,} metacells from {src.n_obs:,} nuclei (k={K_METACELL})\n")
print(metacells.obs.groupby(CELLTYPE_KEY, observed=True)
        .agg(metacells=("n_cells","size"), median_nuclei=("n_cells","median"),
             min_nuclei=("n_cells","min")).to_string())
print("\nMetacells per mouse x cell type:")
print(pd.crosstab(metacells.obs[SAMPLE_KEY], metacells.obs[CELLTYPE_KEY]).to_string())

In [ ]:
# B
def sparsity_qc(a, mc, n_genes=2000, seed=0):
    rng = np.random.default_rng(seed); out = []
    for ct in sorted(mc.obs[CELLTYPE_KEY].unique()):
        A = a[a.obs[CELLTYPE_KEY] == ct]; M = mc[mc.obs[CELLTYPE_KEY] == ct]
        g = rng.choice(A.n_vars, size=min(n_genes, A.n_vars), replace=False)
        x = A.X[:, g]; x = x.toarray() if sp.issparse(x) else np.asarray(x)
        y = M.X[:, g]; y = y.toarray() if sp.issparse(y) else np.asarray(y)
        cx = np.corrcoef(x[:, x.std(0) > 0], rowvar=False)
        cy = np.corrcoef(y[:, y.std(0) > 0], rowvar=False)
        ix, iy = np.triu_indices_from(cx, 1), np.triu_indices_from(cy, 1)
        out.append({"cell_type": ct, "nuclei": A.n_obs, "metacells": M.n_obs,
                    "sparsity_nuclei": (x == 0).mean(),
                    "sparsity_metacells": (y == 0).mean(),
                    "pct_r_near0_nuclei": (np.abs(cx[ix]) < 0.05).mean() * 100,
                    "pct_r_near0_metacells": (np.abs(cy[iy]) < 0.05).mean() * 100,
                    "_cx": cx[ix], "_cy": cy[iy]})
    return pd.DataFrame(out)


qc = sparsity_qc(src, metacells)
print(qc.drop(columns=["_cx", "_cy"]).round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
w = 0.38; xp = np.arange(len(qc))
axes[0].bar(xp - w/2, qc.sparsity_nuclei * 100, w, label="nuclei")
axes[0].bar(xp + w/2, qc.sparsity_metacells * 100, w, label="metacells")
axes[0].set_xticks(xp); axes[0].set_xticklabels(qc.cell_type, rotation=45, ha="right")
axes[0].set_ylabel("% zeros"); axes[0].set_title("Expression matrix sparsity")
axes[0].legend(frameon=False)
for _, r in qc.iterrows():
    axes[1].hist(r._cx, bins=80, density=True, histtype="step", alpha=.5, color="grey")
    axes[1].hist(r._cy, bins=80, density=True, histtype="step", alpha=.9)
axes[1].set_xlabel("gene-gene Pearson r"); axes[1].set_ylabel("density")
axes[1].set_title("grey = nuclei, coloured = metacells")
plt.tight_layout(); plt.show()

In [ ]:
# C
def _z(X):
    """Standardize columns. Zero-variance columns become all-zero instead of NaN."""
    X = np.asarray(X, dtype=np.float64)
    sd = X.std(0); ok = sd > 0
    Z = np.zeros_like(X)
    Z[:, ok] = (X[:, ok] - X.mean(0)[ok]) / sd[ok]
    return Z


def _stats(Xr, Xt, beta):
    n = Xr.shape[1]
    Zr, Zt = _z(Xr), _z(Xt)
    Cr = np.clip((Zr.T @ Zr) / max(Xr.shape[0] - 1, 1), -1, 1)
    Ct = np.clip((Zt.T @ Zt) / max(Xt.shape[0] - 1, 1), -1, 1)
    iu = np.triu_indices(n, 1)
    cr, ct = Cr[iu], Ct[iu]
    ar, at = np.abs(cr) ** beta, np.abs(ct) ** beta

    def _pc1_kme(Z):
        if not np.isfinite(Z).all() or Z.std() == 0:
            return 0.0, np.zeros(Z.shape[1])
        u, sv, _ = np.linalg.svd(Z, full_matrices=False)
        tot = np.sum(sv ** 2)
        pve = float(sv[0] ** 2 / tot) if tot > 0 else 0.0
        me = u[:, 0]; s = me.std()
        mez = (me - me.mean()) / s if s > 0 else np.zeros_like(me)
        return pve, np.abs((Z.T @ mez) / max(len(mez) - 1, 1))

    pve_t, kME_t = _pc1_kme(Zt)
    _,     kME_r = _pc1_kme(Zr)

    Ar = np.abs(Cr) ** beta; np.fill_diagonal(Ar, 0)
    At = np.abs(Ct) ** beta; np.fill_diagonal(At, 0)

    def _c(a, b):
        a, b = np.asarray(a, float), np.asarray(b, float)
        k = np.isfinite(a) & np.isfinite(b)
        if k.sum() < 3 or np.std(a[k]) == 0 or np.std(b[k]) == 0: return 0.0
        v = np.corrcoef(a[k], b[k])[0, 1]
        return 0.0 if not np.isfinite(v) else float(v)

    return dict(meanCor=float(np.nanmean(np.abs(ct))), meanAdj=float(np.nanmean(at)),
                propVarExplained=pve_t, meanKME=float(np.nanmean(kME_t)),
                cor_kIM=_c(Ar.sum(1), At.sum(1)), cor_kME=_c(kME_r, kME_t),
                cor_cor=_c(cr, ct), cor_adj=_c(ar, at))


def module_preservation(ref_df, test_df, colors, beta=2, n_perm=50,
                        max_module_genes=400, max_ref_samples=3000,
                        min_module_genes=10, seed=0, verbose=True):
    """Zsummary > 10 strong, 2-10 moderate, < 2 not preserved."""
    COLS = ["module","moduleSize","genesTested","Zdensity",
            "Zconnectivity","Zsummary","n_stat_dropped"]
    rng = np.random.default_rng(seed)
    genes = [g for g in colors.index if g in ref_df.columns and g in test_df.columns]
    if len(genes) == 0:
        if verbose: print("    no testable genes")
        return pd.DataFrame(columns=COLS)
    colors = colors.loc[genes]
    R = ref_df[genes]
    if R.shape[0] > max_ref_samples:
        R = R.iloc[rng.choice(R.shape[0], max_ref_samples, replace=False)]
    Rv = R.values.astype(np.float64)
    Tv = test_df[genes].values.astype(np.float64)
    gi = {g: i for i, g in enumerate(genes)}
    DENS = ["meanCor","meanAdj","propVarExplained","meanKME"]
    CONN = ["cor_kIM","cor_kME","cor_cor","cor_adj"]

    out = []
    for mod, grp in colors.groupby(colors):
        idx = np.array([gi[g] for g in grp.index]); size = len(idx)
        if size < min_module_genes: continue
        use = idx if size <= max_module_genes else rng.choice(idx, max_module_genes, replace=False)
        obs = _stats(Rv[:, use], Tv[:, use], beta)
        null = pd.DataFrame([_stats(Rv[:, p], Tv[:, p], beta)
                             for p in (rng.choice(len(genes), len(use), replace=False)
                                       for _ in range(n_perm))])
        Z = {}
        for k in DENS + CONN:
            sd, mu = null[k].std(ddof=1), null[k].mean()
            Z[k] = (np.nan if (not np.isfinite(obs[k]) or not np.isfinite(sd) or sd == 0)
                    else (obs[k] - mu) / sd)
        zd = np.nanmedian([Z[k] for k in DENS]); zc = np.nanmedian([Z[k] for k in CONN])
        zd = 0.0 if not np.isfinite(zd) else float(zd)
        zc = 0.0 if not np.isfinite(zc) else float(zc)
        nbad = sum(1 for k in DENS + CONN if not np.isfinite(Z[k]))
        out.append(dict(module=mod, moduleSize=size, genesTested=len(use),
                        Zdensity=zd, Zconnectivity=zc, Zsummary=(zd + zc) / 2,
                        n_stat_dropped=nbad))
        if verbose:
            flag = f"   ({nbad}/8 stats degenerate)" if nbad else ""
            print(f"    {mod:<18} n={size:<6} Zsummary={((zd + zc) / 2):7.2f}{flag}")
    if not out: return pd.DataFrame(columns=COLS)
    return pd.DataFrame(out).sort_values("Zsummary", ascending=False).reset_index(drop=True)


print("Cell C loaded: _z, _stats, module_preservation")

In [ ]:
# D
import PyWGCNA, os
import numpy as np, pandas as pd, scipy.sparse as sp

FILES = {'CA1':"./CA1 modules_YekBasteh.p", 'CA3':"./CA3 modules_YekBasteh.p",
         'DG':"./DG modules_YekBasteh.p", 'Inhibitory':"./Inhibitory modules_YekBasteh.p",
         'Astro':"./Astro modules_YekBasteh.p", 'Oligo':"./Oligo modules_YekBasteh.p",
         'Microglia':"./Microglia modules_YekBasteh.p"}
NEURON_ORDER, GLIA_ORDER = ['CA1','CA3','DG','Inhibitory'], ['Astro','Oligo','Microglia']
SOFT_POWER = {'CA1':2,'CA3':2,'DG':4,'Inhibitory':2,'Oligo':2,'Astro':2,'Microglia':2}
GREY_FRACTION, N_PERM, CELLTYPE_KEY = 0.75, 50, "celltype" # 0.3, 50, "celltype"

W = {}
for ct, f in FILES.items():
    if os.path.exists(f): W[ct] = PyWGCNA.readWGCNA(f)
    else: print(f"[{ct}] MISSING {f} — re-run findModules for this cell type.")

def _base(order):
    m, i = {}, 1
    for ct in order:
        if ct not in W: continue
        for color in W[ct].datExpr.var['moduleColors'].unique():
            if color not in m: m[color] = i; i += 1
    return m

NEURON, GLIA = _base(NEURON_ORDER), _base(GLIA_ORDER)
COLOR2IDX = {}
for ct in NEURON_ORDER:
    m = NEURON.copy()
    if ct != 'CA3' and 'dimgrey' in m and 'darkgrey' in m:
        m['dimgrey'], m['darkgrey'] = m['darkgrey'], m['dimgrey']
    COLOR2IDX[ct] = m
for ct in GLIA_ORDER:
    m = GLIA.copy()
    if 'dimgrey' in m and 'gainsboro' in m:
        m['dimgrey'], m['gainsboro'] = m['gainsboro'], m['dimgrey']
    COLOR2IDX[ct] = m

print("=" * 84)
print("MODULE INVENTORY — colour -> manuscript name")
print("=" * 84)
inventory = []
for ct in FILES:
    if ct not in W: continue
    s = W[ct].datExpr.var['moduleColors'].astype(str).value_counts()
    tot = int(s.sum()); mp = COLOR2IDX[ct]
    df = pd.DataFrame({"colour": s.index, "genes": s.values,
                       "pct": (100*s.values/tot).round(1)})
    df["module"] = [f"{ct}-{mp[c]}" if c in mp else f"{ct}-?" for c in df.colour]
    df["status"] = np.where(df.genes > GREY_FRACTION*tot, "EXCLUDED (unassigned bin)", "tested")
    df.insert(0, "cell_type", ct)
    inventory.append(df)
    print(f"\n[{ct}]  {len(s)} modules, {tot:,} genes")
    print(df[["module","colour","genes","pct","status"]].to_string(index=False))
inventory = pd.concat(inventory, ignore_index=True) if inventory else pd.DataFrame()

MANUSCRIPT = ["CA1-1","CA1-2","CA1-6","CA3-1","CA3-2","CA3-6","CA3-9","CA3-10","CA3-16",
              "DG-1","DG-12","Inhibitory-1","Inhibitory-6","Astro-1","Oligo-1","Oligo-12",
              "Microglia-2","Microglia-9"]
if len(inventory):
    bad = inventory[(inventory.status.str.startswith("EXCLUDED")) &
                    (inventory.module.isin(MANUSCRIPT))]
    print("\n" + "!" * 84)
    if len(bad):
        print("WARNING — these manuscript modules are unassigned bins, not real modules:")
        print(bad[["cell_type","module","colour","genes","pct"]].to_string(index=False))
    else:
        print("No manuscript module falls in an unassigned bin.")
    print("!" * 84)

print("\n\n" + "=" * 84)
print("MODULE PRESERVATION IN METACELL SPACE")
print("=" * 84)
all_res = []
for ct in FILES:
    if ct not in W: continue
    ref = W[ct].datExpr.to_df()
    colors = pd.Series(W[ct].datExpr.var['moduleColors'].astype(str).values,
                       index=W[ct].datExpr.var_names)
    mp = COLOR2IDX[ct]
    sizes = colors.value_counts()
    grey = set(sizes[sizes > GREY_FRACTION*len(colors)].index) | {"grey","gray"}
    drop = sorted(grey - {"grey","gray"})
    keep = colors[~colors.isin(grey)]
    if drop:
        print(f"\n[{ct}] excluding unassigned bin: " +
              ", ".join(f"{c} = {ct}-{mp.get(c,'?')} ({int(sizes[c]):,} genes)" for c in drop))
    if len(keep) == 0:
        print(f"[{ct}] every module exceeds {GREY_FRACTION:.0%} — no module structure to test.")
        print("       sizes: " + ", ".join(f"{ct}-{mp.get(c,'?')}={int(n):,}" for c, n in sizes.items()))
        continue
    genes = [g for g in keep.index if g in set(metacells.var_names)]
    if len(genes) == 0:
        print(f"[{ct}] no module genes present in the metacell object — skipping."); continue
    sub = metacells[metacells.obs[CELLTYPE_KEY] == ct, genes]
    if sub.n_obs < 20:
        print(f"[{ct}] only {sub.n_obs} metacells — skipping."); continue
    test = pd.DataFrame(sub.X.toarray() if sp.issparse(sub.X) else np.asarray(sub.X),
                        index=sub.obs_names, columns=genes)
    print(f"\n[{ct}] ref {ref.shape[0]:,} nuclei | test {test.shape[0]:,} metacells | "
          f"{len(genes):,} genes | {keep.loc[genes].nunique()} modules")
    r = module_preservation(ref[genes], test, keep.loc[genes],
                            beta=SOFT_POWER.get(ct, 2), n_perm=N_PERM, verbose=False)
    if len(r) == 0:
        print(f"[{ct}] no module reached the minimum gene count — skipping."); continue
    r.insert(0, "cell_type", ct)
    r.insert(1, "module_name", [f"{ct}-{mp[c]}" if c in mp else f"{ct}-?" for c in r.module])
    r = r.rename(columns={"module": "colour"})
    for _, x in r.iterrows():
        flag = f"   ({int(x.n_stat_dropped)}/8 stats degenerate)" if x.n_stat_dropped else ""
        print(f"    {x.module_name:<14} ({x.colour:<14}) n={int(x.moduleSize):<6} "
              f"Zsummary={x.Zsummary:7.2f}{flag}")
    all_res.append(r)
print(f"\n\nDone. {len(all_res)} cell type(s) -> `all_res`")

In [ ]:
# E
if all_res:
    pres = pd.concat(all_res, ignore_index=True)
    pres["verdict"] = pd.cut(pres.Zsummary, [-np.inf, 2, 10, np.inf],
                             labels=["not preserved","moderate","strong"])
    pres = pres.sort_values(["cell_type","Zsummary"], ascending=[True, False])
    show = ["cell_type","module_name","colour","moduleSize","Zdensity",
            "Zconnectivity","Zsummary","verdict"]

    print("\n" + "=" * 84)
    print("ALL MODULES")
    print("=" * 84)
    print(pres[show].round(2).to_string(index=False))
    print("\nOverall:"); print(pres.verdict.value_counts().to_string())
    print("\nBy cell type:")
    print(pres.groupby("cell_type").agg(
        modules=("Zsummary","size"), median_Z=("Zsummary","median"),
        strong=("verdict", lambda s: (s == "strong").sum()),
        failed=("verdict", lambda s: (s == "not preserved").sum())).round(2).to_string())

    print("\n" + "=" * 84)
    print("MODULES THE MANUSCRIPT DEPENDS ON")
    print("=" * 84)
    hit = pres[pres.module_name.isin(MANUSCRIPT)]
    print(hit[show].round(2).to_string(index=False) if len(hit) else "  none found")
    missing = sorted(set(MANUSCRIPT) - set(pres.module_name))
    if missing:
        print(f"\n  Not tested (excluded as unassigned, or below 10 genes): {', '.join(missing)}")

    fig, ax = plt.subplots(figsize=(8, 5.5))
    for ct, s in pres.groupby("cell_type"):
        ax.scatter(s.moduleSize, s.Zsummary, s=34, alpha=.8, label=ct)
    for _, x in pres[pres.module_name.isin(MANUSCRIPT)].iterrows():
        ax.annotate(x.module_name, (x.moduleSize, x.Zsummary), fontsize=7,
                    xytext=(4, 3), textcoords="offset points")
    ax.axhline(10, ls="--", c="k", lw=1); ax.axhline(2, ls=":", c="r", lw=1)
    ax.set_xscale("log"); ax.set_xlabel("Module size (genes)")
    ax.set_ylabel(r"$Z_{summary}$ preservation")
    ax.set_title("Nucleus-level modules evaluated in metacell space")
    ax.legend(fontsize=8, frameon=False, ncol=2); plt.tight_layout(); plt.show()
else:
    print("No results — check the .p files are in the working directory.")

### 15. Module eigengenes per animal
Computes module eigengenes on the metacell matrix and averages them within each animal.

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

SAMPLE_KEY, CELLTYPE_KEY = "sample_id_unique", "celltype"


def _eigengene(M):
    """First PC of samples x genes, sign-aligned to mean module expression."""
    M = np.asarray(M, float)
    sd = M.std(0); ok = sd > 0
    if ok.sum() < 3: return None
    Z = (M[:, ok] - M[:, ok].mean(0)) / sd[ok]
    u, _, _ = np.linalg.svd(Z, full_matrices=False)
    me = u[:, 0]
    if np.std(me) == 0: return None
    if np.corrcoef(me, Z.mean(1))[0, 1] < 0: me = -me
    return (me - me.mean()) / me.std()


def mouse_level_eigengenes(mc, color_map, module_defs, min_obs=10, min_genes=3):
    """Eigengenes on METACELLS, then averaged within each mouse."""
    rows = []
    for ct, colors in module_defs.items():
        sub = mc[mc.obs[CELLTYPE_KEY] == ct]
        if sub.n_obs < min_obs:
            print(f"  [{ct}] {sub.n_obs} metacells — skipped"); continue
        X = pd.DataFrame(sub.X.toarray() if sp.issparse(sub.X) else np.asarray(sub.X),
                         index=sub.obs_names, columns=sub.var_names)
        for colour, genes in colors.groupby(colors):
            g = [x for x in genes.index if x in X.columns]
            if len(g) < min_genes: continue
            me = _eigengene(X[g].values)
            if me is None: continue
            name = f"{ct}-{color_map[ct][colour]}" if colour in color_map.get(ct, {}) else f"{ct}-{colour}"
            d = pd.DataFrame({"ME": me, "mouse": sub.obs[SAMPLE_KEY].values,
                              "group": sub.obs["group"].values,
                              "condition": sub.obs["condition"].values,
                              "n_cells": sub.obs["n_cells"].values})
            agg = (d.groupby(["mouse", "group", "condition"], observed=True)
                     .agg(ME=("ME", "mean"), n_metacells=("ME", "size"),
                          n_nuclei=("n_cells", "sum")).reset_index())
            agg["cell_type"] = ct; agg["module"] = name; agg["colour"] = colour
            agg["n_genes"] = len(g)
            rows.append(agg)
    out = pd.concat(rows, ignore_index=True)
    out["group"] = out["group"].astype(str); out["condition"] = out["condition"].astype(str)
    return out


# module definitions come from the saved networks (W, COLOR2IDX from Cell D)
module_defs = {ct: pd.Series(W[ct].datExpr.var['moduleColors'].astype(str).values,
                             index=W[ct].datExpr.var_names) for ct in W}

me_long = mouse_level_eigengenes(metacells, COLOR2IDX, module_defs)

print(f"{me_long.module.nunique()} modules x {me_long.mouse.nunique()} mice "
      f"= {len(me_long):,} eigengene values")
print("\nMice per genotype x condition:")
print(me_long.drop_duplicates("mouse").groupby(["group", "condition"]).size().to_string())
print("\nMetacells contributing per mouse (CA1 shown):")
print(me_long[me_long.cell_type == "CA1"].drop_duplicates("mouse")
        [["mouse", "group", "condition", "n_metacells", "n_nuclei"]].to_string(index=False))

# carry the Cell E verdict through, if it exists
if "pres" in dir():
    pres_map = dict(zip(pres.module_name, pres.verdict))
    z_map    = dict(zip(pres.module_name, pres.Zsummary))
    me_long["preservation"] = me_long.module.map(pres_map).fillna("not tested")
    me_long["Zsummary"]     = me_long.module.map(z_map)
    print("\nPreservation status of modules entering the eigengene model:")
    print(me_long.drop_duplicates("module").preservation.value_counts().to_string())
else:
    pres_map = None
    me_long["preservation"] = "not tested"
    print("\n  (Cell E result `pres` not found — preservation status unavailable)")

### 16. Mouse-level linear models on module eigengenes
Fits ME ~ genotype + treatment + genotype:treatment for the vehicle and KA3 conditions (n = 12 mice), with Benjamini-Hochberg correction across modules within each cell type and term. Effect sizes are reported with 95% confidence intervals.

In [ ]:
def fit_mouse_models(me_long, conditions=("Veh", "KA3"), ref_group="WT", ref_cond="Veh",
                     preserved_only=None):
    """ME ~ genotype * treatment on mouse-level eigengenes. One model per module."""
    d = me_long[me_long.condition.isin(conditions)].copy()
    if preserved_only is not None:
        keep = set(preserved_only)
        n0 = d.module.nunique(); d = d[d.module.isin(keep)]
        print(f"  restricted to {d.module.nunique()} preserved modules (of {n0})")
    res = []
    for (ct, mod), s in d.groupby(["cell_type", "module"], observed=True):
        n = s.mouse.nunique()
        cells = s.groupby(["group", "condition"], observed=True).size()
        if len(cells) < len(conditions) * 2 or cells.min() < 2:
            res.append(dict(cell_type=ct, module=mod, n_mice=n, term="(not estimable)",
                            estimate=np.nan, se=np.nan, t=np.nan, p=np.nan,
                            df_resid=np.nan, ci_lo=np.nan, ci_hi=np.nan)); continue
        m = smf.ols(f"ME ~ C(group, Treatment('{ref_group}')) "
                    f"* C(condition, Treatment('{ref_cond}'))", data=s).fit()
        ci = m.conf_int()
        for term in m.params.index:
            if term == "Intercept": continue
            lab = ("genotype (KO-WT)" if term.startswith("C(group") and ":" not in term else
                   "treatment (KA-Veh)" if term.startswith("C(condition") else
                   "genotype x treatment")
            res.append(dict(cell_type=ct, module=mod, n_mice=n, term=lab,
                            estimate=m.params[term], se=m.bse[term], t=m.tvalues[term],
                            p=m.pvalues[term], df_resid=int(m.df_resid),
                            ci_lo=ci.loc[term, 0], ci_hi=ci.loc[term, 1]))
    r = pd.DataFrame(res)
    r["p_fdr"] = np.nan
    for _, idx in r.dropna(subset=["p"]).groupby(["cell_type", "term"]).groups.items():
        r.loc[idx, "p_fdr"] = multipletests(r.loc[idx, "p"], method="fdr_bh")[1]
    r["sig"] = np.where(r.p_fdr < .05, "*", "")
    return r.sort_values(["cell_type", "term", "p"]).reset_index(drop=True)


# only modules that survived preservation carry interpretation
preserved = (pres.loc[pres.Zsummary >= 2, "module_name"].tolist()
             if "pres" in dir() else None)

print("=" * 96)
print("PRIMARY — ME ~ genotype * treatment   (Veh + KA3, n = 12 mice, metacell eigengenes)")
print("=" * 96)
lm_res = fit_mouse_models(me_long, conditions=("Veh", "KA3"), preserved_only=preserved)
print(lm_res[["cell_type","module","term","estimate","ci_lo","ci_hi",
              "p","p_fdr","sig","n_mice","df_resid"]].round(4).to_string(index=False))

hits = lm_res[lm_res.p_fdr < 0.05]
print("\nFDR < 0.05:")
print(hits[["cell_type","module","term","estimate","ci_lo","ci_hi","p_fdr"]]
      .round(4).to_string(index=False) if len(hits) else "  none")

print("\n" + "=" * 96)
print("SECONDARY — Veh + KA25  (descriptive; WT KA25 group is underpowered)")
print("=" * 96)
lm_ka25 = fit_mouse_models(me_long, conditions=("Veh", "KA25"), preserved_only=preserved)
print(lm_ka25[["cell_type","module","term","estimate","p","p_fdr","n_mice"]]
      .round(4).to_string(index=False))

In [ ]:
def plot_mouse_level(me_long, lm_res, modules=None, title_suffix="", pres_map=None):
    d = me_long.copy()
    if modules: d = d[d.module.isin(modules)]
    if d.empty: print("  nothing to plot"); return
    d["trait"] = d.group + "_" + d.condition
    order = [t for t in ['WT_Veh','KO_Veh','WT_KA3','KO_KA3','WT_KA25','KO_KA25'] if t in set(d.trait)]
    mean  = d.pivot_table(index="trait", columns="module", values="ME", aggfunc="mean").reindex(order)
    nmice = d.pivot_table(index="trait", columns="module", values="mouse", aggfunc="nunique").reindex(order)
    lab = mean.round(2).astype(str) + "\n(n=" + nmice.astype("Int64").astype(str) + ")"
    cols = list(mean.columns)
    xlab = [f"{c}\n{pres_map.get(c,'not tested')}" for c in cols] if pres_map else cols

    fig, axes = plt.subplots(1, 2, figsize=(max(14, len(cols)*2.7), 6.2),
                             gridspec_kw={"width_ratios":[2,1]}, facecolor="white")
    v = np.nanmax(np.abs(mean.values))
    sns.heatmap(mean, annot=lab, fmt="", cmap="RdBu_r", center=0, vmin=-v, vmax=v,
                annot_kws={"size":11,"weight":"bold"}, linewidths=.5, linecolor="gray",
                ax=axes[0], cbar_kws={"label":"mean eigengene (metacell -> mouse)"})
    axes[0].set_title(f"Mouse-level module eigengene{title_suffix}", fontsize=14, fontweight="bold")
    axes[0].set_xlabel(""); axes[0].set_ylabel("")
    axes[0].set_xticklabels(xlab, rotation=45, ha="right", fontweight="bold", fontsize=9)
    axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontweight="bold")

    L = lm_res[lm_res.module.isin(cols) & lm_res.term.ne("(not estimable)")]
    if L.empty:
        axes[1].axis("off"); axes[1].set_title("no estimable models")
    else:
        est = L.pivot_table(index="term", columns="module", values="estimate")
        pfd = L.pivot_table(index="term", columns="module", values="p_fdr")
        trow = [t for t in ["genotype (KO-WT)","treatment (KA-Veh)","genotype x treatment"] if t in est.index]
        est, pfd = est.reindex(trow), pfd.reindex(trow)
        lab2 = est.round(2).astype(str) + np.where(pfd<.001,"\n***",
               np.where(pfd<.01,"\n**",np.where(pfd<.05,"\n*","\nns")))
        v2 = np.nanmax(np.abs(est.values))
        sns.heatmap(est, annot=lab2, fmt="", cmap="PuOr_r", center=0, vmin=-v2, vmax=v2,
                    annot_kws={"size":11,"weight":"bold"}, linewidths=.5, linecolor="gray",
                    ax=axes[1], cbar_kws={"label":"LM coefficient"})
        axes[1].set_title("ME ~ genotype * treatment\n(mouse level, FDR within cell type)",
                          fontsize=12, fontweight="bold")
        axes[1].set_xlabel(""); axes[1].set_ylabel("")
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right", fontweight="bold")
        axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, fontweight="bold")
    plt.tight_layout(); plt.show()


dic_cell_modules = {
    'DG':        {'genotype': ['dimgrey','lightgrey'], 'drugeffect': ['silver','darkgrey','gainsboro'], 'interactive': ['dimgrey']},
    'CA1':       {'genotype': ['dimgrey','darkgrey'],  'drugeffect': ['silver'],                        'interactive': ['dimgrey']},
    'CA3':       {'genotype': ['dimgrey','indianred'], 'drugeffect': ['silver','darkgrey'],             'interactive': ['dimgrey']},
    'Inhibitory':{'genotype': ['dimgrey','lightcoral'],'drugeffect': [],                                'interactive': ['dimgrey','lightcoral']},
}

for effect in ['genotype', 'drugeffect', 'interactive']:
    want = [f"{ct}-{COLOR2IDX[ct][c]}" for ct, e in dic_cell_modules.items()
            for c in e[effect] if ct in COLOR2IDX and c in COLOR2IDX[ct]]
    want = [m for m in want if m in set(me_long.module)]
    if not want:
        print(f"\nNo modules available for '{effect}'."); continue
    print(f"\n{'='*96}\n{effect.upper()}: {', '.join(want)}\n{'='*96}")
    if pres_map:
        st = pd.Series({m: pres_map.get(m, "not tested") for m in want})
        print("preservation: " + " | ".join(f"{m}={v}" for m, v in st.items()))
    plot_mouse_level(me_long, lm_res, modules=want, title_suffix=f" — {effect}", pres_map=pres_map)
    sub = lm_res[lm_res.module.isin(want)]
    print(sub[["cell_type","module","term","estimate","ci_lo","ci_hi","p","p_fdr","sig"]]
          .round(4).to_string(index=False) if len(sub) else "  (all excluded by the preservation filter)")

In [ ]:
#F
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

SAMPLE_KEY, CELLTYPE_KEY = "sample_id_unique", "celltype"


def _eigengene(M):
    """First PC of samples x genes, sign-aligned to mean module expression."""
    M = np.asarray(M, float)
    sd = M.std(0); ok = sd > 0
    if ok.sum() < 3: return None
    Z = (M[:, ok] - M[:, ok].mean(0)) / sd[ok]
    u, s, _ = np.linalg.svd(Z, full_matrices=False)
    me = u[:, 0]
    if np.std(me) == 0: return None
    if np.corrcoef(me, Z.mean(1))[0, 1] < 0: me = -me
    return (me - me.mean()) / me.std()


def mouse_level_eigengenes(adata, color_map, module_defs, celltype_key=CELLTYPE_KEY):
    """
    adata      : metacell AnnData from Cell A
    color_map  : COLOR2IDX from Cell D
    module_defs: {celltype: Series(gene -> colour)}
    Returns one row per mouse x module.
    """
    rows = []
    for ct, colors in module_defs.items():
        sub = adata[adata.obs[celltype_key] == ct]
        if sub.n_obs < 10:
            print(f"  [{ct}] only {sub.n_obs} observations — skipped"); continue
        X = pd.DataFrame(sub.X.toarray() if sp.issparse(sub.X) else np.asarray(sub.X),
                         index=sub.obs_names, columns=sub.var_names)
        for colour, genes in colors.groupby(colors):
            g = [x for x in genes.index if x in X.columns]
            if len(g) < 3: continue
            me = _eigengene(X[g].values)
            if me is None: continue
            name = f"{ct}-{color_map[ct][colour]}" if colour in color_map.get(ct, {}) else f"{ct}-{colour}"
            d = pd.DataFrame({"ME": me, "mouse": sub.obs[SAMPLE_KEY].values,
                              "group": sub.obs["group"].values,
                              "condition": sub.obs["condition"].values})
            agg = d.groupby(["mouse", "group", "condition"], observed=True)["ME"].mean().reset_index()
            agg["cell_type"] = ct; agg["module"] = name; agg["colour"] = colour
            rows.append(agg)
    out = pd.concat(rows, ignore_index=True)
    out["group"] = out["group"].astype(str); out["condition"] = out["condition"].astype(str)
    return out


# module definitions straight from the saved networks (W and COLOR2IDX come from Cell D)
module_defs = {ct: pd.Series(W[ct].datExpr.var['moduleColors'].astype(str).values,
                             index=W[ct].datExpr.var_names) for ct in W}

me_long = mouse_level_eigengenes(metacells, COLOR2IDX, module_defs)
print(f"{me_long.module.nunique()} modules x {me_long.mouse.nunique()} mice "
      f"= {len(me_long):,} eigengene values")
print("\nMice per genotype x condition:")
print(me_long.drop_duplicates("mouse").groupby(["group", "condition"]).size().to_string())

In [ ]:
#G
def fit_mouse_models(me_long, conditions=("Veh", "KA3"), ref_group="WT", ref_cond="Veh"):
    """ME ~ genotype * treatment on mouse-level eigengenes. One model per module."""
    d = me_long[me_long.condition.isin(conditions)].copy()
    res = []
    for (ct, mod), s in d.groupby(["cell_type", "module"], observed=True):
        n = s.mouse.nunique()
        cells = s.groupby(["group", "condition"], observed=True).size()
        if len(cells) < len(conditions) * 2 or cells.min() < 2:
            res.append(dict(cell_type=ct, module=mod, n_mice=n, term="(not estimable)",
                            estimate=np.nan, se=np.nan, t=np.nan, p=np.nan)); continue
        f = f"ME ~ C(group, Treatment('{ref_group}')) * C(condition, Treatment('{ref_cond}'))"
        m = smf.ols(f, data=s).fit()
        for term in m.params.index:
            if term == "Intercept": continue
            lab = ("genotype (KO-WT)" if term.startswith("C(group") and ":" not in term else
                   "treatment (KA-Veh)" if term.startswith("C(condition") else
                   "genotype x treatment")
            res.append(dict(cell_type=ct, module=mod, n_mice=n, term=lab,
                            estimate=m.params[term], se=m.bse[term],
                            t=m.tvalues[term], p=m.pvalues[term], df_resid=int(m.df_resid)))
    r = pd.DataFrame(res)
    r["p_fdr"] = np.nan
    for (ct, term), idx in r.dropna(subset=["p"]).groupby(["cell_type", "term"]).groups.items():
        r.loc[idx, "p_fdr"] = multipletests(r.loc[idx, "p"], method="fdr_bh")[1]
    r["sig"] = np.where(r.p_fdr < 0.05, "*", "")
    return r.sort_values(["cell_type", "term", "p"]).reset_index(drop=True)


# PRIMARY: Veh + KA3, balanced 2x2, n = 12
lm_res = fit_mouse_models(me_long, conditions=("Veh", "KA3"))
print("=" * 92)
print("PRIMARY MODEL — ME ~ genotype * treatment   (Veh + KA3, n = 12 mice)")
print("=" * 92)
print(lm_res[["cell_type","module","term","estimate","se","t","p","p_fdr","sig","df_resid"]]
      .round(4).to_string(index=False))

print("\nSignificant at FDR < 0.05:")
hits = lm_res[lm_res.p_fdr < 0.05]
print(hits[["cell_type","module","term","estimate","p_fdr"]].round(4).to_string(index=False)
      if len(hits) else "  none")

# SECONDARY: include KA25 descriptively (n=1 WT means the interaction is not estimable there)
lm_ka25 = fit_mouse_models(me_long, conditions=("Veh", "KA25"))
print("\n" + "=" * 92)
print("SECONDARY — Veh + KA25   (report descriptively; WT KA25 group is underpowered)")
print("=" * 92)
print(lm_ka25[["cell_type","module","term","estimate","p","p_fdr","n_mice"]]
      .round(4).to_string(index=False))

In [ ]:
#H
def plot_mouse_level(me_long, lm_res, celltypes=None, modules=None, title_suffix=""):
    """Replacement for the module-trait correlation heatmap, at mouse level."""
    d = me_long.copy()
    if celltypes: d = d[d.cell_type.isin(celltypes)]
    if modules:   d = d[d.module.isin(modules)]
    if d.empty: print("nothing to plot"); return

    d["trait"] = d.group.astype(str) + "_" + d.condition.astype(str)
    order = [t for t in ['WT_Veh','KO_Veh','WT_KA3','KO_KA3','WT_KA25','KO_KA25']
             if t in set(d.trait)]
    mean = d.pivot_table(index="trait", columns="module", values="ME", aggfunc="mean").reindex(order)
    nmice = d.pivot_table(index="trait", columns="module", values="mouse",
                          aggfunc="nunique").reindex(order)
    lab = mean.round(2).astype(str) + "\n(n=" + nmice.astype("Int64").astype(str) + ")"

    fig, axes = plt.subplots(1, 2, figsize=(max(14, mean.shape[1]*2.6), 6),
                             gridspec_kw={"width_ratios": [2, 1]}, facecolor="white")
    v = np.nanmax(np.abs(mean.values))
    sns.heatmap(mean, annot=lab, fmt="", cmap="RdBu_r", center=0, vmin=-v, vmax=v,
                annot_kws={"size": 11, "weight": "bold"}, linewidths=.5,
                linecolor="gray", ax=axes[0], cbar_kws={"label": "mean eigengene"})
    axes[0].set_title(f"Mouse-level module eigengene by group{title_suffix}",
                      fontsize=14, fontweight="bold")
    axes[0].set_xlabel(""); axes[0].set_ylabel("")
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right", fontweight="bold")
    axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontweight="bold")

    L = lm_res[lm_res.module.isin(mean.columns) & lm_res.term.ne("(not estimable)")]
    est = L.pivot_table(index="term", columns="module", values="estimate")
    pfd = L.pivot_table(index="term", columns="module", values="p_fdr")
    trow = [t for t in ["genotype (KO-WT)", "treatment (KA-Veh)", "genotype x treatment"]
            if t in est.index]
    est, pfd = est.reindex(trow), pfd.reindex(trow)
    lab2 = est.round(2).astype(str) + np.where(pfd < .001, "\n***",
           np.where(pfd < .01, "\n**", np.where(pfd < .05, "\n*", "\nns")))
    v2 = np.nanmax(np.abs(est.values))
    sns.heatmap(est, annot=lab2, fmt="", cmap="PuOr_r", center=0, vmin=-v2, vmax=v2,
                annot_kws={"size": 11, "weight": "bold"}, linewidths=.5,
                linecolor="gray", ax=axes[1], cbar_kws={"label": "LM coefficient"})
    axes[1].set_title("ME ~ genotype * treatment\n(FDR within cell type)",
                      fontsize=13, fontweight="bold")
    axes[1].set_xlabel(""); axes[1].set_ylabel("")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right", fontweight="bold")
    axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, fontweight="bold")
    plt.tight_layout(); plt.show()


# your Fig-3 modules, by colour, mapped through COLOR2IDX
dic_cell_modules = {
    'DG':        {'genotype': ['dimgrey','lightgrey'], 'drugeffect': ['silver','darkgrey','gainsboro'], 'interactive': ['dimgrey']},
    'CA1':       {'genotype': ['dimgrey','darkgrey'],  'drugeffect': ['silver'],                        'interactive': ['dimgrey']},
    'CA3':       {'genotype': ['dimgrey','indianred'], 'drugeffect': ['silver','darkgrey'],             'interactive': ['dimgrey']},
    'Inhibitory':{'genotype': ['dimgrey','lightcoral'],'drugeffect': [],                                'interactive': ['dimgrey','lightcoral']},
}

for effect in ['genotype', 'drugeffect', 'interactive']:
    want = [f"{ct}-{COLOR2IDX[ct][c]}" for ct, e in dic_cell_modules.items()
            for c in e[effect] if ct in COLOR2IDX and c in COLOR2IDX[ct]]
    if not want:
        print(f"\nNo modules for '{effect}'."); continue
    print(f"\n{'='*92}\n{effect.upper()} MODULES: {', '.join(want)}\n{'='*92}")
    plot_mouse_level(me_long, lm_res, modules=want, title_suffix=f" — {effect}")
    print(lm_res[lm_res.module.isin(want)]
          [["cell_type","module","term","estimate","p","p_fdr","sig"]]
          .round(4).to_string(index=False))

### 17. Module preservation analysis
Re-evaluates the fixed nucleus-level module assignments in the metacell representation, scoring four density and four connectivity statistics against a permutation null and combining them into Zsummary (Supplementary Table 4).

In [ ]:
# =============================================================================
# F — METACELL-SUPPORTED, MOUSE-LEVEL MODULE EIGENGENES
# =============================================================================
#
# PURPOSE
# -------
# Use metacells to obtain a less sparse / less noisy estimate of each module's
# expression axis, but perform ALL statistical inference at the mouse level.
#
# Workflow:
#
#   nuclei
#       -> metacells created separately within mouse x cell type
#       -> one common PC1 axis per cell type x module
#       -> project every metacell onto that SAME axis
#       -> aggregate metacell scores to one value per mouse
#       -> mouse-level linear model:
#
#          ME ~ genotype * treatment
#
# IMPORTANT:
# ----------
# Metacells are NOT treated as independent biological replicates.
# The final regression contains one row per mouse x module.
#
# Requires objects already created above:
#
#   metacells
#   W
#   COLOR2IDX
#
# =============================================================================

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


SAMPLE_KEY = "sample_id_unique"
CELLTYPE_KEY = "celltype"

GROUP_KEY = "group"
CONDITION_KEY = "condition"

METACELL_SIZE_KEY = "n_cells"


# =============================================================================
# BUILD MODULE DEFINITIONS FROM THE ORIGINAL WGCNA NETWORKS
# =============================================================================

module_defs = {
    ct: pd.Series(
        W[ct].datExpr.var[
            "moduleColors"
        ].astype(str).values,
        index=W[ct].datExpr.var_names.astype(str)
    )
    for ct in W
}


# =============================================================================
# HELPER: DENSE MATRIX
# =============================================================================

def _dense(X):
    """
    Convert sparse/dense matrix to NumPy array.
    """

    if sp.issparse(X):
        return X.toarray()

    return np.asarray(
        X
    )


# =============================================================================
# HELPER: FIT ONE COMMON MODULE AXIS
# =============================================================================

def fit_common_module_axis(
    X
):
    """
    Fit a common module eigengene axis across all available metacells.

    Parameters
    ----------
    X : array-like, shape metacells x module genes

    Returns
    -------
    Dictionary containing:

        gene_mean
        gene_sd
        loadings
        score_mean
        score_sd

    The PC1 sign is aligned with the average standardized expression
    of the module genes so that larger ME values generally correspond
    to larger coordinated expression of the module.

    IMPORTANT:
    The same loadings are subsequently used for every mouse.
    """

    X = np.asarray(
        X,
        dtype=np.float64
    )

    # -------------------------------------------------------------------------
    # Remove genes with zero variance
    # -------------------------------------------------------------------------

    gene_mean = np.nanmean(
        X,
        axis=0
    )

    gene_sd = np.nanstd(
        X,
        axis=0,
        ddof=0
    )

    keep = (
        np.isfinite(
            gene_sd
        )
        & (
            gene_sd > 0
        )
    )

    if keep.sum() < 3:

        return None

    X_use = X[
        :,
        keep
    ]

    mean_use = gene_mean[
        keep
    ]

    sd_use = gene_sd[
        keep
    ]

    # -------------------------------------------------------------------------
    # Standardize genes across metacells
    # -------------------------------------------------------------------------

    Z = (
        X_use
        - mean_use
    ) / sd_use

    if not np.isfinite(
        Z
    ).all():

        return None

    # -------------------------------------------------------------------------
    # PCA through SVD
    #
    # Z = U S V'
    #
    # columns of V are gene loading directions.
    # -------------------------------------------------------------------------

    U, S, VT = np.linalg.svd(
        Z,
        full_matrices=False
    )

    if len(S) == 0:

        return None

    loadings = VT[
        0,
        :
    ].copy()

    # Scores using the actual common loading vector
    scores = (
        Z
        @ loadings
    )

    if np.std(
        scores
    ) == 0:

        return None

    # -------------------------------------------------------------------------
    # Resolve arbitrary PCA sign.
    #
    # Make PC1 positively correlated with mean standardized
    # module expression.
    # -------------------------------------------------------------------------

    mean_module_expression = (
        Z.mean(
            axis=1
        )
    )

    if (
        np.std(
            mean_module_expression
        )
        > 0
    ):

        corr = np.corrcoef(
            scores,
            mean_module_expression
        )[0, 1]

        if (
            np.isfinite(
                corr
            )
            and corr < 0
        ):

            loadings = (
                -loadings
            )

            scores = (
                -scores
            )

    # -------------------------------------------------------------------------
    # Standardize the module score itself
    # -------------------------------------------------------------------------

    score_mean = float(
        scores.mean()
    )

    score_sd = float(
        scores.std(
            ddof=0
        )
    )

    if (
        not np.isfinite(
            score_sd
        )
        or score_sd == 0
    ):

        return None

    # -------------------------------------------------------------------------
    # Fraction of variance explained by PC1
    # -------------------------------------------------------------------------

    total_variance = np.sum(
        S ** 2
    )

    pc1_variance = (
        float(
            S[0] ** 2
            / total_variance
        )
        if total_variance > 0
        else np.nan
    )

    return {
        "keep": keep,
        "gene_mean": mean_use,
        "gene_sd": sd_use,
        "loadings": loadings,
        "score_mean": score_mean,
        "score_sd": score_sd,
        "pc1_variance_explained": pc1_variance
    }


# =============================================================================
# HELPER: PROJECT ONTO THE COMMON MODULE AXIS
# =============================================================================

def project_common_module_axis(
    X,
    axis
):
    """
    Project observations onto a previously fitted module PC1 axis.

    This guarantees that every mouse is represented in exactly the
    same module coordinate system.
    """

    X = np.asarray(
        X,
        dtype=np.float64
    )

    X = X[
        :,
        axis["keep"]
    ]

    Z = (
        X
        - axis["gene_mean"]
    ) / axis["gene_sd"]

    scores = (
        Z
        @ axis["loadings"]
    )

    scores = (
        scores
        - axis["score_mean"]
    ) / axis["score_sd"]

    return scores


# =============================================================================
# METACELL -> MOUSE MODULE EIGENGES
# =============================================================================

def metacell_to_mouse_eigengenes(
    metacells,
    color_map,
    module_defs,
    min_module_genes=3,
    min_metacells=10,
    weight_by_nuclei=True
):
    """
    Construct common module eigengene axes in metacell space,
    then return ONE module eigengene value per mouse.

    Parameters
    ----------
    metacells
        AnnData containing metacells.

        IMPORTANT:
        These metacells must have been generated separately within
        mouse x cell type, as in your build_metacells() function.

    color_map
        COLOR2IDX mapping manuscript module colours to module numbers.

    module_defs
        Dictionary:
            cell type -> Series(gene -> WGCNA module colour)

    min_module_genes
        Minimum number of available genes required to score a module.

    min_metacells
        Minimum number of metacells required to fit a common PC1 axis.

    weight_by_nuclei
        If True, mouse ME is calculated as a nucleus-count-weighted
        average of its metacell scores.

        This is useful because your KMeans metacells can have slightly
        different numbers of constituent nuclei.

    Returns
    -------
    me_long
        One row per:

            mouse x cell type x module

    module_axis_qc
        Information about PC1 and the number of genes/metacells used.
    """

    rows = []
    qc_rows = []

    # -------------------------------------------------------------------------
    # Loop over cell types
    # -------------------------------------------------------------------------

    for ct, colors in module_defs.items():

        if ct not in set(
            metacells.obs[
                CELLTYPE_KEY
            ].astype(str)
        ):

            print(
                f"[{ct}] no metacells found — skipped"
            )

            continue

        sub = metacells[
            metacells.obs[
                CELLTYPE_KEY
            ].astype(str)
            == str(ct)
        ].copy()

        if (
            sub.n_obs
            < min_metacells
        ):

            print(
                f"[{ct}] only "
                f"{sub.n_obs} metacells — skipped"
            )

            continue

        # ---------------------------------------------------------------------
        # Dense expression matrix once per cell type
        # ---------------------------------------------------------------------

        X_all = _dense(
            sub.X
        ).astype(
            np.float64
        )

        gene_to_index = {
            str(g): i
            for i, g
            in enumerate(
                sub.var_names
            )
        }

        print(
            "\n"
            + "=" * 90
        )

        print(
            f"[{ct}] "
            f"{sub.n_obs} metacells, "
            f"{sub.obs[SAMPLE_KEY].nunique()} mice"
        )

        print(
            "=" * 90
        )

        # ---------------------------------------------------------------------
        # Loop through original WGCNA modules
        # ---------------------------------------------------------------------

        for colour, module_gene_series in colors.groupby(
            colors
        ):

            module_genes = [
                str(g)
                for g
                in module_gene_series.index
                if str(g)
                in gene_to_index
            ]

            if (
                len(
                    module_genes
                )
                < min_module_genes
            ):

                continue

            gene_idx = np.asarray(
                [
                    gene_to_index[g]
                    for g in module_genes
                ],
                dtype=int
            )

            X_module = X_all[
                :,
                gene_idx
            ]

            # -----------------------------------------------------------------
            # Fit ONE common PC1/loading axis across all metacells
            # -----------------------------------------------------------------

            axis = fit_common_module_axis(
                X_module
            )

            if axis is None:

                continue

            # -----------------------------------------------------------------
            # Score every metacell on exactly the same PC1 axis
            # -----------------------------------------------------------------

            metacell_scores = (
                project_common_module_axis(
                    X_module,
                    axis
                )
            )

            # -----------------------------------------------------------------
            # Manuscript module name
            # -----------------------------------------------------------------

            if (
                colour
                in color_map.get(
                    ct,
                    {}
                )
            ):

                module_name = (
                    f"{ct}-"
                    f"{color_map[ct][colour]}"
                )

            else:

                module_name = (
                    f"{ct}-{colour}"
                )

            # -----------------------------------------------------------------
            # Metacell-level temporary dataframe
            #
            # DO NOT use this dataframe for inference.
            # -----------------------------------------------------------------

            temp = pd.DataFrame({
                "mouse":
                    sub.obs[
                        SAMPLE_KEY
                    ].astype(str).values,

                "group":
                    sub.obs[
                        GROUP_KEY
                    ].astype(str).values,

                "condition":
                    sub.obs[
                        CONDITION_KEY
                    ].astype(str).values,

                "metacell_ME":
                    metacell_scores
            })

            if (
                METACELL_SIZE_KEY
                in sub.obs.columns
            ):

                temp[
                    "n_nuclei"
                ] = (
                    sub.obs[
                        METACELL_SIZE_KEY
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )

            else:

                temp[
                    "n_nuclei"
                ] = 1.0

            # -----------------------------------------------------------------
            # Aggregate metacells -> ONE value per mouse
            # -----------------------------------------------------------------

            mouse_rows = []

            for (
                mouse,
                group,
                condition
            ), m in temp.groupby(
                [
                    "mouse",
                    "group",
                    "condition"
                ],
                observed=True
            ):

                if (
                    weight_by_nuclei
                    and
                    m[
                        "n_nuclei"
                    ].sum() > 0
                ):

                    mouse_me = np.average(
                        m[
                            "metacell_ME"
                        ].to_numpy(
                            dtype=float
                        ),
                        weights=m[
                            "n_nuclei"
                        ].to_numpy(
                            dtype=float
                        )
                    )

                else:

                    mouse_me = (
                        m[
                            "metacell_ME"
                        ].mean()
                    )

                mouse_rows.append({
                    "mouse":
                        str(mouse),

                    "group":
                        str(group),

                    "condition":
                        str(condition),

                    "ME":
                        float(
                            mouse_me
                        ),

                    "n_metacells":
                        int(
                            len(m)
                        ),

                    "n_nuclei":
                        int(
                            m[
                                "n_nuclei"
                            ].sum()
                        ),

                    "cell_type":
                        ct,

                    "module":
                        module_name,

                    "colour":
                        str(colour)
                })

            rows.extend(
                mouse_rows
            )

            qc_rows.append({
                "cell_type":
                    ct,

                "module":
                    module_name,

                "colour":
                    str(colour),

                "n_module_genes":
                    len(
                        module_genes
                    ),

                "n_variable_genes":
                    int(
                        axis[
                            "keep"
                        ].sum()
                    ),

                "n_metacells":
                    int(
                        sub.n_obs
                    ),

                "n_mice":
                    int(
                        sub.obs[
                            SAMPLE_KEY
                        ].nunique()
                    ),

                "PC1_variance_explained":
                    axis[
                        "pc1_variance_explained"
                    ]
            })

    # -------------------------------------------------------------------------
    # Combine
    # -------------------------------------------------------------------------

    if len(rows) == 0:

        raise RuntimeError(
            "No mouse-level eigengenes could be calculated."
        )

    me_long = pd.DataFrame(
        rows
    )

    axis_qc = pd.DataFrame(
        qc_rows
    )

    # -------------------------------------------------------------------------
    # Sanity check:
    # only one value per mouse x module
    # -------------------------------------------------------------------------

    duplicated = (
        me_long.duplicated(
            [
                "mouse",
                "cell_type",
                "module"
            ]
        )
    )

    if duplicated.any():

        raise RuntimeError(
            "Duplicate mouse x module rows detected."
        )

    return (
        me_long,
        axis_qc
    )


# =============================================================================
# RUN METACELL-SUPPORTED MOUSE-LEVEL SCORING
# =============================================================================

(
    me_long,
    module_axis_qc
) = metacell_to_mouse_eigengenes(
    metacells=metacells,
    color_map=COLOR2IDX,
    module_defs=module_defs,
    min_module_genes=3,
    min_metacells=10,
    weight_by_nuclei=True
)


print(
    "\n"
    + "=" * 100
)

print(
    "METACELL-SUPPORTED MOUSE-LEVEL MODULE EIGENGENES"
)

print(
    "=" * 100
)


print(
    f"\nModules: "
    f"{me_long['module'].nunique()}"
)

print(
    f"Mice: "
    f"{me_long['mouse'].nunique()}"
)

print(
    f"Total mouse x module observations: "
    f"{len(me_long):,}"
)


print(
    "\nMice per genotype x condition:"
)

print(
    me_long[
        [
            "mouse",
            "group",
            "condition"
        ]
    ]
    .drop_duplicates()
    .groupby(
        [
            "group",
            "condition"
        ],
        observed=True
    )
    .size()
    .to_string()
)


print(
    "\nExample mouse-level eigengene values:"
)

print(
    me_long[
        [
            "mouse",
            "group",
            "condition",
            "cell_type",
            "module",
            "ME",
            "n_metacells",
            "n_nuclei"
        ]
    ]
    .head(
        20
    )
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print(
    "\nModule-axis QC:"
)

print(
    module_axis_qc[
        [
            "cell_type",
            "module",
            "colour",
            "n_module_genes",
            "n_variable_genes",
            "n_metacells",
            "n_mice",
            "PC1_variance_explained"
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# G — MOUSE-LEVEL LINEAR MODELS
# =============================================================================

def fit_mouse_models(
    me_long,
    conditions=("Veh", "KA3"),
    ref_group="WT",
    ref_cond="Veh",
    min_mice_per_cell=2
):
    """
    Fit:

        ME ~ genotype * treatment

    separately for every cell type x module.

    IMPORTANT:
    The rows entering the regression are mice, NOT metacells.

    Reference levels:
        genotype  = WT
        condition = Veh

    Therefore for Veh + KA3:

      genotype (KO-WT)
          = KO vs WT difference specifically under Veh.

      treatment (KA3-Veh)
          = KA3 vs Veh difference specifically in WT.

      genotype x treatment
          = difference in the KA3 effect between KO and WT.

    FDR:
        BH adjustment is performed within each
        cell type x model term.
    """

    d = me_long[
        me_long[
            "condition"
        ].isin(
            conditions
        )
    ].copy()

    # -------------------------------------------------------------------------
    # Ensure one mouse per module
    # -------------------------------------------------------------------------

    d = (
        d
        .drop_duplicates(
            [
                "mouse",
                "cell_type",
                "module"
            ]
        )
        .copy()
    )

    d[
        "group"
    ] = (
        d[
            "group"
        ].astype(str)
    )

    d[
        "condition"
    ] = (
        d[
            "condition"
        ].astype(str)
    )

    res = []

    # Non-reference treatment name
    treatment_conditions = [
        x
        for x in conditions
        if x != ref_cond
    ]

    treatment_name = (
        treatment_conditions[0]
        if len(
            treatment_conditions
        ) == 1
        else "Treatment"
    )

    # -------------------------------------------------------------------------
    # Fit one model per cell type x module
    # -------------------------------------------------------------------------

    for (
        ct,
        mod
    ), s in d.groupby(
        [
            "cell_type",
            "module"
        ],
        observed=True
    ):

        s = s.copy()

        n_mice = (
            s[
                "mouse"
            ].nunique()
        )

        # ---------------------------------------------------------------------
        # Count biological replicates in each 2 x 2 group
        # ---------------------------------------------------------------------

        biological_counts = (
            s[
                [
                    "mouse",
                    "group",
                    "condition"
                ]
            ]
            .drop_duplicates()
            .groupby(
                [
                    "group",
                    "condition"
                ],
                observed=True
            )[
                "mouse"
            ]
            .nunique()
        )

        expected_cells = (
            len(
                conditions
            )
            * 2
        )

        if (
            len(
                biological_counts
            )
            < expected_cells
            or biological_counts.min()
            < min_mice_per_cell
        ):

            res.append({
                "cell_type":
                    ct,

                "module":
                    mod,

                "n_mice":
                    n_mice,

                "term":
                    "(not estimable)",

                "estimate":
                    np.nan,

                "se":
                    np.nan,

                "t":
                    np.nan,

                "p":
                    np.nan,

                "df_resid":
                    np.nan
            })

            continue

        # ---------------------------------------------------------------------
        # Mouse-level OLS
        # ---------------------------------------------------------------------

        formula = (
            f"ME ~ "
            f"C(group, Treatment('{ref_group}')) "
            f"* "
            f"C(condition, Treatment('{ref_cond}'))"
        )

        model = smf.ols(
            formula,
            data=s
        ).fit()

        # ---------------------------------------------------------------------
        # Store coefficients
        # ---------------------------------------------------------------------

        for term in model.params.index:

            if (
                term
                == "Intercept"
            ):

                continue

            if (
                term.startswith(
                    "C(group"
                )
                and ":" not in term
            ):

                label = (
                    "genotype (KO-WT)"
                )

            elif (
                term.startswith(
                    "C(condition"
                )
                and ":" not in term
            ):

                label = (
                    f"treatment "
                    f"({treatment_name}-{ref_cond})"
                )

            else:

                label = (
                    "genotype x treatment"
                )

            res.append({
                "cell_type":
                    ct,

                "module":
                    mod,

                "n_mice":
                    n_mice,

                "term":
                    label,

                "estimate":
                    float(
                        model.params[
                            term
                        ]
                    ),

                "se":
                    float(
                        model.bse[
                            term
                        ]
                    ),

                "t":
                    float(
                        model.tvalues[
                            term
                        ]
                    ),

                "p":
                    float(
                        model.pvalues[
                            term
                        ]
                    ),

                "df_resid":
                    int(
                        model.df_resid
                    )
            })

    result = pd.DataFrame(
        res
    )

    # -------------------------------------------------------------------------
    # FDR
    #
    # Correction family:
    # all modules for one cell type and one model term.
    # -------------------------------------------------------------------------

    result[
        "p_fdr"
    ] = np.nan

    valid = (
        result.dropna(
            subset=[
                "p"
            ]
        )
    )

    for (
        ct,
        term
    ), idx in valid.groupby(
        [
            "cell_type",
            "term"
        ],
        observed=True
    ).groups.items():

        result.loc[
            idx,
            "p_fdr"
        ] = multipletests(
            result.loc[
                idx,
                "p"
            ].to_numpy(
                dtype=float
            ),
            method="fdr_bh"
        )[1]

    result[
        "sig"
    ] = np.select(
        [
            result[
                "p_fdr"
            ] < 0.001,

            result[
                "p_fdr"
            ] < 0.01,

            result[
                "p_fdr"
            ] < 0.05
        ],
        [
            "***",
            "**",
            "*"
        ],
        default=""
    )

    return (
        result
        .sort_values(
            [
                "cell_type",
                "term",
                "p"
            ],
            na_position="last"
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# PRIMARY ANALYSIS: VEH + KA3
# =============================================================================

lm_res = fit_mouse_models(
    me_long,
    conditions=(
        "Veh",
        "KA3"
    ),
    ref_group="WT",
    ref_cond="Veh",
    min_mice_per_cell=2
)


print(
    "\n"
    + "=" * 110
)

print(
    "PRIMARY MOUSE-LEVEL MODEL — "
    "ME ~ genotype * treatment "
    "(Veh + KA3)"
)

print(
    "=" * 110
)


print(
    lm_res[
        [
            "cell_type",
            "module",
            "term",
            "estimate",
            "se",
            "t",
            "p",
            "p_fdr",
            "sig",
            "n_mice",
            "df_resid"
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print(
    "\nSignificant at FDR < 0.05:"
)


hits = lm_res[
    lm_res[
        "p_fdr"
    ] < 0.05
]


if len(
    hits
):

    print(
        hits[
            [
                "cell_type",
                "module",
                "term",
                "estimate",
                "p",
                "p_fdr"
            ]
        ]
        .round(
            4
        )
        .to_string(
            index=False
        )
    )

else:

    print(
        "  none"
    )


# =============================================================================
# SECONDARY ANALYSIS: VEH + KA25
# =============================================================================
#
# This will automatically return "(not estimable)" where a genotype x
# condition cell contains fewer than two independent mice.
# =============================================================================

lm_ka25 = fit_mouse_models(
    me_long,
    conditions=(
        "Veh",
        "KA25"
    ),
    ref_group="WT",
    ref_cond="Veh",
    min_mice_per_cell=2
)


print(
    "\n"
    + "=" * 110
)

print(
    "SECONDARY MOUSE-LEVEL MODEL — "
    "Veh + KA25"
)

print(
    "=" * 110
)


print(
    lm_ka25[
        [
            "cell_type",
            "module",
            "term",
            "estimate",
            "p",
            "p_fdr",
            "n_mice"
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# H — DISPLAY MOUSE-LEVEL RESULTS FOR MANUSCRIPT MODULES
# =============================================================================

def plot_mouse_level(
    me_long,
    lm_res,
    celltypes=None,
    modules=None,
    title_suffix=""
):
    """
    Display:

    LEFT:
        Mean mouse-level eigengene by genotype x treatment.

    RIGHT:
        Mouse-level linear-model coefficients.

    Nothing is written to disk.
    """

    d = me_long.copy()

    if celltypes is not None:

        d = d[
            d[
                "cell_type"
            ].isin(
                celltypes
            )
        ]

    if modules is not None:

        d = d[
            d[
                "module"
            ].isin(
                modules
            )
        ]

    if d.empty:

        print(
            "Nothing to plot."
        )

        return

    d[
        "trait"
    ] = (
        d[
            "group"
        ].astype(str)
        + "_"
        + d[
            "condition"
        ].astype(str)
    )

    desired_order = [
        "WT_Veh",
        "KO_Veh",
        "WT_KA3",
        "KO_KA3",
        "WT_KA25",
        "KO_KA25"
    ]

    observed_traits = set(
        d[
            "trait"
        ]
    )

    order = [
        x
        for x in desired_order
        if x in observed_traits
    ]

    # -------------------------------------------------------------------------
    # Mouse-level means
    # -------------------------------------------------------------------------

    mean = (
        d.pivot_table(
            index="trait",
            columns="module",
            values="ME",
            aggfunc="mean"
        )
        .reindex(
            order
        )
    )

    nmice = (
        d.pivot_table(
            index="trait",
            columns="module",
            values="mouse",
            aggfunc="nunique"
        )
        .reindex(
            order
        )
    )

    labels = (
        mean.round(
            2
        ).astype(str)
        + "\n(n="
        + nmice.astype(
            "Int64"
        ).astype(str)
        + ")"
    )

    # -------------------------------------------------------------------------
    # Figure
    # -------------------------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(
            max(
                14,
                mean.shape[1]
                * 2.6
            ),
            6
        ),
        gridspec_kw={
            "width_ratios": [
                2,
                1
            ]
        },
        facecolor="white"
    )

    v = np.nanmax(
        np.abs(
            mean.values
        )
    )

    if (
        not np.isfinite(
            v
        )
        or v == 0
    ):

        v = 1

    sns.heatmap(
        mean,
        annot=labels,
        fmt="",
        cmap="RdBu_r",
        center=0,
        vmin=-v,
        vmax=v,
        annot_kws={
            "size": 11,
            "weight": "bold"
        },
        linewidths=0.5,
        linecolor="gray",
        ax=axes[0],
        cbar_kws={
            "label":
                "Mean mouse-level module eigengene"
        }
    )

    axes[
        0
    ].set_title(
        "Metacell-supported mouse-level "
        f"module eigengene{title_suffix}",
        fontsize=14,
        fontweight="bold"
    )

    axes[
        0
    ].set_xlabel(
        ""
    )

    axes[
        0
    ].set_ylabel(
        ""
    )

    axes[
        0
    ].set_xticklabels(
        axes[
            0
        ].get_xticklabels(),
        rotation=45,
        ha="right",
        fontweight="bold"
    )

    axes[
        0
    ].set_yticklabels(
        axes[
            0
        ].get_yticklabels(),
        rotation=0,
        fontweight="bold"
    )

    # -------------------------------------------------------------------------
    # Linear-model coefficients
    # -------------------------------------------------------------------------

    L = lm_res[
        (
            lm_res[
                "module"
            ].isin(
                mean.columns
            )
        )
        &
        (
            lm_res[
                "term"
            ]
            != "(not estimable)"
        )
    ].copy()

    if L.empty:

        axes[
            1
        ].axis(
            "off"
        )

        axes[
            1
        ].text(
            0.5,
            0.5,
            "No estimable mouse-level models",
            ha="center",
            va="center"
        )

        plt.tight_layout()

        plt.show()

        return

    est = L.pivot_table(
        index="term",
        columns="module",
        values="estimate"
    )

    pfd = L.pivot_table(
        index="term",
        columns="module",
        values="p_fdr"
    )

    desired_terms = [
        "genotype (KO-WT)",
        "treatment (KA3-Veh)",
        "genotype x treatment"
    ]

    term_order = [
        x
        for x in desired_terms
        if x in est.index
    ]

    # Fall back to whatever is present
    if len(
        term_order
    ) == 0:

        term_order = list(
            est.index
        )

    est = est.reindex(
        term_order
    )

    pfd = pfd.reindex(
        term_order
    )

    stars = np.where(
        pfd < 0.001,
        "\n***",
        np.where(
            pfd < 0.01,
            "\n**",
            np.where(
                pfd < 0.05,
                "\n*",
                "\nns"
            )
        )
    )

    labels2 = (
        est.round(
            2
        ).astype(str)
        + stars
    )

    v2 = np.nanmax(
        np.abs(
            est.values
        )
    )

    if (
        not np.isfinite(
            v2
        )
        or v2 == 0
    ):

        v2 = 1

    sns.heatmap(
        est,
        annot=labels2,
        fmt="",
        cmap="PuOr_r",
        center=0,
        vmin=-v2,
        vmax=v2,
        annot_kws={
            "size": 11,
            "weight": "bold"
        },
        linewidths=0.5,
        linecolor="gray",
        ax=axes[1],
        cbar_kws={
            "label":
                "Mouse-level LM coefficient"
        }
    )

    axes[
        1
    ].set_title(
        "ME ~ genotype × treatment\n"
        "(mouse is the biological replicate)",
        fontsize=13,
        fontweight="bold"
    )

    axes[
        1
    ].set_xlabel(
        ""
    )

    axes[
        1
    ].set_ylabel(
        ""
    )

    axes[
        1
    ].set_xticklabels(
        axes[
            1
        ].get_xticklabels(),
        rotation=45,
        ha="right",
        fontweight="bold"
    )

    axes[
        1
    ].set_yticklabels(
        axes[
            1
        ].get_yticklabels(),
        rotation=0,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.show()


# =============================================================================
# YOUR FIGURE-3 MODULE GROUPS
# =============================================================================

dic_cell_modules = {

    "DG": {
        "genotype": [
            "dimgrey",
            "lightgrey"
        ],
        "drugeffect": [
            "silver",
            "darkgrey",
            "gainsboro"
        ],
        "interactive": [
            "dimgrey"
        ]
    },

    "CA1": {
        "genotype": [
            "dimgrey",
            "darkgrey"
        ],
        "drugeffect": [
            "silver"
        ],
        "interactive": [
            "dimgrey"
        ]
    },

    "CA3": {
        "genotype": [
            "dimgrey",
            "indianred"
        ],
        "drugeffect": [
            "silver",
            "darkgrey"
        ],
        "interactive": [
            "dimgrey"
        ]
    },

    "Inhibitory": {
        "genotype": [
            "dimgrey",
            "lightcoral"
        ],
        "drugeffect": [],
        "interactive": [
            "dimgrey",
            "lightcoral"
        ]
    }
}


# =============================================================================
# DISPLAY PREVIOUSLY HIGHLIGHTED MODULES
# =============================================================================

for effect in [
    "genotype",
    "drugeffect",
    "interactive"
]:

    want = []

    for ct, effects in dic_cell_modules.items():

        for colour in effects[
            effect
        ]:

            if (
                ct in COLOR2IDX
                and colour in COLOR2IDX[
                    ct
                ]
            ):

                want.append(
                    f"{ct}-"
                    f"{COLOR2IDX[ct][colour]}"
                )

    if len(
        want
    ) == 0:

        print(
            f"\nNo modules for '{effect}'."
        )

        continue

    # Only modules actually represented in the new mouse-level object
    available = set(
        me_long[
            "module"
        ].astype(str)
    )

    want_present = [
        x
        for x in want
        if x in available
    ]

    missing = [
        x
        for x in want
        if x not in available
    ]

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{effect.upper()} MODULES:"
    )

    print(
        ", ".join(
            want_present
        )
    )

    print(
        "=" * 100
    )

    if missing:

        print(
            "\nRequested modules not found "
            "in the current module-score object:"
        )

        print(
            ", ".join(
                missing
            )
        )

    if len(
        want_present
    ) == 0:

        continue

    plot_mouse_level(
        me_long,
        lm_res,
        modules=want_present,
        title_suffix=f" — {effect}"
    )

    selected_results = lm_res[
        lm_res[
            "module"
        ].isin(
            want_present
        )
    ][
        [
            "cell_type",
            "module",
            "term",
            "estimate",
            "se",
            "p",
            "p_fdr",
            "sig",
            "n_mice"
        ]
    ]

    print(
        "\nMouse-level linear-model results:"
    )

    print(
        selected_results
        .round(
            4
        )
        .to_string(
            index=False
        )
    )

### 18. Approach 2 - WGCNA co-expression networks
Builds cell-type-specific weighted co-expression networks. Within each treatment condition, KO and WT nuclei are randomly downsampled to the smaller group so that every network is genotype-balanced.

In [ ]:
import networkx as nx
from scipy.stats import spearmanr
from sklearn.cluster import AgglomerativeClustering
import PyWGCNA

adata_combined_all_genes = adata_filtered_significant_genes.copy()
clusters = adata_combined_all_genes.obs["celltype"].unique()
top_genes_after_remove = {}
up_counts = []
down_counts = []

log2fc_dict = {}  # Store log2 fold changes for each cluster
p_values_dict={}
"""
For cluster ('Inhibitory', <class 'str'>) ...
For cluster ('Oligo', <class 'str'>) ...
For cluster ('Micro-glia', <class 'str'>) ...
For cluster ('CA1', <class 'str'>) ...
For cluster ('CA3', <class 'str'>) ...
For cluster ('Astro', <class 'str'>) ...
For cluster ('DG', <class 'str'>) ...
"""
#dic_name_cells = {'Inhibitory':'Inhibitory', 'CA1':'CA1', 'CA3':'CA3', 'DG':'DG', 'Oligo':'Oligo', 'Micro-glia':'Microglia', 'Astro':'Astro'}
dic_name_cells = {'Inhibitory':'Inhibitory', 'CA1':'CA1', 'CA3':'CA3', 'DG':'DG', 'Oligo':'Oligo', 'Microglia':'Microglia', 'Astro':'Astro'}

for CLUSTER_TO_LOOK in [['Inhibitory'], ['CA1'], ['CA3'], ['DG'], ['Oligo'], ['Micro-glia'], ['Astro']]:
    # CLUSTER_TO_LOOK = ["Micro-glia"]
    for cluster in CLUSTER_TO_LOOK:
    
        print(f"For cluster {cluster, type(cluster)} ...")
        
        # Subset the data for the current cluster
        cluster_data = adata_combined_all_genes[adata_combined_all_genes.obs['celltype'] == cluster]
        
        #cluster_data = cluster_data[cluster_data.obs['group'] == 'KO']
        
        # # Get condition-wise counts
        # condition_counts = cluster_data.obs['condition'].value_counts()
        # print(f"condition_counts {condition_counts}")
        # min_cells = condition_counts.min()
        # print(f"min_cells {min_cells}")
        # # Downsample each condition to min_cells
        # downsampled_indices = []
        # for cond in condition_counts.index:
        #     cond_indices = cluster_data.obs[cluster_data.obs['condition'] == cond].index
        #     sampled = np.random.choice(cond_indices, size=min_cells, replace=False)
        #     downsampled_indices.extend(sampled)
    
    
    
    
        print("Before downsampling:")
        print(cluster_data.obs.groupby(['condition', 'group']).size())
        
        # Get the list of unique conditions
        conditions = cluster_data.obs['condition'].unique()
        groups = cluster_data.obs['group'].unique()
        
        downsampled_indices = []
    
        for cond in conditions:
            # For this condition, get cell counts in each group
            cond_mask = cluster_data.obs['condition'] == cond
            group_counts = cluster_data.obs[cond_mask].groupby('group').size()
    
            # Only proceed if both groups exist for this condition
            if len(group_counts) == len(groups):
                min_cells = group_counts.min()
                print(f"Condition: {cond} | min cells across KO/WT: {min_cells}")
    
                for grp in groups:
                    cond_grp_mask = (cluster_data.obs['condition'] == cond) & (cluster_data.obs['group'] == grp)
                    indices = cluster_data.obs[cond_grp_mask].index
                    sampled = np.random.choice(indices, size=min_cells, replace=False)
                    downsampled_indices.extend(sampled)
            else:
                print(f"Skipping condition {cond} for cluster {cluster} due to missing group(s).")
        
        # Filter to only the downsampled cells
        # Downsampled AnnData
        cluster_data = cluster_data[downsampled_indices]
    
        #### Change #######
        Background_gene_names = pd.DataFrame(cluster_data.var_names, columns=["Gene"])
        Background_output_path = f"./CSV_files/Background_genes_{cluster}_YekBasteh.csv"
        Background_gene_names.to_csv(Background_output_path, index=False)
        #### Change #######
        print("After downsampling:")
        print(cluster_data.obs.groupby(['condition', 'group']).size())
        
        
    
        
        # print(cluster_data.obs['condition'].unique())
    
        # #print(cluster_data.obs['condition'].value_counts())
        # print(eeee)
        # Perform differential expression analysis
    
        #reference='Veh',
        # sc.tl.rank_genes_groups(cluster_data, groupby='condition', reference='Veh', method='wilcoxon')#    sc.tl.rank_genes_groups(cluster_data, groupby='group', reference='WT', method='wilcoxon')
    
        # sc.pl.rank_genes_groups(cluster_data, n_genes=50, sharey=False)
    
    
        
        
        ###
        # Convert the gene expression matrix (adata.X) to a DataFrame
        gene_expression = pd.DataFrame(cluster_data.X.toarray() if hasattr(cluster_data.X, "toarray") else cluster_data.X)
        
        # Add gene names from adata.var as column names
        gene_expression.columns = cluster_data.var.index
        
        # Add sample names from adata.obs as row indices
        gene_expression.index = cluster_data.obs.index
        
        # Save to CSV
        gene_expression.to_csv(f'./gene_expression_data_{cluster}_YekBasteh.csv')
        geneExp = f'./gene_expression_data_{cluster}_YekBasteh.csv'
        pyWGCNA_KO_CD47 = PyWGCNA.WGCNA(name=f'{dic_name_cells[cluster]} modules_YekBasteh', 
                                  species='mus musculus', 
                                  geneExpPath=geneExp, 
                                  outputPath='',
                                  save=True)
        pyWGCNA_KO_CD47.geneExpr.to_df().head(5)
        pyWGCNA_KO_CD47.preprocess(show=False)
        pyWGCNA_KO_CD47.findModules()
    
        
    
        print(f"cluster ..... {cluster_data.obs.keys()} ")
        selected_data = cluster_data.obs[['group', 'condition']]
        
        # Save the DataFrame to a CSV file
        selected_data.to_csv(f'./secondary_info_{cluster}_YekBasteh.csv', index=True) 
    
    
        pyWGCNA_KO_CD47.updateSampleInfo(path=f'./secondary_info_{cluster}_YekBasteh.csv', sep=',')
        
        # add color for metadata
        # pyWGCNA_KO_CD47.setMetadataColor('sex', {'female': 'green',
        #                                        'male': 'yellow'})
        
        # # This assumes 'group' and 'condition' columns exist in the input CSV
        # pyWGCNA_KO_CD47.datExpr.obs['group_condition'] = pyWGCNA_KO_CD47.datExpr.obs['group'].astype(str) + '_' + pyWGCNA_KO_CD47.datExpr.obs['condition'].astype(str)
        
        # Step 3: Assign colors to each group_condition combination
        # pyWGCNA_5xFAD.setMetadataColor('sex', {'female': 'green',
        #                                        'male': 'yellow'})
        pyWGCNA_KO_CD47.setMetadataColor('group', {'KO': 'darkviolet',
                                               'WT': 'deeppink'})
        pyWGCNA_KO_CD47.setMetadataColor('condition', {'Veh': 'blue',
                                                    'KA3': 'red',
                                                    'KA25': 'purple'})
        
        # pyWGCNA_KO_CD47.setMetadataColor('group_condition', {
        #     'KO_Veh': 'blue',
        #     'KO_Low': 'red',
        #     'KO_High': 'orange',
        #     'WT_Veh': 'green',
        #     'WT_Low': 'purple',
        #     'WT_High': 'brown'
        # })
        # print(f"Done")
    
        cluster_data_filtered = cluster_data
        gene_names = cluster_data_filtered.var['gene_ids']
    
        gene_list_df = pd.DataFrame(gene_names)
        gene_list_df.columns = ['gene_name']
        #### *******
        print(gene_list_df.columns)
        
        #### *******
        # keys_df = pd.DataFrame(cluster_data.var.keys(), columns=['Variable Keys'])
    
        # # Save the DataFrame to a CSV file
        # keys_df.to_csv('var_keys.csv', index=False)
        pyWGCNA_KO_CD47.updateGeneInfo(gene_list_df)
    
        
        
        #pyWGCNA_KO_CD47.analyseWGCNA()
        # print(f"pyWGCNA_KO_CD47.datExpr.obs.columns.tolist() {pyWGCNA_KO_CD47.datExpr.obs.columns.tolist()}")
        # pyWGCNA_KO_CD47.module_trait_relationships_heatmap(metaData=['group_condition'],file_name='module-trait Relationships for Microglia')
    
        
        #pyWGCNA_KO_CD47.datExpr.var.head(5)
        modules = pyWGCNA_KO_CD47.datExpr.var.moduleColors.unique().tolist()
        pyWGCNA_KO_CD47.analyseWGCNA()
    
        # This assumes 'group' and 'condition' columns exist in the input CSV
        pyWGCNA_KO_CD47.datExpr.obs['group_condition'] = pyWGCNA_KO_CD47.datExpr.obs['group'].astype(str) + '_' + pyWGCNA_KO_CD47.datExpr.obs['condition'].astype(str)
        
        pyWGCNA_KO_CD47.setMetadataColor('group_condition', {
            'KO_Veh': 'blue',
            'KO_Low': 'red',
            'KO_High': 'orange',
            'WT_Veh': 'green',
            'WT_Low': 'purple',
            'WT_High': 'brown'
        })
        print(f"Done")
        
        print(f"pyWGCNA_KO_CD47.datExpr.obs.columns.tolist() {pyWGCNA_KO_CD47.datExpr.obs.columns.tolist()}")
        result_corr_pvalue_df = pyWGCNA_KO_CD47.module_trait_relationships_heatmap(metaData=['group_condition'],file_name=f'module-trait Relationships for {dic_name_cells[cluster]}_YekBasteh')
        
        result_corr_pvalue_df.reset_index().to_csv(f"./CSV_files/trait_module_{dic_name_cells[cluster]}_YekBasteh.csv", index=False)
    
        
        pyWGCNA_KO_CD47.saveWGCNA()
    
    
        
    
    
        
        # # -----------------  top genes   ----------------------
        # top_genes = get_top_genes_from_rank(cluster_data)
        # if top_genes:
        #     enrichment_results = sc.queries.enrich(top_genes, org='mmusculus')
        #     print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
        
        #     # Transform p-values to -log10(p-value)
        #     enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
        #     top_n = 50
        #     top_enriched = enrichment_results.head(top_n)
        #     print(f"top enriched {top_enriched}")
        
        #     # Create a bar plot using transformed p-values
        #     plt.figure(figsize=(10, 6))
        #     sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        #     plt.xlabel('-log10(p-value)')
        #     plt.ylabel('GO Term')
        #     plt.title(f'Top {top_n} Enriched GO Terms')
        #     plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        #     plt.show()
        # else:
        #     print(f"No significant DEGs found for cluster {cluster}.")
        # # -----------------------------------------------------
        
        # # Extract top genes and scores for the current cluster
        # cluster_genes = cluster_data.uns['rank_genes_groups']['names']  # Structured array
        # cluster_scores = cluster_data.uns['rank_genes_groups']['scores']  # Structured array
        # #print(f"cluster_scores {cluster_scores}")
        # log2fc_values = cluster_data.uns['rank_genes_groups']['logfoldchanges']  # Extract log2 fold changes
        # p_values = cluster_data.uns['rank_genes_groups']['pvals'] 
        # #print(f"log2fc_values {log2fc_values}")
        # # Check available group names in rank_genes_groups
        # group_names = list(cluster_genes.dtype.names)
        # #print(f"Available groups in rank_genes_groups: {group_names}")
    
        # log2fc_dict[cluster] = dict()
        # p_values_dict[cluster]=dict()
        # for group_name in group_names:
        #     # Convert to dictionary for easy retrieval
        #     log2fc_dict[cluster][group_name] = {gene: fc for gene, fc in zip(cluster_genes[group_name], log2fc_values[group_name])}
        #     #print(f"group_name{group_name} cluster {cluster}: {log2fc_dict[cluster][group_name]}")
        #     p_values_dict[cluster][group_name]={gene: pval for gene, pval in zip(cluster_genes[group_name], p_values[group_name])}
        #     # Convert dict values to NumPy array
        #     log2fc_values_array = np.array(list(log2fc_dict[cluster][group_name].values()))
        #     p_values_array = np.array(list(p_values_dict[cluster][group_name].values()))
        #     genes_array = np.array(cluster_genes[group_name]) 
        #     # Count upregulated and downregulated genes
        #     up_count = np.sum(log2fc_values_array > 0)  # Count upregulated genes
        #     down_count = np.sum(log2fc_values_array < 0)  # Count downregulated genes
    
        #     up_counts.append(up_count)
        #     down_counts.append(down_count)
    
        #     #print(f"Cluster {cluster}, Condition {group_name}: {up_count} upregulated, {down_count} downregulated genes.")
        #     # -----------------  Volcano Plot with Gene Annotations  ----------------------
        #     plt.figure(figsize=(8, 6))
    
        #     # Scatter plot: All genes (gray)
        #     plt.scatter(log2fc_values_array, -np.log10(p_values_array), color="grey", alpha=0.6)
    
        #     # Highlight upregulated genes (red)
        #     upregulated = log2fc_values_array > 0
        #     plt.scatter(
        #         log2fc_values_array[upregulated], 
        #         -np.log10(p_values_array[upregulated]), 
        #         color="red", label="Upregulated"
        #     )
    
        #     # Highlight downregulated genes (blue)
        #     downregulated = log2fc_values_array < 0
        #     plt.scatter(
        #         log2fc_values_array[downregulated], 
        #         -np.log10(p_values_array[downregulated]), 
        #         color="blue", label="Downregulated"
        #     )
    
        #     # Annotate top upregulated and downregulated genes
        #     top_n_genes = 10  # Adjust number of labels for each side
            
        #     # Separate significant upregulated and downregulated genes
        #     sorted_idx_up = np.argsort(p_values_array[upregulated])[:top_n_genes]  # Select top upregulated genes
        #     sorted_idx_down = np.argsort(p_values_array[downregulated])[:top_n_genes]  # Select top downregulated genes
            
        #     # Label upregulated genes on the right side of the dashed line (log2FC > 0)
        #     for idx in sorted_idx_up:
        #         gene_name = genes_array[upregulated][idx]  # Extract gene name
        #         plt.text(log2fc_values_array[upregulated][idx], -np.log10(p_values_array[upregulated][idx]), 
        #                  gene_name, fontsize=9, ha='left', va='bottom', color='black')  # Align text to the left
            
        #     # Label downregulated genes on the left side of the dashed line (log2FC < 0)
        #     for idx in sorted_idx_down:
        #         gene_name = genes_array[downregulated][idx]  # Extract gene name
        #         plt.text(log2fc_values_array[downregulated][idx], -np.log10(p_values_array[downregulated][idx]), 
        #                  gene_name, fontsize=9, ha='right', va='bottom', color='black')  # Align text to the right
            
        #     # Add significance threshold lines
        #     plt.axhline(y=-np.log10(0.05), linestyle="dashed", color="black", alpha=0.7)  # p-value threshold
        #     plt.axvline(x=0, linestyle="dashed", color="black", alpha=0.7)  # No change threshold
            
        #     plt.xlabel("log2 Fold Change")
        #     plt.ylabel("-log10(p-value)")
        #     plt.title(f"Volcano Plot - Combined Cluster {cluster} ({group_name})")
        #     plt.legend()
        #     plt.show()
            


### 19. Enrichment background
Defines the retained gene set used as the universe for enrichment testing.

In [ ]:
print(adata_combined.shape)
print(adata_combined_all_genes.shape)
print(adata_filtered_significant_genes.shape)
# print(adata_filtered_LowQuality.shape)

In [ ]:
import os
import pandas as pd

# Define the output path
Background_ALL_output_path = "./CSV_files/Background_ALL_gene_names_YekBasteh.csv"

# Check if file exists
# if os.path.exists(Background_ALL_output_path):
#     print("✅ Background gene list found. Loading...")
#     Background_ALL_gene_names = pd.read_csv(Background_ALL_output_path)
# else:
print("🚀 Background gene list not found. Creating and saving...")

"""
adata_combined has cortex data
adata_filtered_LowQuality does not have that info
"""
# Background_ALL_gene_names = pd.DataFrame(adata_combined.var_names, columns=["Gene"])
Background_ALL_gene_names = pd.DataFrame(adata_combined_all_genes.var_names, columns=["Gene"])

Background_ALL_gene_names.to_csv(Background_ALL_output_path, index=False)

# Convert to list
background_all = Background_ALL_gene_names['Gene'].dropna().tolist()

# import pandas as pd
# #### Change #######
# Background_ALL_gene_names = pd.DataFrame(adata_combined.var_names, columns=["Gene"])
# Background_ALL_output_path = f"./CSV_files/Background_ALL_gene_names.csv"
# Background_ALL_gene_names.to_csv(Background_ALL_output_path, index=False)
# background_all =  Background_ALL_gene_names['Gene'].dropna().tolist()
# #### Change #######

In [ ]:
import pandas as pd
from itertools import combinations
import os
# Path to your CSV files (adjust the path as needed)
directory = "./CSV_files"

# Dictionary to store each list of genes
dictionary_background = {}

# Iterate over files in the directory
for file_name in os.listdir(directory):
    # Ensure the file is a CSV file
    if file_name.startswith("Background_genes_") and file_name.endswith("_YekBasteh.csv"):
        # Construct the full file path
        file_path = os.path.join(directory, file_name)
        
        # Extract the base name without extension
        base_name = os.path.splitext(file_name)[0].split('_')[-1]
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Extract the "Gene" column as a list
        if "Gene" in df.columns:
            genes = df["Gene"].dropna().tolist()  # Drop NaN values to avoid empty entries
            dictionary_background[base_name] = genes
        else:
            print(f"Warning: 'Gene' column not found in {file_name}")

# Example: Access the gene list for a specific file
# print(dictionary_background)


# Function to calculate Jaccard similarity
def jaccard_similarity(list1, list2):
    set1,set2 = set(list1), set(list2)
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union != 0 else 0

jaccard_scores = {}

for (cell_type1, gene_list1),(cell_type2, gene_list2) in combinations(dictionary_background.items(),2):
    score = jaccard_similarity(gene_list1, gene_list2)
    jaccard_scores[(cell_type1, cell_type2)] = score

# Display the Jaccard similarity scores
for pair, score in jaccard_scores.items():
    print(f"Jaccard similarity between {pair[0]} and {pair[1]}: {score:.4f}")

### 20. Module membership
Reads the PyWGCNA objects and reports the number of genes per module (Supplementary Table 2).

In [ ]:
import PyWGCNA
import os

pyWGCNA_CA1 = PyWGCNA.readWGCNA("./CA1 modules_YekBasteh.p")
pyWGCNA_CA3 = PyWGCNA.readWGCNA("./CA3 modules_YekBasteh.p")
pyWGCNA_DG = PyWGCNA.readWGCNA("./DG modules_YekBasteh.p")
pyWGCNA_Inhibitory = PyWGCNA.readWGCNA("./Inhibitory modules_YekBasteh.p")


pyWGCNA_list = [pyWGCNA_CA1, pyWGCNA_CA3, pyWGCNA_DG, pyWGCNA_Inhibitory]  # each for a different cell type

# Collect module colors from each object

given_mapper_color_to_idx_NEURON = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_NEURON: 
            given_mapper_color_to_idx_NEURON[color]=index_colorname
            index_colorname +=1



pyWGCNA_Oligo = PyWGCNA.readWGCNA("./Oligo modules_YekBasteh.p")
pyWGCNA_Micro = PyWGCNA.readWGCNA("./Microglia modules_YekBasteh.p")
pyWGCNA_Astro = PyWGCNA.readWGCNA("./Astro modules_YekBasteh.p")

pyWGCNA_list = [pyWGCNA_Astro, pyWGCNA_Oligo, pyWGCNA_Micro]  # each for a different cell type

# Collect module colors from each object
given_mapper_color_to_idx_GLIA = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_GLIA: 
            given_mapper_color_to_idx_GLIA[color]=index_colorname
            index_colorname +=1
        


## --------- adjusting mapper to module names -> indices  -----------
dict_celltype_color_to_idx_WHOLE = {}

# Neuronal cell types
for celltype in ['CA1', 'CA3', 'DG', 'Inhibitory']:
    m = given_mapper_color_to_idx_NEURON.copy()  # make an independent copy
    if celltype != 'CA3':
        # swap dimgrey <-> darkgrey if both exist
        if 'dimgrey' in m and 'darkgrey' in m:
            m['dimgrey'], m['darkgrey'] = m['darkgrey'], m['dimgrey']
    dict_celltype_color_to_idx_WHOLE[celltype] = m

# Glial cell types
for celltype in ['Astro', 'Microglia', 'Oligo']:
    m = given_mapper_color_to_idx_GLIA.copy()  # independent copy
    # swap dimgrey <-> gainsboro if both exist
    if 'dimgrey' in m and 'gainsboro' in m:
        m['dimgrey'], m['gainsboro'] = m['gainsboro'], m['dimgrey']
    dict_celltype_color_to_idx_WHOLE[celltype] = m
print(f"whole mapping: {dict_celltype_color_to_idx_WHOLE}")

In [ ]:
## print the number of genes per module

# Group your objects into a dictionary to match the keys in dict_celltype_color_to_idx_WHOLE
all_objects = {
    "CA1": pyWGCNA_CA1, 
    "CA3": pyWGCNA_CA3, 
    "DG": pyWGCNA_DG, 
    "Inhibitory": pyWGCNA_Inhibitory,
    "Astro": pyWGCNA_Astro,
    "Microglia": pyWGCNA_Micro,
    "Oligo": pyWGCNA_Oligo
}

print(f"{'Cell Type':<12} | {'Module ID':<10} | {'Gene Count':<10}")
print("-" * 38)

for cell_type, obj in all_objects.items():
    # 1. Get the gene counts for each color in this specific object
    module_counts = obj.datExpr.var['moduleColors'].value_counts()
    
    # 2. Get the specific color-to-index mapper for this cell type
    current_mapper = dict_celltype_color_to_idx_WHOLE.get(cell_type, {})

    for color, count in module_counts.items():
        # 3. Look up the index number using the color name
        # .get() is safer in case a color exists in the data but not in your mapper
        module_id = current_mapper.get(color, "N/A")
        
        print(f"{cell_type}-{module_id:<10} | {count:<10}")
    
    print("-" * 38) # Optional: line break between different cell types

In [ ]:
## Print Genes in 
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

import PyWGCNA
import pandas as pd

import networkx as nx
from scipy.stats import spearmanr
from sklearn.cluster import AgglomerativeClustering


libraries = gp.get_library_name(organism='Mouse')
print(libraries)

libraries = gp.get_library_name()
kegg_libraries = [lib for lib in libraries if "KEGG" in lib and "Mouse" in lib]
print(kegg_libraries)
# print(cc)



pyWGCNA_CA1 = PyWGCNA.readWGCNA("./CA1 modules_YekBasteh.p")
pyWGCNA_CA3 = PyWGCNA.readWGCNA("./CA3 modules_YekBasteh.p")
pyWGCNA_DG = PyWGCNA.readWGCNA("./DG modules_YekBasteh.p")
pyWGCNA_Inhibitory = PyWGCNA.readWGCNA("./Inhibitory modules_YekBasteh.p")


pyWGCNA_list = [pyWGCNA_CA1, pyWGCNA_CA3, pyWGCNA_DG, pyWGCNA_Inhibitory]  # each for a different cell type

# Collect module colors from each object

given_mapper_color_to_idx_NEURON = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_NEURON: 
            given_mapper_color_to_idx_NEURON[color]=index_colorname
            index_colorname +=1
print(f"given_mapper_color_to_idx_NEURON: {given_mapper_color_to_idx_NEURON}")



pyWGCNA_Oligo = PyWGCNA.readWGCNA("./Oligo modules_YekBasteh.p")
pyWGCNA_Micro = PyWGCNA.readWGCNA("./Microglia modules_YekBasteh.p")
pyWGCNA_Astro = PyWGCNA.readWGCNA("./Astro modules_YekBasteh.p")

pyWGCNA_list = [pyWGCNA_Astro, pyWGCNA_Oligo, pyWGCNA_Micro]  # each for a different cell type

# Collect module colors from each object
given_mapper_color_to_idx_GLIA = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_GLIA: 
            given_mapper_color_to_idx_GLIA[color]=index_colorname
            index_colorname +=1
        
# Now all_colors is the union of all module colors
print(f"given_mapper_color_to_idx_GLIA: {given_mapper_color_to_idx_GLIA}")
"""
given_mapper_color_to_idx_GLIA: {'dimgrey': 1, 'gainsboro': 2, 'silver': 3, 'black': 4, 'lightcoral': 5, 'brown': 6, 'snow': 7, 'darkgrey': 8, 'whitesmoke': 9, 'indianred': 10, 'maroon': 11, 'lightgrey': 12, 'firebrick': 13, 'rosybrown': 14, 'white': 15, 'mistyrose': 16, 'tomato': 17, 'salmon': 18, 'darkred': 19, 'red': 20}
"""


## --------- adjusting mapper to module names -> indices  -----------
dict_celltype_color_to_idx_WHOLE = {}

# Neuronal cell types
for celltype in ['CA1', 'CA3', 'DG', 'Inhibitory']:
    m = given_mapper_color_to_idx_NEURON.copy()  # make an independent copy
    if celltype != 'CA3':
        # swap dimgrey <-> darkgrey if both exist
        if 'dimgrey' in m and 'darkgrey' in m:
            m['dimgrey'], m['darkgrey'] = m['darkgrey'], m['dimgrey']
    dict_celltype_color_to_idx_WHOLE[celltype] = m

# Glial cell types
for celltype in ['Astro', 'Microglia', 'Oligo']:
    m = given_mapper_color_to_idx_GLIA.copy()  # independent copy
    # swap dimgrey <-> gainsboro if both exist
    if 'dimgrey' in m and 'gainsboro' in m:
        m['dimgrey'], m['gainsboro'] = m['gainsboro'], m['dimgrey']
    dict_celltype_color_to_idx_WHOLE[celltype] = m
print(f"whole mapping: {dict_celltype_color_to_idx_WHOLE}")
# for cell_type_name in ['CA1','CA3','DG','Inhibitory','Oligodendrocytes','Microglia','Astrocytes']:
# for cell_type_name in ['CA1','CA3','DG','Inhibitory','Oligo','Microglia','Astro']:
for cell_type_name in ['CA1', 'Inhibitory']:#['CA1', 'CA3', 'DG', 'Inhibitory','Oligo','Microglia','Astro']:

    pyWGCNA_obj = PyWGCNA.readWGCNA(f"{cell_type_name} modules_YekBasteh.p")
    
    
    module_colors = pyWGCNA_obj.datExpr.var['moduleColors']
    
    # Group genes by module and print
    modules = module_colors.unique()
    
    for module in modules:
        genes_in_module = module_colors[module_colors == module].index.tolist()
        # print(f"\nModule: {module} ({len(genes_in_module)} genes)")
        # for gene in genes_in_module:
        #     print(f"  {gene}")
        
        #### finding enrichment
        # enrichment_results = sc.queries.enrich(genes_in_module, org='mmusculus')
        # print(f"enrichment_results {(enrichment_results[['name','p_value']])}")
        
        # # Transform p-values to -log10(p-value)
        # enrichment_results['neg_log10_p_value'] = -np.log10(enrichment_results['p_value'])
        
        # top_n = 50
        # top_enriched = enrichment_results.head(top_n)
        # print(f"top enriched {top_enriched}")
        
        # # Create a bar plot using transformed p-values
        # plt.figure(figsize=(15, 10))
        # sns.barplot(x='neg_log10_p_value', y='name', data=top_enriched, palette='viridis')
        # plt.xlabel('-log10(p-value)')
        # plt.ylabel('GO Term')
        # plt.title(f'Top {top_n} Enriched GO Terms for module {module} of {cell_type_name}')
        # plt.gca().invert_yaxis()  # Invert y-axis to have the most significant term at the top
        # plt.show()    
    
    
        
    
        # Input: List of gene symbols or IDs
        # gene_list = ["TP53", "BRCA1", "EGFR", "MYC", "MTOR"]
        
        # Run ORA
        
        gene_list = genes_in_module
        if cell_type_name in {'CA1','CA3','DG','Inhibitory'}:
            gene_sets = ['GO_Biological_Process_2025', 'GO_Cellular_Component_2025',
    'GO_Molecular_Function_2025', 'WikiPathways_2024_Mouse','Reactome_Pathways_2024','Reactome_2022','SynGO_2024']
        else:
            gene_sets = ['GO_Biological_Process_2025', 'WikiPathways_2024_Mouse','Reactome_Pathways_2024','Reactome_2022']
        
        #gene_sets=['DisGeNET', 'PanglaoDB_Augmented_2021', 'GWAS_Catalog_2023']
        
        print("Gene List Length:", len(gene_list))
        #print("First 5 Genes:", gene_list)
        print(f"Cell type:{cell_type_name}. - MODULE: {module}")
        # gene_sets = ['WikiPathways_2024_Mouse','Reactome_Pathways_2024','Panther_2016']
        
        # [
        #     "Allen_Brain_Atlas_10x_scRNA_2021",
        #     "HDSigDB_Mouse_2021",
        #     "MGI_Mammalian_Phenotype_Level_4_2024",
        #     "Mouse_Gene_Atlas",
        #     "Tabula_Muris",
        #     "WikiPathways_2024_Mouse"
        # ]
        try:
            enrichment_results = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gene_sets,#background= background_all,#dictionary_background[cell_type_name],
                organism='Mouse',
                cutoff=0.05,
                format='png'
            )
            
            ##  KEGG pathways
            # enrichment_results = gp.enrichr(
            #     gene_list=gene_list,
            #     gene_sets=kegg_libraries,  ## THIS SHOULD BE FOR KEGG ONLY. #background= dictionary_background[cluster],
            #     organism='Mouse',
            #     outdir=None,
            #     cutoff=0.05,  # More lenient
            #     format='png'
            # )
            # Check if the result has any significant terms
            if enrichment_results.results.empty:
                print(f"No significant enrichment for Cell type:{cell_type_name}. - MODULE: {module} terms found with cutoff 0.05.")
            else:
                # --- START: ADDED CODE TO PRINT ENRICHMENT NAMES ---
                print(f"\n--- Top Enrichment Results for {cell_type_name} - Module {module} ---")
                
                # Select the most relevant columns to display from the results DataFrame
                top_results = enrichment_results.results[['Gene_set', 'Term', 'Adjusted P-value', 'Genes']].head(30)
                
                # Use .to_string() to ensure the full pathway names are printed without being cut off
                print(top_results.to_string())
                print("-" * 20) # Add a separator for readability
                # --- END: ADDED CODE ---
            
                # Display the top 5 results
                # Display the top 5 results
                print(f"--- plotting for Cell type:{cell_type_name}. - MODULE: {module}")
                #print(enrichment_results.results.head(100))


                # if cell_type_name in ['CA1','CA3','DG','Inhibitory']:
                #     title_enrichment = f"Enrichment for {cell_type_name}-{dict_celltype_color_to_idx_WHOLE[cell_type_name]given_mapper_color_to_idx_NEURON[module]} module"
                # else:
                #     title_enrichment = f"Enrichment for {cell_type_name}-{given_mapper_color_to_idx_GLIA[module]} module"
                
                
                from gseapy import barplot, dotplot

                # title_enrichment=f"Enrichment for {cell_type_name}-{dict_celltype_color_to_idx_WHOLE[cell_type_name][module]} module"
                # output_filename = f"/home/kiamari/Single_cell/Single_cell_codes/figures/ModulePathways-GO/YekBasteh/enrichment_dotplot_{cell_type_name}_Module_{dict_celltype_color_to_idx_WHOLE[cell_type_name][module]}_ALL_GO.png"

                # ax = dotplot(enrichment_results.results,
                #           column="Adjusted P-value",
                #           x='Gene_set', # set x axis, so you could do a multi-sample/library comparsion ofname='./figures/ModulePathways-GO',
                #           size=5,
                #           top_term=5,
                #           figsize=(5,8),#(3,5)
                #           title = title_enrichment,#f"Enrichment for {module} module of {cell_type_name}",
                #           ofname=None, #f"/home/kiamari/Single_cell/Single_cell_codes/figures/ModulePathways-GO/YekBasteh/enrichment_dotplot_{cell_type_name}_Module_{dict_celltype_color_to_idx_WHOLE[cell_type_name][module]}_ALL_GO.png", 
                #           xticklabels_rot=45, # rotate xtick labels
                #           show_ring=False, # set to False to revmove outer ring
                #           marker='o',
                #          )


                # # --- Part 1: Adjust Legends (from previous request) ---
                # if ax.legends:
                #     for legend in ax.legends:
                #         for text in legend.get_texts():
                #             text.set_fontsize('large') # e.g., 12
                #         if legend.get_title():
                #             legend.get_title().set_fontsize('x-large') # e.g., 14
                #         if legend.get_title().get_text() == 'Gene_Count':
                #             for handle in legend.legendHandles:
                #                 handle.set_sizes([s * 2.0 for s in handle.get_sizes()])

                # # --- START: Part 2: Adjust Color Bar (New Change) ---
                # # The color bar is attached to the scatter plot data collection
                # if ax.collections:
                #     cbar = ax.collections[0].colorbar
                #     if cbar:
                #         # Increase the font size of the color bar's main label
                #         cbar.set_label(cbar.get_label(), fontsize=14) # Adjust size as needed
                        
                #         # Increase the font size of the numbers (tick labels) on the color bar
                #         cbar.ax.tick_params(labelsize=12) # Adjust size as needed
                # # --- END: Part 2 ---

                # # Save the fully modified figure
                # fig = ax.figure
                # fig.savefig(output_filename, dpi=300, bbox_inches='tight')
                # plt.close(fig)
                title_enrichment = f"Enrichment for {cell_type_name}-{dict_celltype_color_to_idx_WHOLE[cell_type_name][module]} module"
                output_filename = f"/home/kiamari/Single_cell/Single_cell_codes/figures/ModulePathways-GO/YekBasteh/enrichment_dotplot_{cell_type_name}_Module_{dict_celltype_color_to_idx_WHOLE[cell_type_name][module]}_ALL_GO.png"

                ax = dotplot(enrichment_results.results,
                          column="Adjusted P-value",
                          x='Gene_set',
                          size=5,
                          top_term=5,
                          figsize=(5, 8),
                          title=title_enrichment,
                          ofname=None,  # Set to None to modify before saving
                          xticklabels_rot=45,
                          show_ring=False,
                          marker='o',
                         )

                # --- START: LEGEND AND LABEL ADJUSTMENT ---

                # 1. Adjust the font size for the 'Gene_Count' legend
                legend = ax.get_legend()
                if legend:
                    for text in legend.get_texts():
                        text.set_fontsize(16) # Smaller font for legend text
                    if legend.get_title():
                        legend.get_title().set_fontsize(20) # Smaller font for legend title

                # 2. Adjust the font size for the color bar legend
                if ax.collections:
                    cbar = ax.collections[0].colorbar
                    if cbar:
                        # label_text = cbar.ax.get_ylabel()
                        # cbar.ax.set_ylabel(label_text, fontsize=20)
                        # cbar.ax.tick_params(labelsize=20)
                        
                        cbar.ax.yaxis.label.set_size(20)  # <- This directly changes the title font size
                        # Optionally, make it bold too
                        cbar.ax.yaxis.label.set_weight('bold')
                
                        # Increase font size of the tick labels (numbers)
                        cbar.ax.tick_params(labelsize=20)
                # --- THIS IS THE NEW LINE TO ADD ---
                # 3. Decrease the font size for the Y-axis term labels
                ax.tick_params(axis='y', labelsize=14) # Adjust size as needed (e.g., 8, 9, 10)
                
                # --- END: LEGEND AND LABEL ADJUSTMENT ---

                # Save the fully modified figure
                fig = ax.figure
                fig.savefig(output_filename, dpi=300, bbox_inches='tight')
                plt.close(fig)
                
        except ValueError as e:
            print(f"Error: {e}")
            print(f"No enrichment terms found for {module} {cell_type_name}. Adjust the cutoff or gene sets.")
    
    
        
        print('*'*50)
        print('*'*50)
        print('*'*50)
        

### 21. Functional enrichment, neuronal modules
Hypergeometric enrichment with FDR correction over GO, KEGG, WikiPathways, REACTOME and SynGO (Fig. 5, Figs. S11-S13).

In [ ]:
import PyWGCNA
import os

pyWGCNA_CA1 = PyWGCNA.readWGCNA("./CA1 modules_YekBasteh.p")
pyWGCNA_CA3 = PyWGCNA.readWGCNA("./CA3 modules_YekBasteh.p")
pyWGCNA_DG = PyWGCNA.readWGCNA("./DG modules_YekBasteh.p")
pyWGCNA_Inhibitory = PyWGCNA.readWGCNA("./Inhibitory modules_YekBasteh.p")


pyWGCNA_list = [pyWGCNA_CA1, pyWGCNA_CA3, pyWGCNA_DG, pyWGCNA_Inhibitory]  # each for a different cell type

# Collect module colors from each object

given_mapper_color_to_idx_NEURON = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_NEURON: 
            given_mapper_color_to_idx_NEURON[color]=index_colorname
            index_colorname +=1



pyWGCNA_Oligo = PyWGCNA.readWGCNA("./Oligo modules_YekBasteh.p")
pyWGCNA_Micro = PyWGCNA.readWGCNA("./Microglia modules_YekBasteh.p")
pyWGCNA_Astro = PyWGCNA.readWGCNA("./Astro modules_YekBasteh.p")

pyWGCNA_list = [pyWGCNA_Astro, pyWGCNA_Oligo, pyWGCNA_Micro]  # each for a different cell type

# Collect module colors from each object
given_mapper_color_to_idx_GLIA = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_GLIA: 
            given_mapper_color_to_idx_GLIA[color]=index_colorname
            index_colorname +=1
        


## --------- adjusting mapper to module names -> indices  -----------
dict_celltype_color_to_idx_WHOLE = {}

# Neuronal cell types
for celltype in ['CA1', 'CA3', 'DG', 'Inhibitory']:
    m = given_mapper_color_to_idx_NEURON.copy()  # make an independent copy
    if celltype != 'CA3':
        # swap dimgrey <-> darkgrey if both exist
        if 'dimgrey' in m and 'darkgrey' in m:
            m['dimgrey'], m['darkgrey'] = m['darkgrey'], m['dimgrey']
    dict_celltype_color_to_idx_WHOLE[celltype] = m

# Glial cell types
for celltype in ['Astro', 'Microglia', 'Oligo']:
    m = given_mapper_color_to_idx_GLIA.copy()  # independent copy
    # swap dimgrey <-> gainsboro if both exist
    if 'dimgrey' in m and 'gainsboro' in m:
        m['dimgrey'], m['gainsboro'] = m['gainsboro'], m['dimgrey']
    dict_celltype_color_to_idx_WHOLE[celltype] = m
print(f"whole mapping: {dict_celltype_color_to_idx_WHOLE}")

In [ ]:
# For Neurons
import PyWGCNA
import os


# pyWGCNA_CA1 = PyWGCNA.readWGCNA("./CA1 modules_YekBasteh.p")
# pyWGCNA_CA3 = PyWGCNA.readWGCNA("./CA3 modules_YekBasteh.p")
# pyWGCNA_DG = PyWGCNA.readWGCNA("./DG modules_YekBasteh.p")
# pyWGCNA_Inhibitory = PyWGCNA.readWGCNA("./Inhibitory modules_YekBasteh.p")


pyWGCNA_list = [pyWGCNA_CA1, pyWGCNA_CA3, pyWGCNA_DG, pyWGCNA_Inhibitory]  # each for a different cell type

# Collect module colors from each object

given_mapper_color_to_idx_NEURON = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_NEURON: 
            given_mapper_color_to_idx_NEURON[color]=index_colorname
            index_colorname +=1



# Now all_colors is the union of all module colors
print(f"given_mapper_color_to_idx_NEURON: {given_mapper_color_to_idx_NEURON}")

comparison_Neurons = PyWGCNA.compareNetworks(PyWGCNAs = [pyWGCNA_CA1, pyWGCNA_CA3, pyWGCNA_DG, pyWGCNA_Inhibitory])
comparison_Neurons.compareNetworks()
comparison_Neurons.jaccard_similarity.head(5)
comparison_Neurons.fraction.head(5)

comparison_Neurons.P_value.head(5)


# color = {"CA1 modules": "lightblue", 
#          "CA3 modules": "lightgreen",
#          "DG modules": "orange",
#         "Inhibitory modules": "#DDA0DD"}

color = {
    "CA1 modules_YekBasteh": "orange",  # Dark Magenta
    "CA3 modules_YekBasteh": "green",  # Dark Orange (earthy, no yellow tint)
    "DG modules_YekBasteh": "red",   # Saddle Brown
    "Inhibitory modules_YekBasteh": "purple"  # Indigo (deep violet, no blue-green)
}

dict_resolution_cutoff = {'low_resolution':0.05, 'high_resolution':0.20}
bubble_plot_size = {
    'low_resolution': (14,18), 
    'high_resolution': (14,10)
}
jaccard_font_node = {
    'low_resolution': 16, 
    'high_resolution': 19
}
jaccard_font_edge = {
    'low_resolution': 14, 
    'high_resolution': 20
}
node_size ={
    'low_resolution': 6000, 
    'high_resolution': 11000
}
# dic_ColorName_to_Index = None
for resolution, cutoff in sorted(dict_resolution_cutoff.items(), key=lambda x:x[1]):
    # if resolution == 'high_resolution':
    #     _ = comparison_Neurons.plotJaccardSimilarity(color=color,
    #                                      cutoff=cutoff,#0.1, 0.05,
    #                                      figsize = (15,10),
    #                                      node_size = 4000,
    #                                      plot_format="png",
    #                                      file_name=f"./figures/comparison/jaccard_similarity_Neurons_{resolution}",
    #                                      given_mapper_color_to_idx=None)
    # else:
    #     dic_ColorName_to_Index = comparison_Neurons.plotJaccardSimilarity(color=color,
    #                                      cutoff=cutoff,#0.1, 0.05,
    #                                      figsize = (15,10),
    #                                      node_size = 4000,
    #                                      plot_format="png",
    #                                      file_name=f"./figures/comparison/jaccard_similarity_Neurons_{resolution}",
    #                                      given_mapper_color_to_idx=dic_ColorName_to_Index)

    comparison_Neurons.plotJaccardSimilarity(color=color,
                                         cutoff=cutoff,#0.1, 0.05,
                                         figsize = (15,10),
                                         node_size = node_size[resolution],#7000,
                                         font_size_node = jaccard_font_node[resolution],#16,
                                         font_size_edge = jaccard_font_edge[resolution],#14,
                                         spacing_factor=0.15,
                                         scale_factor=1.0, 
                                         plot_format="png",
                                         file_name=f"./figures/comparison/jaccard_similarity_Neurons_{resolution}_YekBasteh",
                                         given_mapper_color_to_idx=dict_celltype_color_to_idx_WHOLE)
    
    comparison_Neurons.plotHeatmapComparison(plot_format="png",
                                     file_name=f"./figures/comparison/heatmap_comparison_Neurons_{resolution}_YekBasteh")
    
    # if not dic_ColorName_to_Index:
    #     print(f"it is NONE: {dic_ColorName_to_Index}")
    comparison_Neurons.plotBubbleComparison(color=color,
                                    cutoff = cutoff,#0.05
                                    figsize=bubble_plot_size[resolution],#(14,18),
                                    plot_format="png",
                                    file_name=f"./figures/comparison/bubble_comparison_Neurons_{resolution}_YekBasteh",
                                    given_mapper_color_to_idx = dict_celltype_color_to_idx_WHOLE)
    
    comparison_Neurons.saveComparison(name=f"comparison_Neurons_{resolution}_YekBasteh")
    
    
    comparison_Neurons = PyWGCNA.readComparison(f'comparison_Neurons_{resolution}_YekBasteh.p')


In [ ]:
dict_celltype_color_to_idx_WHOLE

In [ ]:
gene = "Apoe"

for cell_type, obj in zip(["CA1", "CA3", "DG", "Inhibitory"], pyWGCNA_list):
    if gene in obj.datExpr.var_names:
        module = obj.datExpr.var.loc[gene, "moduleColors"]
        print(f"{gene} is in module '{module}' for {cell_type}")
    else:
        print(f"{gene} not found in {cell_type}")


### 22. Functional enrichment, glial modules
Same procedure for astrocyte, microglial and oligodendrocyte modules.

In [ ]:
# For GLIA
import PyWGCNA

pyWGCNA_Oligo = PyWGCNA.readWGCNA("./Oligo modules_YekBasteh.p")
pyWGCNA_Micro = PyWGCNA.readWGCNA("./Microglia modules_YekBasteh.p")
pyWGCNA_Astro = PyWGCNA.readWGCNA("./Astro modules_YekBasteh.p")



pyWGCNA_list = [pyWGCNA_Astro, pyWGCNA_Oligo, pyWGCNA_Micro]  # each for a different cell type

# Collect module colors from each object
given_mapper_color_to_idx_GLIA = dict()
index_colorname = 1
for obj in pyWGCNA_list:
    colors = obj.datExpr.var['moduleColors'].unique()
    for color in colors:
        if color not in given_mapper_color_to_idx_GLIA: 
            given_mapper_color_to_idx_GLIA[color]=index_colorname
            index_colorname +=1
        
# Now all_colors is the union of all module colors
print(f"given_mapper_color_to_idx_GLIA: {given_mapper_color_to_idx_GLIA}")





comparison_Glia = PyWGCNA.compareNetworks(PyWGCNAs = [pyWGCNA_Astro, pyWGCNA_Oligo, pyWGCNA_Micro])
comparison_Glia.compareNetworks()
comparison_Glia.jaccard_similarity.head(5)
comparison_Glia.fraction.head(5)

comparison_Glia.P_value.head(5)


color = {"Astro modules_YekBasteh": "cornflowerblue", 
         "Oligo modules_YekBasteh": "pink",
        "Microglia modules_YekBasteh": "brown"}

dict_resolution_cutoff = {'low_resolution':0.05, 'high_resolution':0.19}

bubble_plot_size = {
    'low_resolution': (14,18), 
    'high_resolution': (14,9)
}
jaccard_font_node = {
    'low_resolution': 16, 
    'high_resolution': 19
}
jaccard_font_edge = {
    'low_resolution': 14, 
    'high_resolution': 20
}
node_size ={
    'low_resolution': 6000, 
    'high_resolution': 11000
}

for resolution, cutoff in sorted(dict_resolution_cutoff.items(), key=lambda x:x[1]):
    comparison_Glia.plotJaccardSimilarity(color=color,
                                     cutoff=cutoff,#0.1, 0.05
                                     figsize = (15,10),
                                     node_size = node_size[resolution],#6500,
                                     font_size_node = jaccard_font_node[resolution],#16,
                                     font_size_edge = jaccard_font_edge[resolution],#14,
                                     spacing_factor=0.15,
                                     scale_factor=1.0, 
                                     plot_format="png",
                                     file_name=f"./figures/comparison/jaccard_similarity_Glia_{resolution}_YekBasteh",
                                     given_mapper_color_to_idx = dict_celltype_color_to_idx_WHOLE)
    
    comparison_Glia.plotHeatmapComparison(plot_format="png",
                                     file_name=f"./figures/comparison/heatmap_comparison_Glia_{resolution}_YekBasteh")
    
    
    comparison_Glia.plotBubbleComparison(color=color,
                                    cutoff=cutoff,#0.1, 0.05
                                    figsize=bubble_plot_size[resolution],#(14,18),
                                    plot_format="png",
                                    file_name=f"./figures/comparison/bubble_comparison_Glia_{resolution}_YekBasteh",
                                    given_mapper_color_to_idx = dict_celltype_color_to_idx_WHOLE)
    
    comparison_Glia.saveComparison(name=f"comparison_Glia_{resolution}_YekBasteh")
    
    
    comparison_Glia = PyWGCNA.readComparison(f'comparison_Glia_{resolution}_YekBasteh.p')


### 23. Module-trait correlation heatmaps
Correlates each module eigengene with each genotype-treatment pair (Figs. S7-S8) and computes the delta correlation contrasts summarised in Fig. 3.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

dic_cell_modules = {'DG':{'genotype':['dimgrey','lightgrey'],
                          'drugeffect':['silver','darkgrey', 'gainsboro'],
                          'interactive':['dimgrey']
                           },
                    'CA1':{'genotype':['dimgrey','darkgrey'],
                           'drugeffect':['silver'],
                           'interactive':['dimgrey']
                           },
                    'CA3':{'genotype':['dimgrey','indianred'],
                           'drugeffect':['silver','darkgrey'],
                           'interactive':['dimgrey']
                           },
                    'Inhibitory':{'genotype':['dimgrey','lightcoral'],
                           'drugeffect':[],
                           'interactive':['dimgrey','lightcoral'] 
                           }
                   }

# Create output directory if it doesn't exist
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

# Loop through each effect type
for effect_type in ['genotype', 'drugeffect','interactive']:
    df_list = []

    # Loop through each cell type and its modules for the current effect
    for celltype, effects in dic_cell_modules.items():
        modules = effects[effect_type]

        # Skip if no modules for this effect
        if len(modules) == 0:
            continue

        # Load the CSV file for this cell type
        df = pd.read_csv(f"./CSV_files/trait_module_{celltype}_YekBasteh.csv")

        # Remove "ME" prefix
        df['Module'] = df['Module'].str.replace('^ME', '', regex=True)

        # Filter only modules for the current effect
        df_effect = df[df['Module'].isin(modules)].copy()

        # Append cell type name to module names
        # df_effect['Module'] = df_effect['Module'] + f' ({celltype})'
        df_effect['Module'] = df_effect['Module'].apply(lambda color: f"{celltype}-{given_mapper_color_to_idx_NEURON[color]}")


        # new_order = [2,5,1,4,0,3]  # change this based on your case
        # df_effect = df_effect.iloc[new_order].reset_index(drop=True)
        # Append to the list
        df_list.append(df_effect)

    # Concatenate all modules into a single DataFrame
    if len(df_list) == 0:
        print(f"No modules found for effect '{effect_type}'. Skipping plot.")
        continue

    df_all_effect = pd.concat(df_list, ignore_index=True)


    # Pivot the data to create correlation matrix (traits as rows, modules as columns)
    correlation_matrix = df_all_effect.pivot(index='Trait', columns='Module', values='Correlation')

    # Pivot the data to create p-value matrix (traits as rows, modules as columns)
    pvalue_matrix = df_all_effect.pivot(index='Trait', columns='Module', values='P-value')

    # Define desired trait order (row order)
    if effect_type == 'genotype':
        desired_trait_order = ['WT_Veh', 'KO_Veh', 'WT_KA3', 'KO_KA3', 'WT_KA25', 'KO_KA25']
        
        # Reorder rows in both matrices
        correlation_matrix = correlation_matrix.reindex(desired_trait_order)
        pvalue_matrix = pvalue_matrix.reindex(desired_trait_order)



    
    # Round correlation for better readability
    correlation_rounded = correlation_matrix.round(2)

    # Create labels combining correlation and p-value with exactly four decimal digits
    labels = correlation_rounded.astype(str) + "\n(" + pvalue_matrix.applymap(lambda x: f"{x:.4f}") + ")"

    # Set figure size (modules on x-axis, traits on y-axis)
    figsize = (max(20, correlation_matrix.shape[1]*2), correlation_matrix.shape[0]*1.5)

    fig, ax = plt.subplots(figsize=figsize, facecolor='white')

    # Plot heatmap (traits as rows, modules as columns)
    sns.set(font_scale=1.2)
    heatmap = sns.heatmap(correlation_matrix, annot=labels, fmt='', cmap='RdBu_r',
                          vmin=-1, vmax=1, annot_kws={'size': 14, 'weight':'bold'},
                          linewidths=0.5, linecolor='gray', ax=ax)

    # Customize tick labels
    heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=16, fontweight='bold', rotation=45, ha='right')
    heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=16, fontweight='bold', rotation=0)

    # Set title
    if effect_type == 'genotype':
        title_plot = f"Heatmap of Module-Trait Relationships for Modules Capturing Genotype Effect in Neuronal Cells"
    elif effect_type == 'drugeffect':
        title_plot = f"Heatmap of Module-Trait Relationships for Modules Capturing Drug Effect in Neuronal Cells"
    else:
        title_plot = f"Heatmap of Module-Trait Relationships for Module Capturing the Effects of Treatments on Knockout Cells Only"
    
    ax.set_title(title_plot,
                 fontsize=24, fontweight='bold')


    plt.tight_layout()

    # Save the plot
    plot_filename = os.path.join(output_dir, f"heatmap_Neuronal_{effect_type}_YekBasteh.pdf")
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.show()
    
    plt.close(fig)

    print(f"Saved heatmap for '{effect_type}' effect as '{plot_filename}'")



In [ ]:
dic_cell_modules = {'Astro':{'genotype':[],
                           'drugeffect':['dimgrey', 'whitesmoke', 'darkgrey', 'gainsboro'],
                             'interactive':[]
                           },
                    'Oligo':{'genotype':['lightgrey'],
                           'drugeffect':[],
                           'interactive':['lightgrey']  
                           },
                    'Microglia':{'genotype':['dimgrey','black'],
                           'drugeffect':[],
                            'interactive':['dimgrey','black']
                           }
                   }


# Loop through each effect type
for effect_type in ['genotype', 'drugeffect', 'interactive']:
    df_list = []

    # Loop through each cell type and its modules for the current effect
    for celltype, effects in dic_cell_modules.items():
        modules = effects[effect_type]

        # Skip if no modules for this effect
        if len(modules) == 0:
            continue

        # Load the CSV file for this cell type
        df = pd.read_csv(f"./CSV_files/trait_module_{celltype}.csv")

        # Remove "ME" prefix
        df['Module'] = df['Module'].str.replace('^ME', '', regex=True)

        # Filter only modules for the current effect
        df_effect = df[df['Module'].isin(modules)].copy()

        # Append cell type name to module names
        # df_effect['Module'] = df_effect['Module'] + f' ({celltype})'
        df_effect['Module'] = df_effect['Module'].apply(lambda color: f"{celltype}-{given_mapper_color_to_idx_GLIA[color]}")


        # Append to the list
        df_list.append(df_effect)

    # Concatenate all modules into a single DataFrame
    if len(df_list) == 0:
        print(f"No modules found for effect '{effect_type}'. Skipping plot.")
        continue

    df_all_effect = pd.concat(df_list, ignore_index=True)

    
    
    
    # Pivot the data to create correlation matrix (traits as rows, modules as columns)
    correlation_matrix = df_all_effect.pivot(index='Trait', columns='Module', values='Correlation')

    # Pivot the data to create p-value matrix (traits as rows, modules as columns)
    pvalue_matrix = df_all_effect.pivot(index='Trait', columns='Module', values='P-value')

    # Define desired trait order (row order)
    if effect_type == 'genotype':
        desired_trait_order = ['WT_Veh', 'KO_Veh', 'WT_KA3', 'KO_KA3', 'WT_KA25', 'KO_KA25']
        
        # Reorder rows in both matrices
        correlation_matrix = correlation_matrix.reindex(desired_trait_order)
        pvalue_matrix = pvalue_matrix.reindex(desired_trait_order)

    
    # Round correlation for better readability
    correlation_rounded = correlation_matrix.round(2)

    # Create labels combining correlation and p-value with exactly four decimal digits
    labels = correlation_rounded.astype(str) + "\n(" + pvalue_matrix.applymap(lambda x: f"{x:.4f}") + ")"

    # Set figure size (modules on x-axis, traits on y-axis)
    figsize = (max(20, correlation_matrix.shape[1]*2), correlation_matrix.shape[0]*1.5)

    fig, ax = plt.subplots(figsize=figsize, facecolor='white')

    # Plot heatmap (traits as rows, modules as columns)
    sns.set(font_scale=1.2)
    heatmap = sns.heatmap(correlation_matrix, annot=labels, fmt='', cmap='RdBu_r',
                          vmin=-1, vmax=1, annot_kws={'size': 14, 'weight':'bold'},
                          linewidths=0.5, linecolor='gray', ax=ax)

    # Customize tick labels
    heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=16, fontweight='bold', rotation=45, ha='right')
    heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=16, fontweight='bold', rotation=0)

    # Set title
    if effect_type == 'genotype':
        title_effect = 'Genotype'
    elif effect_type == 'drugeffect':
        title_effect = 'Drug Effect'
    
    ax.set_title(f"Heatmap of Module-Trait Relationships for Modules Capturing {title_effect} in Glial Cells",
                 fontsize=24, fontweight='bold')


    plt.tight_layout()

    # Save the plot
    plot_filename = os.path.join(output_dir, f"heatmap_Glial_{effect_type}.pdf")
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.show()
    
    plt.close(fig)

    print(f"Saved heatmap for '{effect_type}' effect as '{plot_filename}'")


### 24. Combined neuronal and glial heatmaps
Assembles the module-trait panels across all seven cell types.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Create output directory if it doesn't exist
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

for celltype in ['CA1','CA3','DG','Inhibitory']:
    # Load the CSV file for this cell type
    df = pd.read_csv(f"./CSV_files/trait_module_{celltype}_YekBasteh.csv")

    # Remove "ME" prefix from module names
    df['Module'] = df['Module'].str.replace('^ME', '', regex=True)

    # Rename modules using your mapper dictionary
    df['Module'] = df['Module'].apply(lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(color, 'NA')}")

    # Pivot the data
    correlation_matrix = df.pivot(index='Trait', columns='Module', values='Correlation')
    pvalue_matrix = df.pivot(index='Trait', columns='Module', values='P-value')
    correlation_rounded = correlation_matrix.round(2)
    labels = correlation_rounded.astype(str) + "\n(" + pvalue_matrix.applymap(lambda x: f"{x:.4f}") + ")"

    # ---- Special case: CA3 too large, split in half ----
    if celltype == 'CA3':
        n_cols = correlation_matrix.shape[1]
        split_point = n_cols // 2  # divide modules into two halves

        # Split across columns (modules)
        left_corr = correlation_matrix.iloc[:, :split_point]
        right_corr = correlation_matrix.iloc[:, split_point:]
        left_labels = labels.iloc[:, :split_point]
        right_labels = labels.iloc[:, split_point:]

        # Create 2 stacked plots (top = first half of modules, bottom = second half)
        fig, axes = plt.subplots(
            2, 1,
            figsize=(max(14, n_cols * 0.9), max(10, correlation_matrix.shape[0] * 1.2)),
            facecolor='white',
            gridspec_kw={'height_ratios': [1, 1]}
        )

        sns.set(font_scale=1.1)

        # --- Top Half (first set of modules) ---
        sns.heatmap(left_corr, annot=left_labels, fmt='', cmap='RdBu_r',
                    vmin=-1, vmax=1, annot_kws={'size': 12, 'weight': 'bold'},
                    linewidths=0.5, linecolor='gray', ax=axes[0])
        axes[0].set_title("CA3 Modules (Part 1)", fontsize=18, fontweight='bold')
        axes[0].set_xticklabels(axes[0].get_xticklabels(), fontsize=13, fontweight='bold', rotation=45, ha='right')
        axes[0].set_yticklabels(axes[0].get_yticklabels(), fontsize=14, fontweight='bold', rotation=0)

        # --- Bottom Half (remaining modules) ---
        sns.heatmap(right_corr, annot=right_labels, fmt='', cmap='RdBu_r',
                    vmin=-1, vmax=1, annot_kws={'size': 12, 'weight': 'bold'},
                    linewidths=0.5, linecolor='gray', ax=axes[1])
        axes[1].set_title("CA3 Modules (Part 2)", fontsize=18, fontweight='bold')
        axes[1].set_xticklabels(axes[1].get_xticklabels(), fontsize=13, fontweight='bold', rotation=45, ha='right')
        axes[1].set_yticklabels(axes[1].get_yticklabels(), fontsize=14, fontweight='bold', rotation=0)

        # --- Connection text ---
        fig.text(0.5, 0.51, '↓ continued (same traits) ↓', ha='center', va='center',
                 fontsize=15, fontweight='bold')

        plt.tight_layout(h_pad=2)
        plot_filename = os.path.join(output_dir, f"trait_module_CA3_YekBasteh_split.png")
        plt.savefig(plot_filename, bbox_inches='tight')
        plt.show()
        plt.close(fig)

        print(f"Saved split heatmap for 'CA3' as '{plot_filename}'")

    # ---- Normal case for others ----
    else:
        figsize = (max(10, correlation_matrix.shape[1]*1.5), max(6, correlation_matrix.shape[0]*1.2))
        fig, ax = plt.subplots(figsize=figsize, facecolor='white')
        sns.set(font_scale=1.1)
        heatmap = sns.heatmap(correlation_matrix, annot=labels, fmt='', cmap='RdBu_r',
                              vmin=-1, vmax=1, annot_kws={'size': 12, 'weight':'bold'},
                              linewidths=0.5, linecolor='gray', ax=ax)
        heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=14, fontweight='bold', rotation=45, ha='right')
        heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=14, fontweight='bold', rotation=0)

        ax.set_title(f"Heatmap of Module-Trait Relationships for {celltype}",
                     fontsize=20, fontweight='bold')
        plt.tight_layout()
        plot_filename = os.path.join(output_dir, f"trait_module_{celltype}_YekBasteh.PNG")
        plt.savefig(plot_filename, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"Saved heatmap for '{celltype}' as '{plot_filename}'")



## ================================  OLD VERSION ============================
# import pandas as pd
# import numpy as np
# import seaborn as sns
# import matplotlib.pyplot as plt
# import os
# # Create output directory if it doesn't exist
# output_dir = "./heatmap_plots"
# os.makedirs(output_dir, exist_ok=True)

# # Loop through each cell type separately
# for celltype in ['CA1','CA3','DG','Inhibitory']:
#     # Load the CSV file for this cell type
#     df = pd.read_csv(f"./CSV_files/trait_module_{celltype}_YekBasteh.csv")

#     # Remove "ME" prefix from module names
#     df['Module'] = df['Module'].str.replace('^ME', '', regex=True)

#     # Rename modules using your mapper dictionary
#     df['Module'] = df['Module'].apply(lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(color, 'NA')}")

#     # Pivot the data to create correlation matrix (traits as rows, modules as columns)
#     correlation_matrix = df.pivot(index='Trait', columns='Module', values='Correlation')

#     # Pivot the data to create p-value matrix (traits as rows, modules as columns)
#     pvalue_matrix = df.pivot(index='Trait', columns='Module', values='P-value')

#     # Round correlation for better readability
#     correlation_rounded = correlation_matrix.round(2)

#     # Create labels combining correlation and p-value with exactly four decimal digits
#     labels = correlation_rounded.astype(str) + "\n(" + pvalue_matrix.applymap(lambda x: f"{x:.4f}") + ")"

#     # Set figure size dynamically based on data dimensions
#     figsize = (max(10, correlation_matrix.shape[1]*1.5), max(6, correlation_matrix.shape[0]*1.2))

#     fig, ax = plt.subplots(figsize=figsize, facecolor='white')

#     # Plot heatmap (traits as rows, modules as columns)
#     sns.set(font_scale=1.1)
#     heatmap = sns.heatmap(correlation_matrix, annot=labels, fmt='', cmap='RdBu_r',
#                           vmin=-1, vmax=1, annot_kws={'size': 12, 'weight':'bold'},
#                           linewidths=0.5, linecolor='gray', ax=ax)

#     # Customize tick labels
#     heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=14, fontweight='bold', rotation=45, ha='right')
#     heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=14, fontweight='bold', rotation=0)

#     # Set title with full cell type names
#     full_celltype_names = {
#         'DG': 'Dentate Gyrus',
#         'CA1': 'CA1',
#         'CA3': 'CA3',
#         'Inhibitory': 'Inhibitory Neurons'
#     }
#     celltype_fullname = full_celltype_names.get(celltype, celltype)

#     ax.set_title(f"Heatmap of Module-Trait Relationships for {celltype_fullname}",
#                  fontsize=20, fontweight='bold')

#     plt.tight_layout()

#     # Save the plot
#     plot_filename = os.path.join(output_dir, f"trait_module_{celltype}_YekBasteh.png")
#     plt.savefig(plot_filename, bbox_inches='tight')
#     plt.show()
#     plt.close(fig)

#     print(f"Saved heatmap for '{celltype}' as '{plot_filename}'")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.lines as mlines

# --- Configuration ---
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES = ['CA1', 'CA3', 'DG', 'Inhibitory', 'Astro', 'Microglia', 'Oligo']
N_TOP_MODULES = 13#35 # 35 for all 13 for considerable
SIGNIFICANCE_THRESHOLD = 0.205#-.201
P_VALUE_THRESHOLD = 0.05   # show a difference only if both contributing p-values (same genotype) ≤ 0.05

# --- 1) Load correlations and p-values ---
all_correlation_mats, all_pvalue_mats = [], []
print("--- Aggregating data from all cell types... ---")
for celltype in CELL_TYPES:
    csv_path = f"./CSV_files/trait_module_{celltype}_YekBasteh.csv"
    if not os.path.exists(csv_path):
        continue

    df = pd.read_csv(csv_path)
    if 'P-value' not in df.columns:
        raise ValueError(f"CSV for {celltype} is missing 'P-value' column.")

    # Module name -> "<CellType>-<Index>"
    df['Module'] = df['Module'].str.replace('^ME', '', regex=True)
    df['Module'] = df['Module'].apply(
        lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE.get(celltype, {}).get(color, 'NA')}"
    )

    corr_mat = df.pivot(index='Trait', columns='Module', values='Correlation')
    pval_mat = df.pivot(index='Trait', columns='Module', values='P-value')

    all_correlation_mats.append(corr_mat)
    all_pvalue_mats.append(pval_mat)

# Correlations
combined_corr = pd.concat(all_correlation_mats, axis=1)
r = combined_corr.T
r.columns = pd.MultiIndex.from_tuples(
    r.columns.str.split('_', expand=True),
    names=['Genotype', 'Condition']
)
r = r.rename(columns={'Veh': 'veh', 'KA3': 'low', 'KA25': 'high'}, level='Condition')

# P-values (make numeric)
combined_pvals = pd.concat(all_pvalue_mats, axis=1)
p = combined_pvals.T
p.columns = pd.MultiIndex.from_tuples(
    p.columns.str.split('_', expand=True),
    names=['Genotype', 'Condition']
)
p = p.rename(columns={'Veh': 'veh', 'KA3': 'low', 'KA25': 'high'}, level='Condition')
p = p.apply(pd.to_numeric, errors='coerce')

# --- 2) Six within-genotype differences (3 WT + 3 KO) ---
def make_contrasts(r_df, genotype):
    rv = r_df[(genotype, 'veh')]
    rl = r_df[(genotype, 'low')]
    rh = r_df[(genotype, 'high')]
    return pd.DataFrame({
        'low - veh':  rl - rv,
        'high - veh': rh - rv,
        'high - low': rh - rl
    }, index=r_df.index)

contr_wt = make_contrasts(r, 'WT')
contr_ko = make_contrasts(r, 'KO')

# --- 3) Mask each difference by its own genotype's two p-values (both ≤ 0.05) ---
contrast_map = {
    'low - veh':  ('low',  'veh'),
    'high - veh': ('high', 'veh'),
    'high - low': ('high', 'low')
}

contr_wt_masked = contr_wt.copy()
contr_ko_masked = contr_ko.copy()

for cname, (c1, c2) in contrast_map.items():
    wt_mask = (p[('WT', c1)] <= P_VALUE_THRESHOLD) & (p[('WT', c2)] <= P_VALUE_THRESHOLD)
    ko_mask = (p[('KO', c1)] <= P_VALUE_THRESHOLD) & (p[('KO', c2)] <= P_VALUE_THRESHOLD)
    contr_wt_masked[cname] = contr_wt[cname].where(wt_mask)
    contr_ko_masked[cname] = contr_ko[cname].where(ko_mask)

# --- 4) Sort modules by max |Δr| across the SIX masked differences ---
all_masked = pd.concat([contr_wt_masked, contr_ko_masked], axis=1, keys=['WT', 'KO'])

max_abs = all_masked.abs().max(axis=1)          # NaN if all six are NaN for a module
score   = max_abs.fillna(-np.inf)               # ensure fully-NaN rows sort last
sorted_idx = score.sort_values(ascending=False).index
top_modules_to_plot = list(sorted_idx[:N_TOP_MODULES])

plot_data_wt = contr_wt_masked.loc[top_modules_to_plot]
plot_data_ko = contr_ko_masked.loc[top_modules_to_plot]

print(f"Selected {len(top_modules_to_plot)} modules (sorted by max |Δr| of the six masked differences).")
n_all_nan = int(all_masked.loc[top_modules_to_plot].isna().all(axis=1).sum())
print(f"Of these, modules with no valid differences (no icons): {n_all_nan}")

# --- 5) Plot ---
fig, ax = plt.subplots(figsize=(14, max(1, len(top_modules_to_plot) * 0.5)))

markers = {'low - veh': 'o', 'high - veh': 's', 'high - low': '^'}
colors  = {'WT': 'royalblue', 'KO': 'firebrick'}
negligible_color = 'lightgrey'

y_pos = np.arange(len(top_modules_to_plot))

# Safe x-span (handles all-NaN gracefully)
vals = np.concatenate([
    plot_data_wt.to_numpy().ravel(),
    plot_data_ko.to_numpy().ravel()
], dtype=float)
vals = vals[~np.isnan(vals)]
if vals.size == 0:
    xmin, xmax = -1.0, 1.0  # fallback span
else:
    xmin, xmax = float(vals.min()), float(vals.max())
    if xmin == xmax:
        xmin, xmax = xmin - 0.5, xmax + 0.5  # avoid zero-width span

ax.hlines(y=y_pos, xmin=xmin, xmax=xmax, color='grey', alpha=0.3, zorder=1)

# # Draw: drop NaNs so rows with no valid diffs show no icons
# for contrast, marker in markers.items():
#     for genotype, data in [('WT', plot_data_wt), ('KO', plot_data_ko)]:
#         s = data[contrast].dropna()
#         if s.empty:
#             continue
#         idx = s.index
#         y_idx = pd.Index(top_modules_to_plot).get_indexer(idx)
#         xs = s.values
#         ys = y_pos[y_idx]

#         sig = np.abs(xs) > SIGNIFICANCE_THRESHOLD
#         neg = ~sig

#         if neg.any():
#             ax.scatter(xs[neg], ys[neg], color=negligible_color, marker=marker,
#                        s=60, zorder=2, alpha=0.7)
#         if sig.any():
#             ax.scatter(xs[sig], ys[sig], color=colors[genotype], marker=marker,
#                        s=80, zorder=3, alpha=0.9, edgecolors='black')

# --- NEW: Add a grey shaded area for the negligible range ---
ax.axvspan(xmin=-SIGNIFICANCE_THRESHOLD, xmax=SIGNIFICANCE_THRESHOLD, color='grey', alpha=0.2, zorder=0)
# --- END OF NEW CODE ---

# --- MODIFIED PLOTTING LOGIC ---
for contrast, marker in markers.items():
    for genotype, data in [('WT', plot_data_wt), ('KO', plot_data_ko)]:
        s = data[contrast].dropna()
        if s.empty:
            continue
        idx = s.index
        y_idx = pd.Index(top_modules_to_plot).get_indexer(idx)
        xs = s.values
        ys = y_pos[y_idx]

        sig = np.abs(xs) > SIGNIFICANCE_THRESHOLD
        neg = ~sig

        # Set the perimeter color based on the genotype
        edge_color = colors[genotype]

        if neg.any():
            # Set fill to grey, but perimeter to the genotype color
            ax.scatter(xs[neg], ys[neg], color=negligible_color, marker=marker,
                       s=60, zorder=2, alpha=0.9, edgecolors=edge_color, linewidths=1.2)
        if sig.any():
            # Set both fill and perimeter to the genotype color
            ax.scatter(xs[sig], ys[sig], color=colors[genotype], marker=marker,
                       s=80, zorder=3, alpha=0.9, edgecolors=edge_color, linewidths=1.2)



# Aesthetics
ax.set_yticks(y_pos)
ax.set_yticklabels(top_modules_to_plot, fontsize=10)
ax.set_xlabel("Change in Correlation (Δr)", fontsize=12, fontweight='bold')
ax.set_ylabel('Module', fontsize=12, fontweight='bold') # <-- ADD THIS LINE

ax.set_title("Significant Modules with the Highest Response to Kainic Acid (WT & KO)", fontsize=16, fontweight='bold')
# ax.set_title("Response of All Significant Modules to Kainic Acid (WT & KO)", fontsize=16, fontweight='bold')

ax.axvline(0, color='black', linestyle='--', lw=1.5)
ax.grid(axis='x', linestyle='--', alpha=0.6)

# # Legends
# legend_elements_color = [
#     mlines.Line2D([0], [0], marker='o', color='w', label='WT', markerfacecolor=colors['WT'], markersize=10),
#     mlines.Line2D([0], [0], marker='o', color='w', label='KO', markerfacecolor=colors['KO'], markersize=10),
# ]
# legend1 = ax.legend(handles=legend_elements_color, loc='upper left', title="Genotype", bbox_to_anchor=(1.02, 1))
# Legends
legend_elements_color = [
    mlines.Line2D([], [], color=colors['WT'], linestyle='-', linewidth=4, label='WT'),
    mlines.Line2D([], [], color=colors['KO'], linestyle='-', linewidth=4, label='KO'),
]
legend1 = ax.legend(handles=legend_elements_color, loc='upper left', title="Genotype", bbox_to_anchor=(1.02, 1))



legend_elements_marker = [
    mlines.Line2D([0], [0], marker=markers['low - veh'],  color='w', label='$r_{KA3}-r_{Veh}$',  markerfacecolor='grey', markersize=10),
    mlines.Line2D([0], [0], marker=markers['high - veh'], color='w', label='$r_{KA25}-r_{Veh}$', markerfacecolor='grey', markersize=10),
    mlines.Line2D([0], [0], marker=markers['high - low'], color='w', label='$r_{KA25}-r_{KA3}$', markerfacecolor='grey', markersize=10),
]
ax.legend(handles=legend_elements_marker, loc='upper left', title="Contrast", bbox_to_anchor=(1.02, 0.7))
ax.add_artist(legend1)

# --- THIS IS THE ADDED LINE ---
ax.invert_yaxis()
# -----------------------------

plt.tight_layout(rect=[0, 0, 0.85, 1])

# Save & show
plot_filename = os.path.join(output_dir, "heatmap_drug_effect_barplots_IMPORTANT_MODULES_YekBasteh.png")
# plot_filename = os.path.join(output_dir, "heatmap_drug_effect_barplots_ALL_MODULES_YekBasteh.png")

plt.savefig(plot_filename, dpi=300)
plt.show()
plt.close(fig)

# print(f"Finished. Saved to: {plot_filename}")


In [ ]:
#pyWGCNA_CA1, pyWGCNA_CA3, pyWGCNA_DG, pyWGCNA_Inhibitory]  # each for a different cell type

#pyWGCNA_list = [pyWGCNA_Astro, pyWGCNA_Oligo, pyWGCNA_Micro]  # each for a different cell type

print((pyWGCNA_CA1.getGeneModule('darkgrey')))
module_dict = pyWGCNA_Astro.dataExpr_module

# 2. Calculate the number of genes for each module using a dictionary comprehension
module_gene_counts = {module: len(genes) for module, genes in module_dict.items()}

# 3. Print the results
print("Number of genes per module:")
for module, count in module_gene_counts.items():
    print(f"- {module}: {count} genes")

# Optional: If you prefer to work with a pandas Series (similar to value_counts())
import pandas as pd
module_counts_series = pd.Series(module_gene_counts).sort_values(ascending=False)

print("\nAs a pandas Series:")
print(module_counts_series)


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from adjustText import adjust_text # <-- Import the new library

# --- Configuration ---
# Create output directory if it doesn't exist
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

# Define ALL cell types you want to process
CELL_TYPES = ['CA1', 'CA3', 'DG', 'Inhibitory','Astro', 'Microglia', 'Oligo']

# Define the treatments and their desired order/labels for the plot
TREATMENTS = ['Veh']#, 'KA3', 'KA25']
TREATMENT_LABELS = ['Vehicle']#, 'Low-dose (KA3)', 'High-dose (KA25)']

# Define a color palette for cell types for consistent plotting
CELL_TYPE_COLORS = {
    'CA1': '#1f77b4', 'CA3': '#ff7f0e', 'DG': '#2ca02c', 'Inhibitory': '#d62728',
    'Astro': '#9467bd', 'Microglia': '#8c564b', 'Oligo': '#e377c2'
}



# This list will store the calculated difference data from all files
all_delta_data = []

# Loop through each cell type to gather and process data
print("--- Aggregating data from all cell types... ---")
for celltype in CELL_TYPES:
    csv_path = f"./CSV_files/trait_module_{celltype}_YekBasteh.csv"
    if not os.path.exists(csv_path):
        print(f"Warning: File not found for {celltype}. Skipping.")
        continue

    df = pd.read_csv(csv_path)
    df['Module_Color'] = df['Module'].str.replace('^ME', '', regex=True)
    df['Module'] = df['Module_Color'].apply(
        lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE.get(celltype, {}).get(color, 'NA')}"
    )
    correlation_matrix = df.pivot(index='Trait', columns='Module', values='Correlation')

    for treatment in TREATMENTS:
        ko_trait, wt_trait = f'KO_{treatment}', f'WT_{treatment}'
        if ko_trait in correlation_matrix.index and wt_trait in correlation_matrix.index:
            delta_series = correlation_matrix.loc[ko_trait] - correlation_matrix.loc[wt_trait]
            for module_name, diff_value in delta_series.items():
                all_delta_data.append({
                    'Module': module_name,
                    'CellType': celltype,
                    'Treatment': treatment,
                    'Difference': diff_value
                })

# --- Create the final DataFrame for plotting ---
if not all_delta_data:
    print("No data was processed. Exiting.")
else:
    delta_df = pd.DataFrame(all_delta_data)

    # --- Generate the Plot ---
    print("--- Generating annotated plot... ---")
    fig, ax = plt.subplots(figsize=(16, 12))

    # Use stripplot to create the categorical dot plot
    sns.stripplot(
        x='Treatment',
        y='Difference',
        hue='CellType',
        data=delta_df,
        order=TREATMENTS,
        palette=CELL_TYPE_COLORS,
        jitter=0.1,
        size=8,
        ax=ax,
        alpha=0.8,
        edgecolor='w',
        linewidth=0.5
    )

    # Add a horizontal line at y=0 for reference
    ax.axhline(0, ls='--', color='black', zorder=0)

    # --- NEW: Add annotations with adjustText ---
    texts = []
    # Create a mapping from treatment name to its numerical position on the x-axis (0, 1, 2)
    x_coords_map = {treatment: i for i, treatment in enumerate(TREATMENTS)}

    for index, row in delta_df.iterrows():
        # Get the numerical x-position for the current treatment
        x_pos = x_coords_map[row['Treatment']]
        # Add a small jitter to the x-position to match the stripplot's jitter
        x_pos_jittered = x_pos# + (np.random.rand() - 0.05) * 0.2 # This jitter should be similar to stripplot's
        
        texts.append(
            ax.text(
                x_pos_jittered,
                row['Difference'],
                row['Module'],
                fontsize=8 # Use a smaller font size for labels
            )
        )
    
    # Run the adjust_text algorithm
    # This can be slow if there are many points
    print("Adjusting text labels to avoid overlap. This may take a moment...")
    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5)
    )
    # --- End of new annotation section ---

    # --- Final Plot Customization ---
    ax.set_xticks(range(len(TREATMENTS)))
    ax.set_xticklabels(TREATMENT_LABELS, fontsize=14, fontweight='bold')
    ax.set_xlabel('Treatment Condition', fontsize=16, fontweight='bold')
    ax.set_ylabel('Difference in Correlation (r_KO - r_WT)', fontsize=16, fontweight='bold')
    ax.set_title('Genotype Effect on Module Correlation Across Treatments', fontsize=20, fontweight='bold')

    # Improve legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title='Cell Type', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for legend

    # Save the plot
    plot_filename = os.path.join(output_dir, "delta_correlation_plot_ALL_YekBasteh.pdf")
    plt.savefig(plot_filename)
    plt.show()
    plt.close(fig)

    print(f"Saved annotated plot as '{plot_filename}'")



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import math
import anndata as ad
import re

# ==============================
# --- Configuration you can edit
# ==============================
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

H5AD_PATH = "adata_filtered_significant_genes_NO_mt_YekBasteh.h5ad"

CELL_TYPES = ['CA1', 'CA3', 'DG', 'Inhibitory','Astro', 'Microglia', 'Oligo']
P_VALUE_THRESHOLD = 0.05   # 0.05 in practice
SMALL_DIFF_THRESHOLD = 0.2

CELL_TYPE_COLORS = {
    'CA1': '#1f77b4', 'CA3': '#ff7f0e', 'DG': '#2ca02c', 'Inhibitory': '#d62728',
    'Astro': '#9467bd', 'Microglia': '#8c564b', 'Oligo': '#e377c2'
}

WT_TRAIT = 'WT_Veh'
KO_TRAIT = 'KO_Veh'
CONDITION_TO_KEEP = 'Veh'
GROUP_CONDITION_COL = 'group_condition'
CELLTYPE_COL = 'celltype'
CONDITION_COL = 'condition'

# If this dict isn’t defined in your session, make an empty fallback
try:
    dict_celltype_color_to_idx_WHOLE
except NameError:
    dict_celltype_color_to_idx_WHOLE = {ct: {} for ct in CELL_TYPES}

# ==================================
# Helpers: Ns from .h5ad, Fisher test
# ==================================
def get_trait_ns_from_h5ad(adata, celltype,
                           wt_trait=WT_TRAIT, ko_trait=KO_TRAIT,
                           condition_col=CONDITION_COL, condition_keep=CONDITION_TO_KEEP,
                           group_condition_col=GROUP_CONDITION_COL):
    """
    Return (n_WT, n_KO) for the given cell type by simply counting rows (cells)
    with group_condition == WT_Veh or KO_Veh under condition == 'Veh'.
    """
    obs = adata.obs
    mask_ct   = obs[CELLTYPE_COL].astype(str) == str(celltype)
    mask_cond = obs[condition_col].astype(str) == str(condition_keep)
    mask_wt   = mask_ct & mask_cond & (obs[group_condition_col].astype(str) == str(wt_trait))
    mask_ko   = mask_ct & mask_cond & (obs[group_condition_col].astype(str) == str(ko_trait))
    return int(mask_wt.sum()), int(mask_ko.sum())

def fisher_diff_p(r_wt, r_ko, n_wt, n_ko):
    """Two-sided p-value for H0: r_wt == r_ko using independent-samples Fisher r-to-z."""
    if n_wt is None or n_ko is None or n_wt < 4 or n_ko < 4:
        return np.nan, np.nan
    r_wt = float(np.clip(r_wt, -0.999999, 0.999999))
    r_ko = float(np.clip(r_ko, -0.999999, 0.999999))
    z_wt = math.atanh(r_wt)
    z_ko = math.atanh(r_ko)
    se = math.sqrt(1/(n_wt-3) + 1/(n_ko-3))
    z = (z_ko - z_wt) / se
    p = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))
    return z, p

def make_module_label(celltype, me_col, mapper_for_ct):
    """
    Convert 'MEblue' -> 'celltype-<idx>' if color in map, else 'celltype-blue'.
    Ensures uniqueness across rows even when idx is missing.
    """
    color = re.sub(r'^ME', '', str(me_col))
    idx = mapper_for_ct.get(color, None)
    return f"{celltype}-{idx}" if idx is not None and str(idx) != 'NA' else f"{celltype}-{color}"

# =========================
# --- Load .h5ad once
# =========================
print("Loading AnnData…")
# try:
#     adata  # if already in memory, reuse
# except NameError:
#     adata = ad.read_h5ad(H5AD_PATH)

# =========================
# --- Collect Δ and p_diff
# =========================
rows = []
print("--- Aggregating data (Vehicle only)… ---")

for celltype in CELL_TYPES:
    # Ns from h5ad (cell counts)
    n_wt, n_ko = get_trait_ns_from_h5ad(adata, celltype)
    print(f"{celltype}: WT cells = {n_wt}, KO cells = {n_ko}")
    if n_wt < 4 or n_ko < 4:
        print(f"[{celltype}] Not enough cells at {CONDITION_TO_KEEP}; skipping.")
        continue

    csv_path = f"./CSV_files/trait_module_{celltype}_YekBasteh.csv"
    if not os.path.exists(csv_path):
        print(f"Warning: missing file for {celltype}, skipping.")
        continue

    df = pd.read_csv(csv_path)
    if 'P-value' not in df.columns or 'Correlation' not in df.columns or 'Trait' not in df.columns or 'Module' not in df.columns:
        raise ValueError(f"CSV for {celltype} missing required columns (need Trait, Module, Correlation, P-value).")

    # Build robust Module labels to avoid duplicate collisions
    mapper_ct = dict_celltype_color_to_idx_WHOLE.get(celltype, {})
    df['Module_Color'] = df['Module'].astype(str).str.replace('^ME', '', regex=True)
    df['Module_label'] = df['Module'].apply(lambda c: make_module_label(celltype, c, mapper_ct))

    # (Optional) debug: duplicates before pivoting
    dup_ct = df.duplicated(['Trait','Module_label']).sum()
    if dup_ct > 0:
        print(f"[{celltype}] Found {dup_ct} duplicate (Trait, Module_label) rows; collapsing via aggfunc.")

    # Use pivot_table with aggregation to tolerate duplicates
    corr = df.pivot_table(index='Trait', columns='Module_label', values='Correlation', aggfunc='mean')
    pval = df.pivot_table(index='Trait', columns='Module_label', values='P-value',    aggfunc='min')
    # ensure numeric
    pval = pval.apply(pd.to_numeric, errors='coerce')

    if WT_TRAIT not in corr.index or KO_TRAIT not in corr.index:
        print(f"[{celltype}] Missing {WT_TRAIT}/{KO_TRAIT} correlation rows; skipping.")
        continue
    if WT_TRAIT not in pval.index or KO_TRAIT not in pval.index:
        print(f"[{celltype}] Missing {WT_TRAIT}/{KO_TRAIT} p-value rows; skipping.")
        continue

    # Δ and mask (both individual correlations significant if threshold < 1)
    delta_full = corr.loc[KO_TRAIT] - corr.loc[WT_TRAIT]
    mask = (pval.loc[WT_TRAIT] < P_VALUE_THRESHOLD) & (pval.loc[KO_TRAIT] < P_VALUE_THRESHOLD)
    mask = mask.reindex(delta_full.index).fillna(False)
    # if you set P_VALUE_THRESHOLD=1, mask is all True; keep everything
    delta = delta_full[mask | (P_VALUE_THRESHOLD >= 1)]

    # Compute Fisher difference p-value per module using n_wt, n_ko from h5ad
    for mod in delta.index:
        r_wt = corr.loc[WT_TRAIT, mod]
        r_ko = corr.loc[KO_TRAIT, mod]
        z, p_diff = fisher_diff_p(r_wt, r_ko, n_wt, n_ko)

        rows.append({
            'Module': mod,
            'CellType': celltype,
            'Delta': float(delta[mod]),
            'Small': abs(delta[mod]) < SMALL_DIFF_THRESHOLD,
            'N_WT': n_wt,
            'N_KO': n_ko,
            'Z_diff': z,
            'P_diff': p_diff,
            'P_WT': pval.loc[WT_TRAIT, mod],
            'P_KO': pval.loc[KO_TRAIT, mod],
        })

# =========================
# --- Plot
# =========================
if not rows:
    print("No modules passed the p-value filter for Vehicle (or insufficient Ns).")
else:
    df_plot = pd.DataFrame(rows)
    df_plot['AbsDelta'] = df_plot['Delta'].abs()
    df_plot = df_plot.sort_values('AbsDelta', ascending=False).reset_index(drop=True)
    df_plot['y'] = np.arange(len(df_plot))[::-1]

    def color_for_row(r):
        return 'lightgrey' if r['Small'] else CELL_TYPE_COLORS.get(r['CellType'], '#333333')
    colors = df_plot.apply(color_for_row, axis=1)

    fig_h = max(4, 0.35 * len(df_plot))
    fig, ax = plt.subplots(figsize=(14, fig_h))

    ax.hlines(y=df_plot['y'], xmin=0, xmax=df_plot['Delta'], color='lightgrey', linewidth=1)
    edge_widths = np.where((df_plot['P_diff'] < 0.05), 1.8, 0.5)
    ax.scatter(df_plot['Delta'], df_plot['y'], s=60, c=colors,
               edgecolors='black', linewidths=edge_widths, zorder=3)

    for _, r in df_plot.iterrows():
        if pd.notna(r['P_diff']):
            ax.text(r['Delta'], r['y'], f"  p={r['P_diff']:.3g}", va='center', fontsize=8)

    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(df_plot['y'])
    ax.set_yticklabels(df_plot['Module'], fontsize=9)
    ax.set_xlabel('Δ correlation  (r_{KO-Veh} − r_{WT-Veh})', fontsize=12, fontweight='bold')
    ax.set_title('Genotype Effects at Vehicle (WT vs KO)\nFisher r→z p-values (Ns from h5ad cell counts)',
                 fontsize=14, fontweight='bold')

    xmin = min(0.0, df_plot['Delta'].min()) - 0.05
    xmax = max(0.0, df_plot['Delta'].max()) + 0.05
    if xmin == xmax:
        xmin, xmax = xmin - 0.5, xmax + 0.5
    ax.set_xlim(xmin, xmax)

    from matplotlib.lines import Line2D
    legend_handles = []
    present_types = df_plot.loc[~df_plot['Small'], 'CellType'].unique().tolist()
    for ct in present_types:
        legend_handles.append(Line2D([0], [0], marker='o', color='w', label=ct,
                                     markerfacecolor=CELL_TYPE_COLORS.get(ct, '#333333'),
                                     markeredgecolor='black', markersize=8))
    legend_handles.append(Line2D([0], [0], marker='o', color='w',
                                 label='P_diff < 0.05 (thick edge)',
                                 markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.8, markersize=8))
    ax.legend(handles=legend_handles, title='Legend', loc='upper right')

    plt.tight_layout()
    outpath = os.path.join(output_dir, 'genotype_delta_lollipop_YekBasteh_withPdiff_fromH5AD.png')
    # plt.savefig(outpath, dpi=300)
    plt.show()
    plt.close(fig)

    stats_out = os.path.join(output_dir, 'genotype_delta_stats_fromH5AD_YekBasteh.csv')
    # df_plot[['Module','CellType','Delta','P_WT','P_KO','N_WT','N_KO','Z_diff','P_diff','AbsDelta']].to_csv(stats_out, index=False)

    print(f"Saved lollipop plot to: {outpath}")
    print(f"Saved stats to: {stats_out}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Configuration ---
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES = ['CA1', 'CA3', 'DG', 'Inhibitory','Astro', 'Microglia', 'Oligo']
P_VALUE_THRESHOLD = 0.05
SMALL_DIFF_THRESHOLD = 0#0.105   # |Δ| < 0.15 -> grey

CELL_TYPE_COLORS = {
    'CA1': 'orange', 'CA3': 'green', 'DG': 'red', 'Inhibitory': 'purple',
    'Astro': 'cornflowerblue', 'Microglia': 'brown', 'Oligo': 'pink'
}

# --- Collect Δ = r(KO_Veh) - r(WT_Veh) passing p-value filter ---
rows = []
print("--- Aggregating data (Vehicle only)… ---")
for celltype in CELL_TYPES:
    csv_path = f"./CSV_files/trait_module_{celltype}_YekBasteh.csv"
    if not os.path.exists(csv_path):
        print(f"Warning: missing file for {celltype}, skipping.")
        continue

    df = pd.read_csv(csv_path)
    if 'P-value' not in df.columns:
        raise ValueError(f"CSV for {celltype} lacks 'P-value' column.")

    # Module -> "<CellType>-<Index>"
    df['Module_Color'] = df['Module'].str.replace('^ME', '', regex=True)
    df['Module'] = df['Module_Color'].apply(
        lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE.get(celltype, {}).get(color, 'NA')}"
    )

    corr = df.pivot(index='Trait', columns='Module', values='Correlation')
    pval = df.pivot(index='Trait', columns='Module', values='P-value').apply(pd.to_numeric, errors='coerce')

    wt_trait, ko_trait = 'WT_Veh', 'KO_Veh'
    if wt_trait not in corr.index or ko_trait not in corr.index: 
        continue
    if wt_trait not in pval.index or ko_trait not in pval.index:
        continue

    delta = corr.loc[ko_trait] - corr.loc[wt_trait]
    mask = (pval.loc[wt_trait] < P_VALUE_THRESHOLD) & (pval.loc[ko_trait] < P_VALUE_THRESHOLD)
    delta = delta[mask.reindex(delta.index).fillna(False)]

    for mod, d in delta.items():
        rows.append({
            'Module': mod,
            'CellType': celltype,
            'Delta': float(d),
            'Small': abs(d) < SMALL_DIFF_THRESHOLD
        })

if not rows:
    print("No modules passed the p-value filter for Vehicle.")
else:
    df_plot = pd.DataFrame(rows)
    # Sort by |Δ| descending (largest at top)
    df_plot['AbsDelta'] = df_plot['Delta'].abs()
    df_plot = df_plot.sort_values('AbsDelta', ascending=False).reset_index(drop=True)

    # Build y positions
    df_plot['y'] = np.arange(len(df_plot))[::-1]  # top=largest

    # Colors: grey for small; else by cell type
    def color_for_row(r):
        return 'lightgrey' if r['Small'] else CELL_TYPE_COLORS.get(r['CellType'], '#333333')
    colors = df_plot.apply(color_for_row, axis=1)

    # --- Lollipop (stem) plot ---
    fig_h = max(4, 0.35 * len(df_plot))
    fig, ax = plt.subplots(figsize=(14, fig_h))

    # stems from 0 to Δ
    ax.hlines(y=df_plot['y'], xmin=0, xmax=df_plot['Delta'], color='lightgrey', linewidth=1)

    # dots at Δ
    ax.scatter(df_plot['Delta'], df_plot['y'], s=60, c=colors, edgecolors='black', linewidths=0.5, zorder=3)

    # vertical reference line at 0
    ax.axvline(0, color='black', linestyle='--', linewidth=1)

    # y-axis labels are module names
    ax.set_yticks(df_plot['y'])
    ax.set_yticklabels(df_plot['Module'], fontsize=9)

    ax.set_xlabel('Δ correlation  (i.e., $r_{KO-Veh}  −  r_{WT-Veh}$)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Module', fontsize=12, fontweight='bold') # <-- ADD THIS LINE

    ax.set_title('Significant Genotype Effects at Vehicle (WT vs KO)', fontsize=14, fontweight='bold')

    # x-limits with some padding
    if len(df_plot):
        xmin = min(0.0, df_plot['Delta'].min()) - 0.05
        xmax = max(0.0, df_plot['Delta'].max()) + 0.05
        if xmin == xmax:
            xmin, xmax = xmin - 0.5, xmax + 0.5
        ax.set_xlim(xmin, xmax)

    # Build a small legend: grey = |Δ|<0.15; colored handles per present cell types
    from matplotlib.lines import Line2D
    legend_handles = []#[Line2D([0], [0], marker='o', color='w', label=f'|Δ| < {SMALL_DIFF_THRESHOLD}',markerfacecolor='lightgrey', markeredgecolor='black', markersize=8)]
    # Only include cell types that appear with non-small points
    present_types = df_plot.loc[~df_plot['Small'], 'CellType'].unique().tolist()
    for ct in present_types:
        legend_handles.append(Line2D([0], [0], marker='h', color='w', label=ct,
                                     markerfacecolor=CELL_TYPE_COLORS.get(ct, '#333333'),
                                     markeredgecolor='black', markersize=8))
    ax.legend(handles=legend_handles, title='Legend', loc='lower right')

    
    plt.tight_layout()
    outpath = os.path.join(output_dir, 'genotype_delta_lollipop_YekBasteh.png')
    plt.savefig(outpath, dpi=300)
    plt.show()
    plt.close(fig)
    print(f"Saved lollipop plot to: {outpath}")


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import matplotlib.lines as mlines

# --- Configuration ---
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES = ['CA1', 'CA3', 'DG', 'Inhibitory','Astro', 'Microglia', 'Oligo']
N_TOP_MODULES = 20
SIGNIFICANCE_THRESHOLD = 0.05  # For coloring the difference in correlation (Δr)
P_VALUE_THRESHOLD = 0.05       # For initial filtering of modules

# --- Data Aggregation and Filtering ---
all_delta_data = []
print("--- Processing and filtering data based on p-values... ---")
for celltype in CELL_TYPES:
    csv_path = f"./CSV_files/trait_module_{celltype}_YekBasteh.csv"
    if not os.path.exists(csv_path):
        continue
    
    df = pd.read_csv(csv_path)
    df['Module_Color'] = df['Module'].str.replace('^ME', '', regex=True)
    df['Module'] = df['Module_Color'].apply(
        lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE.get(celltype, {}).get(color, 'NA')}"
    )
    
    # --- MODIFICATION: Pivot both Correlation and p-value data ---
    # NOTE: This assumes your CSV has a column named 'p-value'. Adjust if the name is different.
    correlation_matrix = df.pivot(index='Trait', columns='Module', values='Correlation')
    pvalue_matrix = df.pivot(index='Trait', columns='Module', values='P-value')

    ko_trait, wt_trait = 'KO_Veh', 'WT_Veh'
    if ko_trait in correlation_matrix.index and wt_trait in correlation_matrix.index:
        
        # --- NEW: Filter modules where BOTH WT and KO are significant at baseline ---
        p_ko_veh = pvalue_matrix.loc[ko_trait]
        p_wt_veh = pvalue_matrix.loc[wt_trait]
        
        # Create a boolean mask for modules that are significant in both conditions
        is_significant_in_both = (p_ko_veh < P_VALUE_THRESHOLD) & (p_wt_veh < P_VALUE_THRESHOLD)
        
        # Get the names of the modules that satisfy the condition
        valid_module_names = is_significant_in_both[is_significant_in_both].index
        
        if not valid_module_names.empty:
            # Calculate the difference ONLY for these pre-filtered, valid modules
            delta_series = correlation_matrix.loc[ko_trait, valid_module_names] - correlation_matrix.loc[wt_trait, valid_module_names]
            
            for module_name, diff_value in delta_series.items():
                all_delta_data.append({
                    'Module': module_name,
                    'Difference': diff_value
                })

delta_df = pd.DataFrame(all_delta_data)

# --- Identify Top Modules from the filtered list and Prepare for Plotting ---
print(f"--- Identifying and plotting top {N_TOP_MODULES} modules from the valid set... ---")

if delta_df.empty:
    print("No modules passed the p-value filter. Cannot generate plot.")
else:
    # Find the top N modules with the largest absolute difference from the pre-filtered list
    top_modules_df = delta_df.set_index('Module')['Difference'].abs().nlargest(N_TOP_MODULES)

    # Get the full data for these top modules and sort them for plotting
    plot_df = delta_df[delta_df['Module'].isin(top_modules_df.index)].copy()
    plot_df = plot_df.sort_values('Difference', ascending=True)

    # --- Generate the Horizontal Dot Plot ---
    fig, ax = plt.subplots(figsize=(10, 8))
    y_pos = np.arange(len(plot_df))

    # Apply conditional coloring based on the effect size threshold
    colors = np.where(plot_df['Difference'].abs() > SIGNIFICANCE_THRESHOLD, 'firebrick', 'lightgrey')

    ax.hlines(y=y_pos, xmin=0, xmax=plot_df['Difference'], color='grey', alpha=0.7, zorder=1)
    ax.scatter(plot_df['Difference'], y_pos, color=colors, s=120, zorder=2, edgecolors='black', linewidth=0.5)

    # --- Finalize plot aesthetics ---
    ax.axvline(0, color='black', linestyle='-', lw=1.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df['Module'], fontsize=12)
    ax.set_xlabel('Difference in Correlation (KO_Veh - WT_Veh)', fontsize=14)
    ax.set_ylabel('Module', fontsize=14)
    ax.set_title(f'Top {len(plot_df)} Baseline Differences (Modules with p < {P_VALUE_THRESHOLD} in both WT & KO)', fontsize=14, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.6)

    legend_elements = [
        mlines.Line2D([0], [0], marker='o', color='w', label=f'Significant (|Δr| > {SIGNIFICANCE_THRESHOLD})', markerfacecolor='firebrick', markersize=12),
        mlines.Line2D([0], [0], marker='o', color='w', label=f'Negligible (|Δr| ≤ {SIGNIFICANCE_THRESHOLD})', markerfacecolor='lightgrey', markersize=12)
    ]
    ax.legend(handles=legend_elements, loc='lower right', title='Effect Size', fontsize=12)

    plt.tight_layout()

    # --- Save the plot ---
    # plot_filename = os.path.join(output_dir, "top20_baseline_difference_pvalue_filtered.pdf")
    # plt.savefig(plot_filename)
    plt.show()
    plt.close(fig)

    print(f"\nFinished generating p-value filtered plot in the '{output_dir}' directory.")


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from adjustText import adjust_text

# --- Configuration ---
# Create output directory if it doesn't exist
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

# Define ALL cell types you want to process
CELL_TYPES = ['CA1', 'CA3', 'DG', 'Inhibitory','Astro', 'Microglia', 'Oligo']

# Define the treatments and their desired order/labels for the plot
TREATMENTS = ['Veh', 'KA3', 'KA25']
TREATMENT_LABELS = ['Vehicle', 'Low-dose (KA3)', 'High-dose (KA25)']

# Define a color palette for cell types for consistent plotting
CELL_TYPE_COLORS = {
    'CA1': '#1f77b4', 'CA3': '#ff7f0e', 'DG': '#2ca02c', 'Inhibitory': '#d62728',
    'Astro': '#9467bd', 'Microglia': '#8c564b', 'Oligo': '#e377c2'
}



# Define the list of modules you want to display
MODULES_TO_DISPLAY = [
    'CA1-1', 'CA1-6', 'CA3-1', 'CA3-2', 'DG-1', 'DG-12',
    'Inhibitory-1', 'Oligo-1', 'Oligo-2', 'Oligo-12',
    'Microglia-1', 'Microglia-9', 'Astro-1',
]
# --- End Configuration ---

# This list will store the calculated difference data from all files
all_delta_data = []

# Loop through each cell type to gather and process data
print("--- Aggregating data from all cell types... ---")
for celltype in CELL_TYPES:
    csv_path = f"./CSV_files/trait_module_{celltype}_YekBasteh.csv"
    if not os.path.exists(csv_path):
        print(f"Warning: File not found for {celltype}. Skipping.")
        continue

    df = pd.read_csv(csv_path)
    df['Module_Color'] = df['Module'].str.replace('^ME', '', regex=True)
    df['Module'] = df['Module_Color'].apply(
        lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE.get(celltype, {}).get(color, 'NA')}"
    )
    correlation_matrix = df.pivot(index='Trait', columns='Module', values='Correlation')

    for treatment in TREATMENTS:
        ko_trait, wt_trait = f'KO_{treatment}', f'WT_{treatment}'
        if ko_trait in correlation_matrix.index and wt_trait in correlation_matrix.index:
            delta_series = (correlation_matrix.loc[ko_trait] - correlation_matrix.loc[wt_trait])
            for module_name, diff_value in delta_series.items():
                all_delta_data.append({
                    'Module': module_name,
                    'CellType': celltype,
                    'Treatment': treatment,
                    'Difference': diff_value
                })

# --- Create the final DataFrame for plotting ---
if not all_delta_data:
    print("No data to plot. Exiting.")
else:
    plot_df = pd.DataFrame(all_delta_data)
    # Filter for the specific modules you want to display
    plot_df = plot_df[plot_df['Module'].isin(MODULES_TO_DISPLAY)]

    # --- Plotting Section ---
    fig, ax = plt.subplots(figsize=(18, 14))

    # 1. Plot the dots
    sns.stripplot(
        x='Treatment',
        y='Difference',
        hue='CellType',
        data=plot_df,
        order=TREATMENTS,
        palette=CELL_TYPE_COLORS,
        jitter=False,  # This aligns all dots vertically per category
        s=10,          # Dot size
        ax=ax
    )

    # 2. Prepare the text labels for adjustment
    texts = []
    # Create a mapping from treatment name to its numerical x-position (0, 1, 2)
    treatment_map = {label: i for i, label in enumerate(TREATMENTS)}

    for _, row in plot_df.iterrows():
        # Get the numerical x-position for the current treatment
        x_pos = treatment_map[row['Treatment']]
        texts.append(ax.text(x_pos, row['Difference'], row['Module'], fontsize=10))

    # 3. Use adjust_text to position labels and draw arrows
    # This is the key step that moves labels and connects them with lines
    adjust_text(
        texts,
        ax=ax,
        # Add padding around points to create distance
        expand_points=(1.5, 1.5),
        # This dictionary defines the arrows connecting labels to points
        arrowprops=dict(arrowstyle='-', color='black', lw=0.7)
    )

    # --- Final Plot Customization ---
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xticks(range(len(TREATMENT_LABELS)))
    ax.set_xticklabels(TREATMENT_LABELS, fontsize=14, fontweight='bold')
    ax.set_xlabel('Treatment Condition', fontsize=16, fontweight='bold')
    ax.set_ylabel('Correlation Difference (KO - WT), i.e. $r_{KO}-r_{WT}$', fontsize=16, fontweight='bold')
    # ax.set_title('Genotype Effect on Module Correlation Across Treatments', fontsize=20, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    
    # Customize legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, title='Cell Type', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12)

    plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust layout to make space for legend

    # Save the plot
    plot_filename = os.path.join(output_dir, "annotated_delta_correlation_YekBasteh.pdf")
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    print(f"Saved annotated plot as '{plot_filename}'")
    print(plot_df)

### 25. Module overlap by Jaccard index
Quantifies shared gene membership between genotype- and KA-associated modules and tests the significance of each overlap (Fig. 4).

In [ ]:
# Output directory for plots
output_dir = "./heatmap_plots"
os.makedirs(output_dir, exist_ok=True)

dic_full_name_celltypes_GLIA = {'Astro':'Astrocytes','Oligo':'Oligodendrocytes', 'Microglia':'Microglia'}
# Loop through each cell type
for celltype in ['Astro','Oligo', 'Microglia']:
    # Load the CSV file for this cell type
    df = pd.read_csv(f"./CSV_files/trait_module_{celltype}_YekBasteh.csv")

    # Remove "ME" prefix from module names
    df['Module'] = df['Module'].str.replace('^ME', '', regex=True)

    # Rename modules using your mapper dictionary
    df['Module'] = df['Module'].apply(lambda color: f"{celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(color, 'NA')}")

    # Pivot the data to create correlation matrix (traits as rows, modules as columns)
    correlation_matrix = df.pivot(index='Trait', columns='Module', values='Correlation')

    # Pivot the data to create p-value matrix (traits as rows, modules as columns)
    pvalue_matrix = df.pivot(index='Trait', columns='Module', values='P-value')

    # Round correlation for better readability
    correlation_rounded = correlation_matrix.round(2)

    # Create labels combining correlation and p-value with exactly four decimal digits
    labels = correlation_rounded.astype(str) + "\n(" + pvalue_matrix.applymap(lambda x: f"{x:.4f}") + ")"

    # Set figure size dynamically based on data dimensions
    figsize = (max(10, correlation_matrix.shape[1]*1.5), max(6, correlation_matrix.shape[0]*1.2))

    fig, ax = plt.subplots(figsize=figsize, facecolor='white')

    # Plot heatmap (traits as rows, modules as columns)
    sns.set(font_scale=1.1)
    heatmap = sns.heatmap(correlation_matrix, annot=labels, fmt='', cmap='RdBu_r',
                          vmin=-1, vmax=1, annot_kws={'size': 12, 'weight':'bold'},
                          linewidths=0.5, linecolor='gray', ax=ax)

    # Customize tick labels
    heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=14, fontweight='bold', rotation=45, ha='right')
    heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=14, fontweight='bold', rotation=0)

    # Set title
    ax.set_title(f"Heatmap of Module-Trait Relationships for {dic_full_name_celltypes_GLIA[celltype]}",
                 fontsize=20, fontweight='bold')

    plt.tight_layout()

    # Save the plot
    plot_filename = os.path.join(output_dir, f"trait_module_{celltype}_YekBasteh.pdf")
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    print(f"Saved heatmap for '{celltype}' as '{plot_filename}'")


In [ ]:
import PyWGCNA

def find_gene_module(celltype_file, gene_name):
    """
    Find the module that contains a given gene in a PyWGCNA object.
    
    Args:
        celltype_file (str): Path to the .p file (PyWGCNA object).
        gene_name (str): Gene of interest (must match index in datExpr.var).
        
    Returns:
        str: Module color the gene belongs to, or None if not found.
        list: Genes in the same module.
    """
    # Load the PyWGCNA object
    wgcna_obj = PyWGCNA.readWGCNA(celltype_file)

    # Ensure gene exists
    if gene_name not in wgcna_obj.datExpr.var.index:
        print(f"⚠️ Gene '{gene_name}' not found in this cell type.")
        return None, []

    # Get the module color for this gene
    module_color = wgcna_obj.datExpr.var.loc[gene_name, "moduleColors"]

    # Get all genes in the same module
    genes_in_module = wgcna_obj.datExpr.var.index[
        wgcna_obj.datExpr.var["moduleColors"] == module_color
    ].tolist()

    return module_color, genes_in_module


# 🔹 Example usage
celltype = "Oligo"
file_path = f"./{celltype} modules_YekBasteh.p"
gene = "Apoe"

module_color, genes = find_gene_module(file_path, gene)

if module_color:
    print(f"Gene {gene} is in module '{module_color}' for {celltype}.")
    print(f"Number of genes in this module: {len(genes)}")
    print("First 10 genes:", genes[:10])


### 26. Co-expression subnetworks of the top hub genes
Draws the top-20 degree-centrality subnetworks for each module (Fig. 6A, 6B, Fig. S14).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import PyWGCNA
import numpy as np
import math 
import matplotlib.colors as colors
import matplotlib.cm as cm

print(f"dict {dict_celltype_color_to_idx_WHOLE}")



pyWGCNA_celltype = PyWGCNA.readWGCNA("./Astro modules_YekBasteh.p")
# print((pyWGCNA_celltype.adjacency))
# print(cc)
# Extract genes belonging to the "dimgrey" module
celltype = 'Astro'
module_color = 'gainsboro'
genes_in_module = pyWGCNA_celltype.datExpr.var.index[pyWGCNA_celltype.datExpr.var['moduleColors'] == module_color].tolist()

print(f"Number of genes in {module_color} module: {len(genes_in_module)}")
status_apoe = 'Apoe' in genes_in_module
print(f"Apoe is in this module: {status_apoe}")
# Extract adjacency matrix for the module genes
adjacency_matrix = pyWGCNA_celltype.adjacency.loc[genes_in_module, genes_in_module]

# Optional: set a threshold to simplify the network (e.g., keep edges with adjacency > 0.1)
threshold = 0.02#0.15
adjacency_matrix_filtered = adjacency_matrix.where(adjacency_matrix > threshold, 0)


# Create NetworkX graph
G = nx.from_pandas_adjacency(adjacency_matrix_filtered)


# Compute centrality (degree centrality as an example)
centrality = nx.degree_centrality(G)
# centrality = nx.closeness_centrality(G)

# Select top N genes by centrality for clearer visualization (optional)
top_n = 20  # Adjust this number as needed
top_genes = sorted(centrality, key=centrality.get, reverse=True)[:top_n]
print(f"Top genes in terms of Criterion: {top_genes}")
G_sub = G.subgraph(top_genes)

# Compute centrality again for subgraph

# centrality_sub = nx.degree_centrality(G_sub)
centrality_sub = {node: centrality[node] for node in G_sub.nodes()}

# Sort the dictionary by value in descending order
sorted_centrality = dict(sorted(centrality_sub.items(), key=lambda item: item[1], reverse=True))

# Print the sorted dictionary
for node, value in sorted_centrality.items():
    print(f"{node}: {value}")


# Define layout: nodes with higher centrality closer to center
def centrality_layout(G, centrality_dict):
    nodes_sorted = sorted(centrality_dict, key=centrality_dict.get, reverse=True)
    pos = {}
    num_nodes = len(nodes_sorted)
    for i, node in enumerate(nodes_sorted):

        angle = 2 * np.pi * i / (num_nodes+5)
        radius = .02 - centrality_sub[node]  # higher centrality closer to center
        pos[node] = (radius * np.cos(angle), radius * np.sin(angle))
    return pos

# def spiral_layout(graph, spacing=0.05, turns=1):
#     nodes = list(graph.nodes())
#     n = len(nodes)
#     pos = {}
#     for i, node in enumerate(nodes):
#         angle = 2 * math.pi * turns * (i / n)  # Spread over multiple turns
#         radius = spacing * i
#         x = radius * math.cos(angle)
#         y = radius * math.sin(angle)
#         pos[node] = (x, y)
#     return pos

# # Use spiral layout
# pos = spiral_layout(G_sub, spacing=0.5, turns=3)

pos = centrality_layout(G_sub, centrality_sub)




# Get centrality values for nodes
centrality_values = np.array([centrality_sub[node] for node in G_sub.nodes()])

# Normalize centrality values for colormap
norm = colors.Normalize(vmin=centrality_values.min(), vmax=centrality_values.max())

# Choose a colormap (dark red for high centrality, blue for low)
cmap = cm.get_cmap('RdYlBu_r')

# Assign colors based on centrality values
node_colors = [cmap(norm(centrality_sub[node])) for node in G_sub.nodes()]

# Create figure and axes explicitly
fig, ax = plt.subplots(figsize=(12, 12))

# Draw nodes and edges on the specified axes
nx.draw_networkx_nodes(G_sub, pos, node_size=800, node_color=node_colors, alpha=0.9, ax=ax)

####
# nx.draw_networkx_edges(G_sub, pos, alpha=0.5, ax=ax)
# Extract edge weights
edges = G_sub.edges(data=True)
edge_weights = [data['weight'] for _, _, data in edges]

# Normalize edge weights for greyscale
edge_norm = colors.Normalize(vmin=min(edge_weights), vmax=max(edge_weights))
greyscale = cm.get_cmap('Greys')  # lighter = lower weight, darker = higher

# Map weights to colors (darker = stronger connection)
edge_colors = [greyscale(edge_norm(w)) for w in edge_weights]

# Draw weighted edges with color
nx.draw_networkx_edges(
    G_sub, pos,
    edgelist=edges,
    edge_color=edge_colors,
    width=2,
    alpha=0.9,
    ax=ax
)

####



# Add gene names inside circles
nx.draw_networkx_labels(G_sub, pos, font_size=7, font_weight='bold', ax=ax)

# Add colorbar to indicate centrality scale
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Explicitly pass the axes to the colorbar
cbar = plt.colorbar(sm, shrink=0.7, ax=ax)
cbar.set_label('Degree Centrality', fontsize=12)

# Set title
ax.set_title(f"Network graph of the top {top_n} genes in {celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(module_color, 'NA')} module", fontsize=16, fontweight='bold')

# Remove axis
ax.axis('off')

# Save the plot (optional)
plt.tight_layout()
# plt.savefig(f"./figures/modules/network_{module_color}_{celltype}_YekBasteh.png", dpi=300)
plt.savefig(f"./figures/modules/network_{celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(module_color, 'NA')}_YekBasteh.png", dpi=300)

# Show plot
plt.show()

# # Compute centrality again for subgraph
# centrality_sub = nx.degree_centrality(G_sub)

# # Compute positions
# pos = nx.spring_layout(G_sub, seed=42)  # or your custom layout

# # Node colors based on centrality
# centrality_values = np.array([centrality_sub[node] for node in G_sub.nodes()])
# norm = colors.Normalize(vmin=centrality_values.min(), vmax=centrality_values.max())
# node_cmap = cm.get_cmap('RdYlBu_r')
# node_colors = [cmap(norm(centrality_sub[node])) for node in G_sub.nodes()]

# # Edge colors based on weights
# edge_weights = np.array([G_sub[u][v]['weight'] for u, v in G_sub.edges()])
# edge_norm = colors.Normalize(vmin=edge_weights.min(), vmax=edge_weights.max())
# edge_cmap = cm.get_cmap('Greys')
# edge_colors = [edge_cmap(edge_norm(weight)) for weight in edge_weights]

# # Plot nodes
# fig, ax = plt.subplots(figsize=(10, 8))
# nx.draw_networkx_nodes(G_sub, pos, node_size=800, node_color=node_colors, alpha=0.9, ax=ax)

# # Draw edges with greyscale colors (darker = higher weight)
# nx.draw_networkx_edges(G_sub, pos, edge_color=edge_colors, width=2, ax=ax)

# # Add labels
# nx.draw_networkx_labels(G_sub, pos, font_size=7, ax=ax)

# # Add colorbar for node centrality
# sm = cm.ScalarMappable(cmap=cmap, norm=norm)
# sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
# cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.7)
# cbar.set_label('Degree Centrality', fontsize=12)

# # Set title and remove axis
# ax.set_title(f"Network plot of top genes in '{module_color}' module ({celltype})", fontsize=16, fontweight='bold')
# ax.axis('off')

# plt.show()

### 27. Pan-neuronal hub gene ranking
Ranks genes by their maximum degree centrality across the neuronal modules of Fig. 3D. Fth1, Rplp1 and Apoe emerge as the most highly connected genes (Fig. 6C).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import PyWGCNA
import numpy as np
import pandas as pd
import matplotlib.colors as colors
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap



# --- Function: Extract top N by degree centrality for specified modules in each cell type ---
def process_specified_modules(celltype_file, module_names, adj_threshold, top_n=None):
    pyWGCNA_celltype = celltype_file#PyWGCNA.readWGCNA(celltype_file)

    # pyWGCNA_celltype = PyWGCNA.readWGCNA(celltype_file)
    results = {}
    for module_color in module_names:
        genes_in_module = pyWGCNA_celltype.datExpr.var.index[
            pyWGCNA_celltype.datExpr.var['moduleColors'] == module_color].tolist()
        if len(genes_in_module) == 0:
            continue
        adjacency_matrix = pyWGCNA_celltype.adjacency.loc[genes_in_module, genes_in_module]
        adjacency_matrix_filtered = adjacency_matrix.where(adjacency_matrix > adj_threshold, 0)
        G = nx.from_pandas_adjacency(adjacency_matrix_filtered)
        centrality = nx.degree_centrality(G)
        # Optionally select only the top N by centrality
        if top_n is not None:
            top_genes = sorted(centrality, key=centrality.get, reverse=True)[:top_n]
            centrality = {gene: centrality[gene] for gene in top_genes}
        results[module_color] = centrality
    return results

# ---- USER CONFIG ----
specified_modules = {
    'DG': ['dimgrey','black'],
    'CA1': ['dimgrey','darkgrey'],
    'CA3': ['darkgrey','silver'],  # 
    'Inhibitory': ['dimgrey']  # , 'lightcoral'
}
celltypes_files = {
    'DG': pyWGCNA_DG,#'./DG modules.p',
    'CA1':pyWGCNA_CA1,# './CA1 modules.p',
    'CA3':pyWGCNA_CA3,# './CA3 modules.p',
    'Inhibitory':pyWGCNA_Inhibitory,# './Inhibitory modules.p'
}
adj_threshold = 0.02
top_n_per_module = 2000  # or None for all

# --- Compute for all modules ---
all_results = {}
for celltype, file in celltypes_files.items():
    if file is None: continue # Skip if placeholder object is not replaced
    all_results[celltype] = process_specified_modules(
        file, specified_modules[celltype], adj_threshold, top_n=top_n_per_module)

# --- Build DataFrame with raw centrality scores ---
genes = set()
colnames = []
for celltype, modules in all_results.items():
    for module_color in modules.keys():
        colnames.append(f"{celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(module_color, 'NA')}")
        genes.update(modules[module_color].keys())
genes = sorted(list(genes))
df_centrality = pd.DataFrame(index=genes, columns=sorted(colnames))
for celltype, modules in all_results.items():
    for module_color, genes_dict in modules.items():
        col = f"{celltype}-{dict_celltype_color_to_idx_WHOLE[celltype].get(module_color, 'NA')}"
        for gene, cent in genes_dict.items():
            df_centrality.at[gene, col] = cent
df_centrality = df_centrality.astype(float)

# ==============================================================================
# --- CHANGE 1: Sort genes based on their MAXIMUM raw centrality score ---
# ==============================================================================
# For each gene (row), find its highest centrality value across all modules.
max_raw_cent = df_centrality.max(axis=1)
df_sorter = pd.DataFrame({'MaxRawCent': max_raw_cent})

# Sort the genes based on this maximum value in descending order.
df_sorter = df_sorter.sort_values('MaxRawCent', ascending=False)
# ==============================================================================

# --- Select top N genes and prepare the data for plotting ---
top_n = 100
df_top_genes = df_sorter.head(top_n).index.tolist()

# Use the raw centrality data for the plot, ordered by our new sorting.
df_centrality_top = df_centrality.loc[df_top_genes]
# Reverse order for top-to-bottom display
df_centrality_top = df_centrality_top.iloc[::-1]

print(f"--- Top {top_n} Genes Being Plotted (Sorted by Maximum Centrality) ---")
print(", ".join(df_centrality_top.index))

# --- Plotting ---
cmap = LinearSegmentedColormap.from_list("BuYlRd", ["blue", "yellow", "red"])
fig, ax = plt.subplots(figsize=(1.8 * len(df_centrality_top.columns), 0.2 * len(df_centrality_top)))
rect_width = 0.6
rect_height = 1.0

# --- CHANGE 2: Determine color scale from the GLOBAL min and max of the raw data ---
# This ensures a consistent color scale across the entire heatmap.
vmin = df_centrality_top.min().min()
vmax = df_centrality_top.max().max()

for x, col in enumerate(df_centrality_top.columns):
    yvals = np.arange(len(df_centrality_top))
    # This variable now holds raw centrality scores
    centrality_values = df_centrality_top[col].values.astype(float)
    for i, y in enumerate(yvals):
        val = centrality_values[i]
        if np.isnan(val):
            color = "white"
        else:
            # Normalize the raw value to the [0, 1] range for the colormap
            color = cmap((val - vmin) / (vmax - vmin))
        rect = mpatches.Rectangle(
            (x - rect_width/2, y - rect_height/2),
            rect_width, rect_height,
            linewidth=1, edgecolor='k', facecolor=color, alpha=0.85
        )
        ax.add_patch(rect)

ax.set_xlim(-0.5, len(df_centrality_top.columns) - 0.5)
ax.set_ylim(-0.5, len(df_centrality_top) - 0.5)

ax.set_xticks(range(len(df_centrality_top.columns)))
ax.set_xticklabels(df_centrality_top.columns, rotation=45, ha='left', fontsize=12)
ax.set_yticks(range(len(df_centrality_top)))
ax.set_yticklabels(df_centrality_top.index, fontsize=9)
ax.xaxis.tick_top()
ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False)

# --- CHANGE 3: Update plot title and color bar label ---
ax.set_title(f'Top {top_n} Genes Sorted by Maximum Centrality Score', pad=40)
ax.set_xlabel('')
ax.set_ylabel('Genes')

# Use the new vmin and vmax for the color bar
sm = plt.cm.ScalarMappable(norm=colors.Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
cbar = plt.colorbar(sm, ax=ax, label='Centrality Score', shrink=0.6)

plt.tight_layout()
plt.savefig(f"./figures/Table_of_genes_across_modules_max_raw_centrality_heatmap_top_{top_n}.png", dpi=300)
plt.show()

